# Setup 03 — Generate sales
Part of the retail-demo setup utility. Re-runnable (overwrite-by-design).

Builds candidate in-store receipts and resolves them against the causal inventory policy (fulfillment). Every receipt-dependent notebook waits for this one.

This notebook runs its slice of the generation stage graph (see `STAGES` in
the engine); stages whose inputs are ready run concurrently. Inputs from
upstream notebooks are read from `gen__*` handoff tables in the Silver
staging schema, and outputs are written there for later notebooks.
`setup-10-publish-silver` validates and publishes everything atomically.
The engine cell below is generated by `scripts/build_notebooks.py` — edit
the package sources under `utility/src/retail_setup/`, never the cell.

In [ ]:
# The inlined engine below imports pydantic v2 on the driver. Newer Fabric runtimes
# preinstall pydantic and may already have its compiled pydantic_core loaded; a
# compiled module cannot be replaced inside a running process, so installing over
# it pairs new Python files with the old binary (ImportError: validate_core_schema).
# Reuse a working runtime pydantic v2; otherwise (older runtimes ship an OLD
# typing_extensions and no pydantic) install pinned versions. Why each step:
#  - subprocess pip (not %pip): a %pip kernel restart tears down the Spark session
#    mid pipeline run and fails the activity.
#  - pin pydantic <2.12: 2.12+ imports typing_extensions.Sentinel.
#  - drop the stale modules from sys.modules: the runtime already imported the old
#    typing_extensions, so the freshly installed copy is shadowed; evicting it lets
#    the engine cell re-import the upgraded versions (pydantic needs symbols such
#    as TypeIs that the runtime's typing_extensions lacks).
#  - pydantic runs only on the driver (engine builds rows, then
#    spark.createDataFrame), so a driver-side install is sufficient.
import importlib
import subprocess
import sys


def _runtime_pydantic_v2() -> bool:
    if "pydantic_core" in sys.modules or "pydantic" in sys.modules:
        loaded = sys.modules.get("pydantic")
        if loaded is not None and not str(getattr(loaded, "VERSION", "2.")).startswith("2."):
            raise RuntimeError("The runtime loaded pydantic v1; the engine requires v2.")
        return True
    probe = (
        "import pydantic, sys; from pydantic import BaseModel, Field, field_validator; "
        "sys.exit(0 if pydantic.VERSION.startswith('2.') else 1)"
    )
    return subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0


if _runtime_pydantic_v2():
    print("Using the Fabric runtime's pydantic v2.")
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "annotated-types==0.7.0", "pydantic==2.11.10",
        "pydantic-core==2.33.2", "typing-inspection==0.4.2",
        "typing_extensions==4.15.0",
    ])
    for _name in [m for m in list(sys.modules)
                  if m.split(".", 1)[0] in {"typing_extensions", "typing_inspection",
                                            "pydantic", "pydantic_core"}]:
        del sys.modules[_name]
    importlib.invalidate_caches()

In [ ]:
# PARAMETERS — rendered by `retail-setup render`; defaults work unrendered
def _param(value: str, default: str) -> str:
    return default if len(value) > 1 and value[0] == value[1] == "{" else value

LAKEHOUSE_NAME = _param("{{LAKEHOUSE_NAME}}", "retail_lakehouse")
SILVER_DB = _param("{{SILVER_DB}}", "ag")
GOLD_DB = _param("{{GOLD_DB}}", "au")
STORE_TYPE = _param("{{STORE_TYPE}}", "supercenter")
START_DATE = _param("{{START_DATE}}", "2025-01-01")
END_DATE = _param("{{END_DATE}}", "2025-03-31")
STORE_COUNT = int(_param("{{STORE_COUNT}}", "50"))
SEED = int(_param("{{SEED}}", "42"))
DICTIONARY_REF = _param("{{DICTIONARY_REF}}", "main")

spark.conf.set("spark.sql.session.timeZone", "UTC")  # engine timestamps depend on it
# Reorder seeded-draw math ((xxhash64(...)/1e12)*k -> (xxhash64(...)*k)/1e12) to
# cut rounding-error propagation and silence Fabric's Spark advisor. The rewrite
# shifts draws by <=1 ULP (generation is statistical, no exact-value contracts);
# set in every engine driver so dims and facts stay mutually consistent.
spark.conf.set("spark.advise.divisionExprConvertRule.enable", "true")

In [ ]:
# ENGINE SOURCE (generated — do not edit)
# Built by scripts/build_notebooks.py from utility/src/retail_setup/.

# --- retail_setup/dictionaries/models.py ---
"""Pydantic models for dictionary JSON files.

Entry models mirror the field names in
datagen/src/retail_datagen/shared/models.py (PascalCase) so the supercenter
conversion is verifiable 1:1. StoreTypeProfile is new: the behavioral knobs
that differentiate store types.
"""

import re
from decimal import Decimal

from pydantic import BaseModel, Field, field_validator


class TaxJurisdictionEntry(BaseModel):
    StateCode: str = Field(..., min_length=2, max_length=2)
    County: str = Field(..., min_length=1)
    City: str = Field(..., min_length=1)
    CombinedRate: Decimal = Field(..., ge=0, le=Decimal("0.20"))

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()

    @field_validator("CombinedRate", mode="before")
    @classmethod
    def _rate_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("CombinedRate must be a valid decimal number") from None


class GeographyEntry(BaseModel):
    City: str = Field(..., min_length=1)
    State: str = Field(..., min_length=2, max_length=2)
    Zip: str
    District: str = Field(..., min_length=1)
    Region: str = Field(..., min_length=1)

    @field_validator("State")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("State must be alphabetic")
        return v.upper()

    @field_validator("Zip")
    @classmethod
    def _zip_format(cls, v: str) -> str:
        if not re.match(r"^\d{5}(-\d{4})?$", v):
            raise ValueError("Zip must be 12345 or 12345-6789")
        return v


class NameEntry(BaseModel):
    """One first or last name (shared across store types).

    The ``Name`` field intentionally unifies the separate ``FirstNameDict`` and
    ``LastNameDict`` models from datagen — both source models carry a single
    name string, so one model suffices here.
    """

    Name: str = Field(..., min_length=1)


class ProductBrandEntry(BaseModel):
    Brand: str = Field(..., min_length=1)
    Company: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)


class ProductEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    BasePrice: Decimal = Field(..., gt=0)
    Department: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)
    Subcategory: str = Field(..., min_length=1)
    Tags: str | None = None

    @field_validator("BasePrice", mode="before")
    @classmethod
    def _price_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("BasePrice must be a valid decimal number") from None


class ProductTagEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    Tags: str = Field(..., min_length=1)


class StoreTypeProfile(BaseModel):
    """Behavioral knobs that make a store type act differently.

    Weight lists are relative (normalized at use time, not here).
    """

    store_type: str = Field(..., min_length=1)
    display_name: str = Field(..., min_length=1)
    basket_lambda: float = Field(..., gt=0, description="Nominal mean basket-line ceiling")
    avg_ticket_target: float = Field(..., gt=0, description="Basket calibration target, USD")
    hourly_weights: list[float] = Field(..., description="24 relative traffic weights")
    daily_weights: list[float] = Field(..., description="7 weights, Monday first")
    monthly_weights: list[float] = Field(..., description="12 seasonality weights, Jan first")
    department_weights: dict[str, float] = Field(..., min_length=1)
    promo_rate: float = Field(..., ge=0, le=1, description="Share of lines with a promotion")
    online_order_share: float = Field(..., ge=0, le=1)
    zones: list[str] = Field(..., min_length=1, description="Store footprint zones for BLE")

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_nonneg(cls, v: list[float]) -> list[float]:
        if any(w < 0 for w in v):
            raise ValueError("weight values must be non-negative")
        return v

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_have_effect(cls, v: list[float]) -> list[float]:
        # An all-zero weight vector is a declared-but-unused control: it can't be
        # normalized and would silently suppress all traffic. Reject it so profile
        # controls provably drive generation (IMP-010 validated profile controls).
        if sum(v) <= 0:
            raise ValueError("weight values must not all be zero")
        return v

    @field_validator("hourly_weights")
    @classmethod
    def _24(cls, v: list[float]) -> list[float]:
        if len(v) != 24:
            raise ValueError("hourly_weights must have 24 entries")
        return v

    @field_validator("daily_weights")
    @classmethod
    def _7(cls, v: list[float]) -> list[float]:
        if len(v) != 7:
            raise ValueError("daily_weights must have 7 entries")
        return v

    @field_validator("monthly_weights")
    @classmethod
    def _12(cls, v: list[float]) -> list[float]:
        if len(v) != 12:
            raise ValueError("monthly_weights must have 12 entries")
        return v

    @field_validator("department_weights")
    @classmethod
    def _departments_have_effect(cls, v: dict[str, float]) -> dict[str, float]:
        # Every declared department must carry positive weight: a zero/negative
        # weight is an unused control that silently excludes a department from
        # every basket (IMP-010 validated profile controls).
        bad = sorted(k for k, w in v.items() if w <= 0)
        if bad:
            raise ValueError(f"department_weights must be positive; got <=0 for {bad}")
        return v

# --- retail_setup/dictionaries/loader.py ---
"""Load and validate dictionary JSON sets (shared + one store type)."""

import json
from dataclasses import dataclass, field
from importlib.resources import files
from pathlib import Path

from pydantic import BaseModel, ValidationError



@dataclass
class DictionarySet:
    store_type: str
    profile: StoreTypeProfile
    first_names: list[NameEntry] = field(default_factory=list)
    last_names: list[NameEntry] = field(default_factory=list)
    geographies: list[GeographyEntry] = field(default_factory=list)
    tax_rates: list[TaxJurisdictionEntry] = field(default_factory=list)
    products: list[ProductEntry] = field(default_factory=list)
    brands: list[ProductBrandEntry] = field(default_factory=list)
    tags: list[ProductTagEntry] = field(default_factory=list)


def default_dictionary_root() -> Path:
    """Return the packaged dictionary root, with a source-tree fallback."""
    packaged_root = files("retail_setup").joinpath("resources", "dictionaries")
    if packaged_root.is_dir():
        if not isinstance(packaged_root, Path):
            raise RuntimeError("Packaged dictionaries are not available as filesystem paths.")
        return packaged_root

    source_root = Path(__file__).resolve().parents[3] / "data" / "dictionaries"
    if source_root.is_dir():
        return source_root
    raise RuntimeError("Default dictionaries are missing from the installed package.")


def available_store_types(root: Path) -> list[str]:
    if not root.is_dir():
        return []
    return sorted(
        p.name
        for p in root.iterdir()
        if p.is_dir() and not p.name.startswith("_") and (p / "profile.json").exists()
    )


def load_list(path: Path, model: type[BaseModel]) -> list:
    """Load and validate a JSON array file, returning a list of model instances.

    Raises ValueError with filename and row index on validation failure.
    """
    try:
        raw = json.loads(path.read_text())
    except FileNotFoundError:
        raise ValueError(f"missing dictionary file: {path}") from None
    if not isinstance(raw, list):
        raise ValueError(f"{path.name}: expected a JSON array")
    result = []
    for i, row in enumerate(raw):
        try:
            result.append(model.model_validate(row))
        except ValidationError as exc:
            raise ValueError(f"{path.name}[{i}]: {exc}") from exc
    return result


# Keep the private alias for any callers that haven't migrated yet.
_load_list = load_list


def load_dictionaries(root: Path, store_type: str) -> DictionarySet:
    type_dir = root / store_type
    if not (type_dir / "profile.json").exists():
        raise ValueError(
            f"unknown store type {store_type!r}; available: {available_store_types(root)}"
        )
    shared = root / "_shared"

    try:
        profile = StoreTypeProfile.model_validate(
            json.loads((type_dir / "profile.json").read_text())
        )
    except (json.JSONDecodeError, ValidationError) as exc:
        raise ValueError(f"profile.json: {exc}") from exc
    if profile.store_type != store_type:
        raise ValueError(
            f"profile.json store_type {profile.store_type!r} does not match folder {store_type!r}"
        )

    tags_path = type_dir / "tags.json"
    return DictionarySet(
        store_type=store_type,
        profile=profile,
        first_names=load_list(shared / "first_names.json", NameEntry),
        last_names=load_list(shared / "last_names.json", NameEntry),
        geographies=load_list(shared / "geographies.json", GeographyEntry),
        tax_rates=load_list(shared / "tax_rates.json", TaxJurisdictionEntry),
        products=load_list(type_dir / "products.json", ProductEntry),
        brands=load_list(type_dir / "brands.json", ProductBrandEntry),
        tags=load_list(tags_path, ProductTagEntry) if tags_path.exists() else [],
    )

# --- retail_setup/config/generation.py ---
"""Generation settings (utility/config.yaml). Environment settings live in deploy/config/."""

import calendar
from datetime import date, timedelta
from pathlib import Path

import yaml
from pydantic import BaseModel, Field, model_validator


# Minimum number of synthetic customers, regardless of store_count. The churn
# model (fabric/lakehouse/09-ml-churn-prediction.ipynb) needs at least two
# customers in each class (active / churned) to build a train/test split. With
# geography affinity concentrating purchases on nearby customers, very small
# customer pools (e.g. a 1-store demo deriving 1,000) can leave fewer than two
# churned customers and abort training. Flooring the count keeps small-store
# demos trainable. This is a no-op for store_count >= 5, where the derived
# value (store_count * 1000) already exceeds the floor.
MIN_CUSTOMER_COUNT = 5000


def _subtract_months(anchor: date, months: int) -> date:
    """Return the date ``months`` calendar months before ``anchor`` (day clamped)."""

    month_index = anchor.month - 1 - months
    year = anchor.year + month_index // 12
    month = month_index % 12 + 1
    day = min(anchor.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)


class GenerationConfig(BaseModel):
    store_type: str = "supercenter"
    # Preferred input: number of months of history to generate. The window ends
    # *yesterday* so real-time streaming continues seamlessly from today.
    # ``start_date``/``end_date`` may be provided explicitly instead (back-compat);
    # they are derived from ``months`` only when not both already supplied.
    months: int | None = Field(default=None, ge=1, le=120)
    start_date: date | None = None
    end_date: date | None = None
    store_count: int = Field(default=50, gt=0, le=2000)
    seed: int = 42
    silver_db: str = "ag"
    gold_db: str = "au"
    # Optional override for dictionary root; when None, default_dictionary_root() is used.
    # Pass an absolute path string to point at a custom dictionary tree (e.g. a Fabric
    # lakehouse Files mount) without mutating the package data directory.
    dictionary_root: str | None = None

    # scale knobs; None -> derived from store_count in the validator below
    dc_count: int | None = Field(default=None, gt=0, le=999)
    customer_count: int | None = Field(default=None, gt=0)
    # base in-store transactions per store-day at multiplier 1.0; profiles'
    # hourly/daily/monthly weights shape it, store daily_traffic_multiplier scales it
    transactions_per_store_day: int = Field(default=400, gt=0)
    # fraction of SALE receipts returned per day (Dec 26 spikes 6x, capped 10%)
    # nominal daily return share; Dec 26 applies a 6x spike capped at 0.10, so
    # values near the ceiling flatten the spike - keep this small (~0.01)
    return_rate: float = Field(default=0.01, ge=0.0, le=0.10)
    # network-wide online orders per day at multiplier 1.0; None -> store_count * 8
    online_orders_per_day: int | None = Field(default=None, gt=0)
    # number of category-matched branded SKUs generated per base catalog product
    # (datagen combinatorial SKUs); 1 = one SKU per dictionary row
    brands_per_product: int = Field(default=3, ge=1, le=10)
    # truck load capacity in units; a store-day shipment exceeding this is split
    # across multiple truck legs (datagen multi-truck shipments)
    truck_capacity: int = Field(default=15000, gt=0)

    @model_validator(mode="after")
    def _known_store_type(self) -> "GenerationConfig":
        root = Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()
        known = available_store_types(root)
        if self.store_type not in known:
            raise ValueError(f"store_type {self.store_type!r} not found; available: {known}")
        return self

    @property
    def resolved_dictionary_root(self) -> Path:
        """Resolved dictionary root path (explicit override or package default)."""
        return Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()

    @model_validator(mode="after")
    def _derive_date_range(self) -> "GenerationConfig":
        """Derive start/end from ``months`` (window ends yesterday) when needed."""

        if self.months is not None and not (self.start_date and self.end_date):
            end = date.today() - timedelta(days=1)
            self.end_date = end
            self.start_date = _subtract_months(end, self.months)
        if self.start_date is None or self.end_date is None:
            raise ValueError("provide `months`, or both `start_date` and `end_date`")
        return self

    @model_validator(mode="after")
    def _date_order(self) -> "GenerationConfig":
        if self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date")
        return self

    @model_validator(mode="after")
    def _derive_scale_defaults(self) -> "GenerationConfig":
        if self.dc_count is None:
            self.dc_count = max(1, self.store_count // 10)
        if self.customer_count is None:
            self.customer_count = self.store_count * 1000
        # Guarantee enough customers for the churn model's two-class train/test
        # split; see MIN_CUSTOMER_COUNT above. Applied to the final value so an
        # explicitly small customer_count is lifted too.
        self.customer_count = max(self.customer_count, MIN_CUSTOMER_COUNT)
        if self.online_orders_per_day is None:
            self.online_orders_per_day = self.store_count * 8
        return self


def load_generation_config(path: Path) -> GenerationConfig:
    return GenerationConfig.model_validate(yaml.safe_load(path.read_text()))

# --- retail_setup/generation/schemas.py ---
"""Authoritative output schemas for generated tables (Plan 2a + 2b scope).

Spark simple type strings. Dimension columns keep the legacy PascalCase names
because the semantic model TMDL binds sourceColumn to them (e.g. StoreNumber,
Cost, MSRP). Most fact tables are snake_case, but several 2b tables have
TMDL-bound PascalCase or mixed-case columns documented per table below.

The TMDL contract test (tests/generation/test_schema_contract.py) is the
arbiter: every sourceColumn in a table's TMDL must be present here with a
compatible type. Extra columns not in TMDL are allowed.

Columns added vs plan (from TMDL audit 2026-06-12):
  fact_receipts: added ("Subtotal", "string") — legacy trace/aggregate string
    column bound as sourceColumn: Subtotal in the semantic model. Not a
    snake_case column; appears to be a pre-existing legacy column the model
    still references.

  --- Plan 2b TMDL deltas (per-table, added to reconcile TMDL bindings) ---

  fact_ble_pings:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId (PascalCase),
      not customer_id. Plan listed customer_id (kept as extra); TMDL column added.
    - ("__index_level_0__", "long") — TMDL-bound legacy pandas-index column.

  fact_customer_zone_changes:
    - TMDL uses PascalCase for all data columns: StoreID, CustomerBLEId,
      FromZone, ToZone (plan listed snake_case equivalents, kept as extras).
    - ("StoreID", "long"), ("CustomerBLEId", "string"), ("FromZone", "string"),
      ("ToZone", "string") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_marketing:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId; plan's
      customer_id kept as extra.
    - ("CostCents", "long") — TMDL sourceColumn is CostCents; plan's cost_cents
      kept as extra.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_promo_lines:
    - TMDL uses PascalCase: ReceiptId, PromoCode, LineNumber, ProductID, Qty,
      DiscountAmount, DiscountCents (plan listed snake_case equivalents, kept
      as extras).
    - ("ReceiptId", "string"), ("PromoCode", "string"), ("LineNumber", "long"),
      ("ProductID", "long"), ("Qty", "long"), ("DiscountAmount", "string"),
      ("DiscountCents", "long") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_online_order_lines:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_reorders:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_moves:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_inventory:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_dc_inventory_txn:
    - ("Source", "string") — TMDL sourceColumn is PascalCase Source. The
      lowercase ``source`` that appears during construction is renamed to
      ``Source`` in the final select (inventory.py); keeping both would cause
      a case-insensitive collision that Delta on Fabric rejects.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_store_inventory_txn:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_stockouts:
    - ("__index_level_0__", "long") — TMDL-bound.

Columns in plan but NOT in TMDL (allowed — Direct Lake ignores extra columns):
  fact_receipts: trace_id, tender_type
  fact_payments: tender_type (not in payments TMDL at all)
  fact_ble_pings: customer_id (TMDL uses CustomerId), trace_id, event_ts,
    event_date
  fact_customer_zone_changes: store_id, customer_ble_id, from_zone, to_zone,
    event_ts, trace_id, event_date
  fact_marketing: customer_id, cost_cents, event_ts, trace_id, event_date,
    channel (extra beyond TMDL-bound channel — already in TMDL, fine)
  fact_promo_lines: receipt_id_ext, promo_code, line_number, product_id,
    quantity, discount_amount, discount_cents, event_ts, trace_id, event_date
  fact_dc_inventory_txn: trace_id, event_ts, event_date
    (lowercase source is renamed to Source in the final select; not an extra)
"""

# table -> list of (column, spark_type)
TABLES: dict[str, list[tuple[str, str]]] = {
    "dim_geographies": [
        ("ID", "long"),
        ("City", "string"),
        ("State", "string"),
        ("ZipCode", "string"),
        ("District", "string"),
        ("Region", "string"),
    ],
    "dim_stores": [
        ("ID", "long"),
        ("StoreNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("tax_rate", "double"),
        ("volume_class", "string"),
        ("store_format", "string"),
        ("operating_hours", "string"),
        ("daily_traffic_multiplier", "double"),
    ],
    "dim_distribution_centers": [
        ("ID", "long"),
        ("DCNumber", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
    ],
    "dim_trucks": [
        ("ID", "long"),
        ("LicensePlate", "string"),
        ("Refrigeration", "boolean"),
        # double, not long: NULL for pool trucks + Direct Lake nullability
        ("DCID", "double"),
    ],
    "dim_customers": [
        ("ID", "long"),
        ("FirstName", "string"),
        ("LastName", "string"),
        ("Address", "string"),
        ("GeographyID", "long"),
        ("LoyaltyCard", "string"),
        ("Phone", "string"),
        ("BLEId", "string"),
        ("AdId", "string"),
    ],
    "dim_products": [
        ("ID", "long"),
        ("ProductName", "string"),
        ("Brand", "string"),
        ("Company", "string"),
        ("Department", "string"),
        ("Category", "string"),
        ("Subcategory", "string"),
        ("Cost", "double"),
        ("MSRP", "double"),
        ("SalePrice", "double"),
        ("RequiresRefrigeration", "boolean"),
        ("LaunchDate", "timestamp"),
        ("taxability", "string"),
        ("Tags", "string"),
    ],
    "dim_date": [
        ("date_key", "long"),
        ("date", "date"),
        ("year", "long"),
        ("quarter", "long"),
        ("month", "long"),
        ("month_name", "string"),
        ("day", "long"),
        ("day_of_week", "long"),
        ("day_name", "string"),
        ("week_of_year", "long"),
        ("is_weekend", "long"),
        ("fiscal_year", "long"),
        ("fiscal_quarter", "long"),
    ],
    "fact_receipts": [
        ("receipt_id_ext", "string"),
        ("trace_id", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("receipt_type", "string"),
        ("tender_type", "string"),
        ("subtotal_cents", "long"),
        ("discount_amount", "string"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        # Legacy column bound by the semantic model (sourceColumn: Subtotal).
        # Present in fact_receipts.tmdl as a string column; added here to
        # satisfy the TMDL contract test (TMDL arbiter rule).
        ("Subtotal", "string"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        # gross_subtotal_cents = subtotal_cents (net, unchanged) + discount_cents;
        # discount before tax is preserved (subtotal_cents stays the net amount).
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        # NULL unless this SALE receipt was selected for marketing attribution.
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    "fact_receipt_lines": [
        ("receipt_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("line_num", "int"),
        ("product_id", "long"),
        ("quantity", "int"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
    ],
    "fact_payments": [
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("payment_method", "string"),
        ("amount_cents", "long"),
        ("amount", "string"),
        ("transaction_id", "string"),
        ("status", "string"),
        ("decline_reason", "string"),
        ("processing_time_ms", "long"),
        ("store_id", "long"),
        ("customer_id", "long"),
        # IMP-007: NULL unless this payment's receipt/order was attributed.
        ("attribution_journey_id", "string"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2b: 15 remaining fact tables
    # -----------------------------------------------------------------------
    "fact_store_ops": [
        # TMDL-bound: trace_id, store_id (int64→long), operation_type, event_date
        # Extra (not in TMDL, allowed): event_ts
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("operation_type", "string"),
        ("event_date", "date"),
    ],
    "fact_foot_traffic": [
        # TMDL-bound: count (int64→long), zone, dwell_seconds (int64→long),
        #   store_id (int64→long), sensor_id, event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("sensor_id", "string"),
        ("zone", "string"),
        ("dwell_seconds", "long"),
        ("count", "long"),
        ("event_date", "date"),
    ],
    "fact_ble_pings": [
        # TMDL-bound: beacon_id, rssi (int64→long), customer_ble_id, zone,
        #   store_id (int64→long), CustomerId (double), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("beacon_id", "string"),
        ("customer_ble_id", "string"),
        ("customer_id", "double"),
        # TMDL-bound PascalCase column (sourceColumn: CustomerId)
        ("CustomerId", "double"),
        ("rssi", "long"),
        ("zone", "string"),
        ("event_date", "date"),
        # TMDL-bound legacy pandas-index column
        ("__index_level_0__", "long"),
    ],
    "fact_customer_zone_changes": [
        # TMDL-bound PascalCase: StoreID (int64→long), CustomerBLEId (string),
        #   FromZone (string), ToZone (string), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   store_id, customer_ble_id, from_zone, to_zone (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("store_id", "long"),
        ("customer_ble_id", "string"),
        ("from_zone", "string"),
        ("to_zone", "string"),
        # TMDL-bound PascalCase columns
        ("StoreID", "long"),
        ("CustomerBLEId", "string"),
        ("FromZone", "string"),
        ("ToZone", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_marketing": [
        # TMDL-bound: device, cost, customer_ad_id, impression_id_ext, campaign_id,
        #   creative_id, channel, CustomerId (double), CostCents (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId),
        #   cost_cents (plan name; TMDL uses CostCents)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("channel", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("customer_ad_id", "string"),
        # snake_case plan columns (extras)
        ("customer_id", "double"),
        ("cost_cents", "long"),
        # TMDL-bound PascalCase columns (sourceColumn: CustomerId, CostCents)
        ("CustomerId", "double"),
        ("CostCents", "long"),
        ("impression_id_ext", "string"),
        ("cost", "string"),
        ("device", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # IMP-007: NULL for organic/background impressions; set on the two
        # synthetic touches created for a selected purchase's journey.
        ("attribution_journey_id", "string"),
    ],
    "fact_promotions": [
        # TMDL-bound: receipt_id_ext, promo_code, discount_amount, discount_cents
        #   (int64→long), discount_type, product_count (int64→long), product_ids,
        #   store_id (int64→long), customer_id (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        ("discount_type", "string"),
        ("product_count", "long"),
        ("product_ids", "string"),
        ("store_id", "long"),
        ("customer_id", "long"),
        ("event_date", "date"),
        # IMP-007: NULL unless the underlying receipt was attributed.
        ("attribution_journey_id", "string"),
    ],
    "fact_promo_lines": [
        # TMDL-bound PascalCase: ReceiptId, PromoCode, LineNumber (int64→long),
        #   ProductID (int64→long), Qty (int64→long), DiscountAmount (string),
        #   DiscountCents (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   receipt_id_ext, promo_code, line_number, product_id, quantity,
        #   discount_amount, discount_cents (snake_case plan names)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("receipt_id_ext", "string"),
        ("promo_code", "string"),
        ("line_number", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("discount_amount", "string"),
        ("discount_cents", "long"),
        # TMDL-bound PascalCase columns
        ("ReceiptId", "string"),
        ("PromoCode", "string"),
        ("LineNumber", "long"),
        ("ProductID", "long"),
        ("Qty", "long"),
        ("DiscountAmount", "string"),
        ("DiscountCents", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_online_order_headers": [
        # TMDL-bound: customer_id (int64→long), subtotal_cents (int64→long),
        #   order_id_ext, payment_method, total_amount, total_cents (int64→long),
        #   tax_amount, subtotal_amount, tax_cents (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts
        ("order_id_ext", "string"),
        ("customer_id", "long"),
        ("subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("subtotal_amount", "string"),
        ("tax_amount", "string"),
        ("total_amount", "string"),
        ("payment_method", "string"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        # --- IMP-007 marketing attribution (additive, snake_case) ---
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("attribution_journey_id", "string"),
        ("campaign_id", "string"),
        ("impression_id_ext", "string"),
    ],
    # -----------------------------------------------------------------------
    # IMP-007: dedicated marketing-attribution fact. One row per in-store SALE
    # receipt or online order, including attributed, unattributed, declined,
    # and reconciliation-failed outcomes (see attribution.py).
    # -----------------------------------------------------------------------
    "fact_marketing_attribution": [
        ("attribution_id", "string"),
        ("attribution_journey_id", "string"),
        ("attribution_status", "string"),
        ("attribution_model", "string"),
        ("attribution_window_days", "int"),
        ("impression_id_ext", "string"),
        ("campaign_id", "string"),
        ("creative_id", "string"),
        ("channel", "string"),
        ("customer_ad_id", "string"),
        ("customer_id", "long"),
        ("touch_ts", "timestamp"),
        ("purchase_ts", "timestamp"),
        ("lag_seconds", "long"),
        ("purchase_type", "string"),
        ("receipt_id_ext", "string"),
        ("order_id_ext", "string"),
        ("store_id", "long"),
        ("gross_subtotal_cents", "long"),
        ("discount_cents", "long"),
        ("net_subtotal_cents", "long"),
        ("tax_cents", "long"),
        ("total_cents", "long"),
        ("payment_cents", "long"),
        ("attributed_revenue_cents", "long"),
        ("event_date", "date"),
    ],
    "fact_online_order_lines": [
        # TMDL-bound: order_id, unit_cents (int64→long), fulfillment_mode,
        #   promo_code, node_type, product_id (int64→long), ext_price, ext_cents
        #   (int64→long), line_num (int64→long), fulfillment_status, unit_price,
        #   quantity (int64→long), node_id (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): picked_ts, shipped_ts, delivered_ts,
        #   event_ts, event_date
        ("order_id", "string"),
        ("product_id", "long"),
        ("line_num", "long"),
        ("quantity", "long"),
        ("unit_price", "string"),
        ("unit_cents", "long"),
        ("ext_price", "string"),
        ("ext_cents", "long"),
        ("promo_code", "string"),
        ("fulfillment_mode", "string"),
        ("fulfillment_status", "string"),
        ("node_type", "string"),
        ("node_id", "long"),
        ("picked_ts", "timestamp"),
        ("shipped_ts", "timestamp"),
        ("delivered_ts", "timestamp"),
        ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_reorders": [
        # TMDL-bound: store_id (int64→long), dc_id (int64→long), product_id
        #   (int64→long), current_quantity (int64→long), reorder_quantity
        #   (int64→long), reorder_point (int64→long), priority,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("current_quantity", "long"),
        ("reorder_quantity", "long"),
        ("reorder_point", "long"),
        ("priority", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_moves": [
        # TMDL-bound: status, shipment_id, dc_id (int64→long), truck_id
        #   (int64→long), store_id (int64→long), actual_unload_duration (double),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, eta, etd,
        #   departure_time, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("dc_id", "long"),
        ("store_id", "long"),
        ("shipment_id", "string"),
        ("status", "string"),
        ("eta", "timestamp"),
        ("etd", "timestamp"),
        ("departure_time", "timestamp"),
        ("actual_unload_duration", "double"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_inventory": [
        # TMDL-bound: truck_id (int64→long), shipment_id, product_id (int64→long),
        #   quantity (int64→long), action, location_id (int64→long), location_type,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("truck_id", "long"),
        ("shipment_id", "string"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("action", "string"),
        ("location_id", "long"),
        ("location_type", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_dc_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), dc_id (int64→long),
        #   balance (int64→long), product_id (int64→long), Source (PascalCase!),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   source (lowercase snake_case plan name)
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("dc_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        # TMDL-bound PascalCase column (sourceColumn: Source). The lowercase
        # ``source`` used during construction is renamed AS "Source" in
        # inventory.py before the final select; keeping both would produce a
        # case-insensitive duplicate that Delta on Fabric rejects.
        ("Source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_store_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), balance (int64→long),
        #   product_id (int64→long), store_id (int64→long), source (lowercase OK),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("store_id", "long"),
        ("product_id", "long"),
        ("quantity", "long"),
        ("balance", "long"),
        ("txn_type", "string"),
        ("source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
        # Attempted demand is retained even when a whole line cannot be filled.
        # LOST_SALE changes no stock (quantity=0); all non-demand rows use zero.
        ("requested_quantity", "long"),
        ("unfulfilled_quantity", "long"),
    ],
    "fact_stockouts": [
        # PascalCase IS the contract: TMDL binds StoreID (double), DCID (double),
        #   ProductID (int64→long), LastKnownQuantity (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"),
        ("trace_id", "string"),
        ("StoreID", "double"),
        ("DCID", "double"),
        ("ProductID", "long"),
        ("LastKnownQuantity", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2c: 10 Gold (au) aggregate tables
    # TMDL audit 2026-06-12: bindings match exactly; `computed_at` and `as_of`
    # are produced by the legacy transforms but unbound in TMDL (extras OK).
    # TMDL `day`/`ts` are dateTime — TYPE_COMPAT accepts timestamp|date.
    # -----------------------------------------------------------------------
    "sales_minute_store": [
        ("store_id", "long"),
        ("ts", "timestamp"),
        ("total_sales", "double"),
        ("receipts", "long"),
        ("avg_basket", "double"),
    ],
    "top_products_15m": [
        ("product_id", "long"),
        ("revenue", "double"),
        ("units", "long"),
        ("computed_at", "timestamp"),  # window end; bound as 'Window End' in TMDL
    ],
    "inventory_position_current": [
        ("store_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "dc_inventory_position_current": [
        ("dc_id", "long"),
        ("product_id", "long"),
        ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "truck_dwell_daily": [
        ("site", "string"),
        ("day", "date"),
        ("avg_dwell_min", "double"),
        ("trucks", "long"),
    ],
    "online_sales_daily": [
        ("day", "date"),
        ("orders", "long"),
        ("subtotal", "double"),
        ("tax", "double"),
        ("total", "double"),
        ("avg_order_value", "double"),
    ],
    "zone_dwell_minute": [
        ("store_id", "long"),
        ("zone", "string"),
        ("ts", "timestamp"),
        ("avg_dwell", "double"),
        ("customers", "long"),
    ],
    "marketing_cost_daily": [
        ("campaign_id", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("cost", "double"),
    ],
    "campaign_performance_daily": [
        ("campaign_id", "string"),
        ("channel", "string"),
        ("day", "date"),
        ("impressions", "long"),
        ("spend_cents", "long"),
        ("conversions", "long"),
        ("attributed_revenue_cents", "long"),
        ("discount_cents", "long"),
        ("tax_cents", "long"),
        ("payment_cents", "long"),
        ("conversion_rate", "double"),
        ("roas", "double"),
    ],
    "tender_mix_daily": [
        ("day", "date"),
        ("payment_method", "string"),
        ("transactions", "long"),
        ("total_amount", "double"),
    ],
}


_SPARK_TYPE_MAP = None


def _type_map():
    """Lazy-import PySpark type map (avoids import cost when pyspark not installed)."""
    global _SPARK_TYPE_MAP
    if _SPARK_TYPE_MAP is None:
        from pyspark.sql.types import (
            BooleanType,
            DateType,
            DoubleType,
            IntegerType,
            LongType,
            StringType,
            TimestampType,
        )

        _SPARK_TYPE_MAP = {
            "long": LongType(),
            "int": IntegerType(),
            "string": StringType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType(),
            "date": DateType(),
        }
    return _SPARK_TYPE_MAP


def spark_schema(table: str):
    """Build a StructType for createDataFrame with explicit types."""
    from pyspark.sql.types import StructField, StructType

    tmap = _type_map()
    fields = [StructField(name, tmap[typ], nullable=True) for name, typ in TABLES[table]]
    return StructType(fields)


def column_names(table: str) -> list[str]:
    return [name for name, _ in TABLES[table]]

# --- retail_setup/generation/runtime.py ---
"""Deterministic seeding + partition grids for the generation engine."""

import hashlib
import math
from datetime import date

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Spark's default of 200 shuffle partitions leaves billion-row full-demo shuffles
# (receipt lines, inventory events, BLE pings) at millions of rows per task: long
# spilling sorts and too few tasks to occupy a large pool. Target roughly one
# million rows of the dominant stream per partition; adaptive query execution
# still coalesces the many small shuffles back to its advisory size.
SHUFFLE_ROWS_PER_PARTITION = 1_000_000
MIN_SHUFFLE_PARTITIONS = 200
MAX_SHUFFLE_PARTITIONS = 8192


def shuffle_partitions_for(cfg: GenerationConfig, basket_lambda: float = 12.0) -> int:
    """Shuffle partition count sized to the configured receipt-line volume.

    Rounded up to a multiple of 64 so tasks spread evenly across executors.
    Generation output is independent of the partition count.
    """
    days = (cfg.end_date - cfg.start_date).days + 1
    lines = cfg.store_count * days * cfg.transactions_per_store_day * basket_lambda
    wanted = math.ceil(lines / SHUFFLE_ROWS_PER_PARTITION / 64) * 64
    return max(MIN_SHUFFLE_PARTITIONS, min(MAX_SHUFFLE_PARTITIONS, wanted))


class seeded_draws:
    """Deterministic draw expressions bound to a seed.

    u(cols, salt)        -> uniform [0,1) double
    gauss(cols, salt)    -> N(0,1) (Box-Muller)
    negative_binomial(cols, salt, mean, shape) -> count, variance mean + mean²/shape
    h64(cols, salt)      -> non-negative long hash
    pick_by_weights(cols, salt, [(value, weight), ...]) -> weighted categorical
    """

    _U_MOD = 10**12

    def __init__(self, seed: int):
        self.seed = seed

    def h64(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return F.pmod(F.xxhash64(*cols, F.lit(f"{salt}|{self.seed}")), F.lit(2**62))

    def u(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return (self.h64(cols, salt) % F.lit(self._U_MOD)) / F.lit(float(self._U_MOD))

    def gauss(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        radius = F.sqrt(-2.0 * F.log(F.lit(1.0) - self.u(cols, f"{salt}|g1")))
        return radius * F.cos(F.lit(6.283185307179586) * self.u(cols, f"{salt}|g2"))

    def negative_binomial(self, cols: list, salt: str, mean: Column, shape: int = 12) -> Column:
        """Exact sum of geometric draws; integer shape controls overdispersion."""
        if shape < 1:
            raise ValueError("negative-binomial shape must be positive")
        valid_mean = F.when((mean >= 0) & (mean < float("inf")), mean).otherwise(
            F.raise_error("negative-binomial mean must be finite and non-negative")
        )
        probability = valid_mean / (valid_mean + F.lit(float(shape)))
        draws = [
            F.when(valid_mean == 0, F.lit(0))
            .otherwise(
                F.floor(F.log(F.lit(1.0) - self.u(cols, f"{salt}|nb{i}")) / F.log(probability))
            )
            .cast("long")
            for i in range(shape)
        ]
        return sum(draws, F.lit(0).cast("long"))

    def pick_by_weights(self, cols: list, salt: str, weighted: list[tuple[str, float]]):
        from pyspark.sql import functions as F

        total = sum(w for _, w in weighted)
        uu = self.u(cols, salt)
        expr, acc = None, 0.0
        for value, w in weighted[:-1]:
            acc += w / total
            expr = expr.when(uu < acc, value) if expr is not None else F.when(uu < acc, value)
        return expr.otherwise(weighted[-1][0]) if expr is not None else F.lit(weighted[0][0])


def derive_seed(global_seed: int, section: str, key: int, day: date) -> int:
    """Stable 31-bit seed from (global_seed, section, key, day).

    Independent of Spark partitioning/execution order — safe for per-row
    or per-group RNG seeding.
    """
    payload = f"{global_seed}|{section}|{key}|{day.isoformat()}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") % (2**31)


def legacy_index(*key_cols):
    """Deterministic long for the legacy __index_level_0__ pandas column.

    The semantic model binds the column by name only; nothing consumes its
    values, so a hash beats a dense row_number — which forced a
    single-partition global sort at full volume.
    """
    from pyspark.sql import functions as F

    return F.pmod(F.xxhash64(*key_cols, F.lit("__legacy_index__")), F.lit(2**62))


def fixed_width_decimal(value: Column, width: int, label: str) -> Column:
    """Zero-pad a non-negative decimal value and fail if it exceeds ``width``."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    invalid = (value < F.lit(0)) | (F.length(text) > F.lit(width))
    error = F.raise_error(F.lit(f"{label} exceeds {width} digits"))
    return (
        F.when(value.isNull(), F.lit(None).cast("string"))
        .when(invalid, error)
        .otherwise(F.lpad(text, width, "0"))
    )


def minimum_width_decimal(value: Column, width: int) -> Column:
    """Zero-pad to at least ``width`` digits without truncating larger values."""
    return minimum_width_text(value.cast("string"), width)


def minimum_width_text(value: Column, width: int) -> Column:
    """Zero-pad text to at least ``width`` characters without truncation."""
    if width < 1:
        raise ValueError("width must be positive")
    text = value.cast("string")
    return F.when(F.length(text) >= width, text).otherwise(F.lpad(text, width, "0"))


def date_grid(
    spark: SparkSession,
    start: date,
    end: date,
    column: str = "day",
) -> DataFrame:
    """Inclusive Spark-native date range without a driver-materialized row list."""
    if end < start:
        raise ValueError("end must be on or after start")
    days = (end - start).days + 1
    return spark.range(days).select(F.date_add(F.lit(start), F.col("id").cast("int")).alias(column))


def store_day_grid(
    spark: SparkSession,
    store_ids: list[int],
    start: date,
    end: date,
    global_seed: int,
    section: str,
) -> DataFrame:
    """store_id x day cross grid.

    Only the small store-ID set is materialized on the driver. Dates and the
    Cartesian expansion remain Spark-native, so long histories do not create a
    large Python row list. Random draws derive from stable row keys rather than
    partition order.
    """
    del global_seed, section
    store_values = F.array(*[F.lit(int(store_id)).cast("long") for store_id in store_ids])
    stores = spark.range(len(store_ids)).select(
        F.element_at(store_values, F.col("id").cast("int") + 1).alias("store_id")
    )
    return stores.crossJoin(date_grid(spark, start, end))

# --- retail_setup/generation/behavior.py ---
"""Shared historical/live shopping behavior, expressed as deterministic Spark columns."""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def customer_visit_tickets(draws: seeded_draws, customer: Column) -> Column:
    """Persistent dimension-sized weights for heterogeneous shopping frequency."""
    return F.ceil(4.0 * F.exp(0.9 * draws.gauss([customer], "visit-propensity") - 0.405)).cast(
        "int"
    )


def repeat_purchase_probability(department: Column) -> Column:
    """Consumables permit multi-unit purchases more often than durable goods."""
    consumable = F.lower(department).rlike(
        "grocery|fresh|food|household|personal|health|beauty|pet|baby"
    )
    return F.when(consumable, F.lit(0.22)).otherwise(F.lit(0.04))


def retail_quantity(draws: seeded_draws, keys: list, department: Column) -> Column:
    return (
        F.lit(1)
        + F.floor(
            F.log(F.lit(1.0) - draws.u(keys, "retail-quantity"))
            / F.log(repeat_purchase_probability(department))
        )
    ).cast("int")


def retail_basket_size(
    draws: seeded_draws,
    purchase_keys: list,
    customer: Column,
    mean: Column,
    offer: Column,
    promo_rate: float,
) -> Column:
    """Mission mixtures with a calibrated mean and persistent customer budgets."""
    trip = draws.u(purchase_keys, "retail-mission")
    mission = (
        F.when(trip < 0.30, 0.30).when(trip < 0.70, 0.90).when(trip < 0.90, 1.60).otherwise(2.50)
    ) / F.lit(1.02)
    budget = F.exp(0.35 * draws.gauss([customer], "retail-budget") - F.lit(0.35**2 / 2))
    # Exposure is decided before demand: coupons increase additional-item demand.
    lift = (F.lit(1.0) + offer.cast("double") * 0.20) / (1.0 + promo_rate * 0.20)
    extra = F.greatest(mean - 1.0, F.lit(0.0)) * mission * budget * lift
    return (1 + draws.negative_binomial(purchase_keys, "retail-basket", extra, 3)).cast("int")


def retail_catalog(products: DataFrame, dates: DataFrame, draws: seeded_draws) -> DataFrame:
    """One launch-eligible weighted product CDF per day and department.

    Popularity is persistent per SKU, with price sensitivity and a long tail.
    Arrays keep selection a bounded lookup rather than a receipt x catalog join.
    """
    eligible = (
        dates.select("event_date")
        .distinct()
        .join(F.broadcast(products), F.col("launch_date") <= F.col("event_date"))
    )
    eligible = eligible.withColumn(
        "_popularity",
        F.exp(0.8 * draws.gauss(["product_id"], "retail-popularity") - 0.32)
        / F.pow(F.col("SalePrice"), 1.15),
    )
    group = Window.partitionBy("event_date", "department")
    ordered = group.orderBy("product_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    eligible = (
        eligible.withColumn("_total_weight", F.sum("_popularity").over(group))
        .withColumn("_probability", F.col("_popularity") / F.col("_total_weight"))
        .withColumn("_cdf", F.sum("_probability").over(ordered))
        .withColumn(
            "_unit_value",
            F.col("SalePrice") / (1.0 - repeat_purchase_probability(F.col("department"))),
        )
    )
    return eligible.groupBy("event_date", "department").agg(
        F.sort_array(
            F.collect_list(F.struct("_cdf", "product_id", "SalePrice", "taxability"))
        ).alias("_catalog"),
        F.sum(F.col("_probability") * F.col("_unit_value")).alias("_expected_line_value"),
        F.sum(
            F.col("_probability")
            * F.col("_unit_value")
            * F.when(F.col("taxability") == "TAXABLE", 1.0)
            .when(F.col("taxability") == "REDUCED_RATE", 0.5)
            .otherwise(0.0)
        ).alias("_expected_taxable_value"),
    )


def pick_retail_product(catalog: Column, uniform: Column) -> Column:
    """Binary CDF search avoids scanning the catalog for every basket line."""
    size = F.size(catalog)
    initial = F.struct(F.lit(1).alias("lo"), size.alias("hi"))

    def narrow(bounds: Column, _step: Column) -> Column:
        middle = F.floor((bounds["lo"] + bounds["hi"]) / 2).cast("int")
        left = F.element_at(catalog, middle)["_cdf"] > uniform
        return F.struct(
            F.when(left, bounds["lo"]).otherwise(F.least(middle + 1, size)).alias("lo"),
            F.when(left, middle).otherwise(bounds["hi"]).alias("hi"),
        )

    steps = F.sequence(F.lit(1), (F.ceil(F.log2(size)) + 1).cast("int"))
    selected = F.aggregate(steps, initial, narrow)["lo"]
    # The final entry absorbs floating-point CDF accumulation error.
    return F.when(size > 0, F.element_at(catalog, selected)).otherwise(
        F.raise_error("Cannot select a product from an empty catalog")
    )


def calibrated_basket_mean(
    expected_line: Column,
    expected_taxable: Column,
    tax_rate: Column,
    target: float,
    promo_rate: float,
    nominal_lines: float,
) -> Column:
    affordable_lines = F.lit(target) / (
        (expected_line + expected_taxable * tax_rate) * (1.0 - promo_rate * 0.20)
    )
    return F.when(
        (expected_line > 0)
        & (expected_line < float("inf"))
        & (expected_taxable >= 0)
        & (tax_rate >= 0),
        F.least(affordable_lines, F.lit(nominal_lines)),
    ).otherwise(F.raise_error("Basket calibration requires valid catalog prices and tax"))

# --- retail_setup/generation/traffic.py ---
"""Visit-first store traffic shared by receipt generation and footfall facts."""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def generate_hourly_visits(
    spark: SparkSession,
    stores: DataFrame,
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> DataFrame:
    draws = seeded_draws(cfg.seed)
    hours = stores.select(
        F.col("ID").alias("store_id"),
        "GeographyID",
        "daily_traffic_multiplier",
        "operating_hours",
    )
    is_24 = F.col("operating_hours") == "24h"
    parts = F.split("operating_hours", "-")
    hours = hours.withColumn(
        "hour",
        F.explode(
            F.sequence(
                F.when(is_24, 0).otherwise(parts.getItem(0).cast("int")),
                F.when(is_24, 24).otherwise(parts.getItem(1).cast("int")) - 1,
            )
        ),
    ).withColumn(
        "_hour_weight",
        F.element_at(F.array(*[F.lit(w) for w in profile.hourly_weights]), F.col("hour") + 1),
    )
    hours = hours.withColumn(
        "_hour_share",
        F.col("_hour_weight") / F.sum("_hour_weight").over(Window.partitionBy("store_id")),
    )
    days = date_grid(spark, cfg.start_date, cfg.end_date).filter(
        ~((F.month("day") == 12) & (F.dayofmonth("day") == 25))
    )
    grid = hours.crossJoin(days)
    daily = F.element_at(
        F.array(*[F.lit(w * 7 / sum(profile.daily_weights)) for w in profile.daily_weights]),
        F.pmod(F.dayofweek("day") + 5, F.lit(7)) + 1,
    )
    monthly = F.element_at(
        F.array(*[F.lit(w * 12 / sum(profile.monthly_weights)) for w in profile.monthly_weights]),
        F.month("day"),
    )
    # Nearby stores share weather; store/hour conversion remains heterogeneous.
    weather = F.exp(0.15 * draws.gauss(["GeographyID", "day"], "weather") - 0.15**2 / 2)
    conversion = 1.0 / (
        1.0
        + F.exp(
            F.lit(1.2657)
            - 0.25 * draws.gauss(["store_id"], "conversion-store")
            - 0.20 * draws.gauss(["store_id", "day", "hour"], "conversion-hour")
        )
    )
    grid = grid.withColumn(
        "_visits_mean",
        F.lit(cfg.transactions_per_store_day / 0.225)
        * daily
        * monthly
        * weather
        * F.col("daily_traffic_multiplier")
        * F.col("_hour_share"),
    ).withColumn("_conversion", conversion)
    grid = grid.withColumn(
        "visitors",
        draws.negative_binomial(["store_id", "day", "hour"], "visitors", F.col("_visits_mean"), 24),
    )
    visitor_ids = F.when(
        F.col("visitors") > 0, F.sequence(F.lit(1).cast("long"), F.col("visitors"))
    ).otherwise(F.array().cast("array<long>"))
    grid = grid.withColumn(
        "n_receipts",
        F.aggregate(
            visitor_ids,
            F.lit(0).cast("long"),
            lambda count, visitor: (
                count
                + (
                    draws.u(["store_id", "day", "hour", visitor], "visit-conversion")
                    < F.col("_conversion")
                ).cast("long")
            ),
        ),
    )
    return grid.withColumn(
        "hour_ts",
        F.timestamp_seconds(
            F.unix_timestamp(F.col("day").cast("timestamp")) + F.col("hour") * 3600
        ),
    ).select("store_id", "day", "hour", "hour_ts", "visitors", "n_receipts")

# --- retail_setup/generation/dims.py ---
"""Dimension generation using bounded driver metadata and Spark-native rows.

Semantics ported from datagen master_generators (ID schemes, tax lookup,
pricing rules); column names/types from schemas.TABLES, which the TMDL
contract test guards.
"""

from datetime import date, datetime, timedelta

import numpy as np
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# datagen StoreProfiler equivalents (volume class -> traffic multiplier range).
# Five tiers incl. KIOSK, matching datagen's StoreVolumeClass (the kiosk tier
# was dropped in the first port; restored here for store-volume variety).
VOLUME_CLASSES = [
    ("flagship", 0.05, (1.8, 2.5)),
    ("high_volume", 0.15, (1.3, 1.8)),
    ("standard", 0.55, (0.8, 1.3)),
    ("low_volume", 0.20, (0.4, 0.8)),
    ("kiosk", 0.05, (0.25, 0.35)),
]
# Five formats incl. EXPRESS (datagen's StoreFormat.EXPRESS). store_activity
# carries a matching express zone-share row.
STORE_FORMATS = ["hypermarket", "superstore", "standard", "neighborhood", "express"]
VOLUME_FORMATS = {
    "flagship": ["hypermarket", "superstore"],
    "high_volume": ["hypermarket", "superstore", "standard"],
    "standard": ["superstore", "standard", "neighborhood"],
    "low_volume": ["standard", "neighborhood", "express"],
    "kiosk": ["express"],
}
OPERATING_HOURS = ["6-22", "7-22", "7-23", "24h"]
DEFAULT_TAX_RATE = 0.07407  # datagen receipts_mixin fallback
REFRIGERATED_CATEGORIES = {
    "Produce",
    "Dairy & Eggs",
    "Dairy & Alternatives",
    "Meat & Poultry",
    "Meat & Seafood",
    "Seafood",
    "Frozen",
}

# Share of customers placed in a geography that has a store (the rest are
# scattered anywhere). Restores datagen's "customers live near stores" locality
# so receipts can draw a same-geography customer most of the time.
CUSTOMER_HOME_AFFINITY = 0.70

# Map a product department/category token to a brand-dictionary Category so a
# product's brand fits its department (datagen product_generator parity: a
# grocery item never carries a hardware brand). Tokens are matched lowercased;
# substring matches (e.g. "home" in "home & garden") are tried before falling
# back to the whole brand pool.
_BRAND_CAT_SYNONYMS = {
    "grocery": "food",
    "fresh": "food",
    "pantry": "food",
    "snacks": "food",
    "beverages": "food",
    "health & beauty": "health",
    "home & garden": "home",
    "office supplies": "office",
    "pet supplies": "pet",
    "sports & recreation": "sports",
    "apparel": "clothing",
}


def _match_brand_category(key: str, brands_by_cat: dict[str, list]) -> list | None:
    """Resolve a product department/category to a brand pool, or None."""
    k = key.strip().lower()
    if k in brands_by_cat:
        return brands_by_cat[k]
    syn = _BRAND_CAT_SYNONYMS.get(k)
    if syn and syn in brands_by_cat:
        return brands_by_cat[syn]
    for cat, pool in brands_by_cat.items():
        if cat and (cat in k or k in cat):
            return pool
    return None


def compute_pricing(base_price: float, rng: np.random.Generator) -> tuple[float, float, float]:
    """Return ``(cost, msrp, sale_price)`` for a base price.

    Ported from datagen ``PricingCalculator`` (shared/validators/pricing.py):

    - MSRP = BasePrice +/-15%
    - SalePrice = MSRP (60% of the time) OR MSRP discounted 5-35% (40%)
    - Cost = 50-85% of SalePrice

    Always guarantees ``Cost < SalePrice <= MSRP``.
    """
    # A positive cost and a strict one-cent margin require a minimum sale price
    # of two cents. This keeps the documented Cost < SalePrice invariant even
    # for adversarial one-cent source prices.
    msrp = max(0.02, round(base_price * (1.0 + rng.uniform(-0.15, 0.15)), 2))
    if rng.random() < 0.60:
        sale = msrp
    else:
        sale = max(0.02, round(msrp * (1.0 - rng.uniform(0.05, 0.35)), 2))
    cost = round(sale * rng.uniform(0.50, 0.85), 2)
    cost = max(0.01, min(cost, round(sale - 0.01, 2)))
    return float(cost), float(msrp), float(sale)


def _addr(rng: np.random.Generator) -> str:
    return (
        f"{int(rng.integers(100, 9999))} "
        f"{str(rng.choice(['Main', 'Oak', 'Maple', 'Market', 'Commerce', 'Liberty']))} "
        f"{str(rng.choice(['St', 'Ave', 'Blvd', 'Rd']))}"
    )


def generate_dimensions(
    spark: SparkSession, dicts: DictionarySet, cfg: GenerationConfig
) -> dict[str, DataFrame]:
    rng = np.random.default_rng(derive_seed(cfg.seed, "dims", 0, cfg.start_date))
    out: dict[str, DataFrame] = {}

    # --- geographies: sample from dictionary, sequential IDs
    n_geo = min(len(dicts.geographies), max(cfg.store_count * 2, cfg.dc_count * 2, 20))
    geo_idx = rng.choice(len(dicts.geographies), size=n_geo, replace=False)
    geos = [dicts.geographies[i] for i in geo_idx]
    geo_rows = [(i + 1, g.City, g.State, g.Zip, g.District, g.Region) for i, g in enumerate(geos)]
    out["dim_geographies"] = spark.createDataFrame(geo_rows, spark_schema("dim_geographies"))

    # tax lookup hierarchy (datagen TaxCalculator parity):
    #   exact (State, City) -> (State, County) average -> State average -> default.
    # County is resolved from the tax dictionary; geographies carry no county of
    # their own, so the county tier only applies to cities present in the dict.
    city_acc: dict[tuple[str, str], list[float]] = {}
    city_county: dict[tuple[str, str], str] = {}
    county_acc: dict[tuple[str, str], list[float]] = {}
    state_rates: dict[str, list[float]] = {}
    for t in dicts.tax_rates:
        rate = float(t.CombinedRate)
        city_acc.setdefault((t.StateCode, t.City), []).append(rate)
        city_county[(t.StateCode, t.City)] = t.County
        county_acc.setdefault((t.StateCode, t.County), []).append(rate)
        state_rates.setdefault(t.StateCode, []).append(rate)
    by_city = {k: float(np.mean(v)) for k, v in city_acc.items()}
    by_county = {k: float(np.mean(v)) for k, v in county_acc.items()}

    def tax_for(state: str, city: str) -> float:
        if (state, city) in by_city:
            return by_city[(state, city)]
        county = city_county.get((state, city))
        if county is not None and (state, county) in by_county:
            return by_county[(state, county)]
        if state in state_rates:
            return float(np.mean(state_rates[state]))
        return DEFAULT_TAX_RATE

    # --- DCs first (datagen: stores constrained to DC states)
    dc_geo_idx = rng.choice(n_geo, size=cfg.dc_count, replace=cfg.dc_count > n_geo)
    dc_rows = [
        (i + 1, f"DC{i + 1:03d}", _addr(rng), int(dc_geo_idx[i]) + 1) for i in range(cfg.dc_count)
    ]
    out["dim_distribution_centers"] = spark.createDataFrame(
        dc_rows, spark_schema("dim_distribution_centers")
    )

    # --- stores in DC states
    dc_states = {geos[int(g)].State for g in dc_geo_idx}
    eligible = [i for i, g in enumerate(geos) if g.State in dc_states] or list(range(n_geo))
    store_rows = []
    store_geo_indices: list[int] = []
    for sid in range(1, cfg.store_count + 1):
        gi = int(rng.choice(eligible))
        store_geo_indices.append(gi)
        g = geos[gi]
        classes, probs = zip(*[(c, p) for c, p, _ in VOLUME_CLASSES])
        vc = str(rng.choice(classes, p=probs))
        lo, hi = next(r for c, _, r in VOLUME_CLASSES if c == vc)
        store_rows.append(
            (
                sid,
                f"S{sid:06d}",
                _addr(rng),
                gi + 1,
                tax_for(g.State, g.City),
                vc,
                str(rng.choice(VOLUME_FORMATS[vc])),
                str(rng.choice(OPERATING_HOURS)),
                float(np.round(rng.uniform(lo, hi), 2)),
            )
        )
    out["dim_stores"] = spark.createDataFrame(store_rows, spark_schema("dim_stores"))

    # --- trucks: 85% assigned round-robin to DCs, 15% pool (DCID NULL); 40% refrigerated
    n_trucks = max(cfg.dc_count * 3, 6)
    n_assigned = int(round(n_trucks * 0.85))
    truck_rows = []
    for tid in range(1, n_trucks + 1):
        plate = f"TRK{tid:04d}{chr(65 + tid % 26)}"
        refrig = bool(rng.random() < 0.4)
        dcid = float((tid - 1) % cfg.dc_count + 1) if tid <= n_assigned else None
        truck_rows.append((tid, plate, refrig, dcid))
    out["dim_trucks"] = spark.createDataFrame(truck_rows, spark_schema("dim_trucks"))

    # --- customers; use Spark range + deterministic hashes so customer scale
    #     does not materialize one Python tuple per customer on the driver.
    #     ~70% are placed in a store's geography (datagen home-store locality).
    first = [n.Name for n in dicts.first_names]
    last = [n.Name for n in dicts.last_names]
    customer_draws = seeded_draws(cfg.seed)
    customers = spark.range(1, cfg.customer_count + 1).select(F.col("id").cast("long").alias("ID"))
    first_names = F.array(*[F.lit(str(name)) for name in first])
    last_names = F.array(*[F.lit(str(name)) for name in last])
    store_geographies = F.array(*[F.lit(index + 1).cast("long") for index in store_geo_indices])
    global_geography = (customer_draws.h64(["ID"], "customer-global-geo") % F.lit(n_geo) + 1).cast(
        "long"
    )
    if store_geo_indices:
        local_geography = F.element_at(
            store_geographies,
            (
                customer_draws.h64(["ID"], "customer-local-geo") % F.lit(len(store_geo_indices)) + 1
            ).cast("int"),
        )
        geography = F.when(
            customer_draws.u(["ID"], "customer-home-affinity") < F.lit(CUSTOMER_HOME_AFFINITY),
            local_geography,
        ).otherwise(global_geography)
    else:
        geography = global_geography

    street_names = F.array(
        *[F.lit(name) for name in ["Main", "Oak", "Maple", "Market", "Commerce", "Liberty"]]
    )
    street_types = F.array(*[F.lit(name) for name in ["St", "Ave", "Blvd", "Rd"]])
    first_index = (customer_draws.h64(["ID"], "customer-first") % F.lit(len(first)) + 1).cast(
        "long"
    )
    last_index = (customer_draws.h64(["ID"], "customer-last") % F.lit(len(last)) + 1).cast("long")
    street_number = (customer_draws.h64(["ID"], "customer-street-number") % F.lit(9900) + 100).cast(
        "long"
    )
    street_name = F.element_at(
        street_names,
        (customer_draws.h64(["ID"], "customer-street-name") % F.lit(6) + 1).cast("int"),
    )
    street_type = F.element_at(
        street_types,
        (customer_draws.h64(["ID"], "customer-street-type") % F.lit(4) + 1).cast("int"),
    )
    loyalty_suffix = (customer_draws.h64(["ID"], "customer-loyalty") % F.lit(1000)).cast("long")
    phone_exchange = (
        customer_draws.h64(["ID"], "customer-phone-exchange") % F.lit(799) + 200
    ).cast("long")
    phone_line = (customer_draws.h64(["ID"], "customer-phone-line") % F.lit(8999) + 1000).cast(
        "long"
    )
    ble_id = F.upper(F.conv(F.col("ID").cast("string"), 10, 36))

    customers = customers.select(
        "ID",
        F.element_at(first_names, first_index.cast("int")).alias("FirstName"),
        F.element_at(last_names, last_index.cast("int")).alias("LastName"),
        F.format_string("%d %s %s", street_number, street_name, street_type).alias("Address"),
        geography.alias("GeographyID"),
        F.concat(
            F.lit("LC"),
            minimum_width_decimal(F.col("ID"), 6),
            fixed_width_decimal(loyalty_suffix, 3, "loyalty suffix"),
        ).alias("LoyaltyCard"),
        F.format_string("555-%03d-%04d", phone_exchange, phone_line).alias("Phone"),
        F.concat(F.lit("BLE"), minimum_width_text(ble_id, 6)).alias("BLEId"),
        F.concat(F.lit("AD"), minimum_width_decimal(F.col("ID"), 8)).alias("AdId"),
    )
    out["dim_customers"] = customers.select(
        *[field.name for field in spark_schema("dim_customers").fields]
    )

    # --- products: each base product is offered by up to brands_per_product
    #     category-matched brands (datagen combinatorial SKUs). Pricing/launch
    #     are re-rolled per branded variant for realistic price spread.
    brands_by_cat: dict[str, list] = {}
    for b in dicts.brands:
        brands_by_cat.setdefault(b.Category.strip().lower(), []).append(b)
    all_brands = list(dicts.brands)
    tags_by_product = {t.ProductName: t.Tags for t in dicts.tags}
    # Use naive UTC datetimes — Spark session timezone is UTC (set in conftest fixture)
    hist_start = datetime.combine(cfg.start_date, datetime.min.time())
    # Guarantee every department has at least one product available from the
    # first day of history: without this, an adversarial seed could push every
    # product in a department past `hist_start`, leaving a sale-eligible
    # department with no launched product and forcing receipts.py to either drop
    # lines or violate the no-pre-launch-sales invariant (IMP-010). The first
    # generated variant per department is pinned to a pre-history launch.
    dept_covered: set[str] = set()
    prod_rows = []
    pid = 0
    for p in dicts.products:
        pool = (
            _match_brand_category(p.Department, brands_by_cat)
            or _match_brand_category(p.Category, brands_by_cat)
            or all_brands
        )
        k = min(cfg.brands_per_product, len(pool))
        brand_idx = rng.choice(len(pool), size=k, replace=False)
        taxability = (
            "NON_TAXABLE"
            if p.Department in {"Fresh", "Grocery"} and "Candy" not in p.Category
            else "REDUCED_RATE"
            if p.Department in {"Clothing", "Apparel"}
            else "TAXABLE"
        )
        refrigerated = bool(p.Category in REFRIGERATED_CATEGORIES)
        tags = p.Tags or tags_by_product.get(p.ProductName)
        for j in brand_idx:
            pid += 1
            chosen = pool[int(j)]
            launch_r = float(rng.random())  # 60% before history, 30% first half, 10% later
            if p.Department not in dept_covered:
                # First product seen for this department: pin pre-history so the
                # department is sale-eligible on day one. Draw the RNG regardless
                # (above) to keep every other product's launch stream unchanged.
                launch = hist_start - timedelta(days=int(rng.integers(30, 1500)))
                dept_covered.add(p.Department)
            elif launch_r < 0.6:
                launch = hist_start - timedelta(days=int(rng.integers(30, 1500)))
            elif launch_r < 0.9:
                launch = hist_start + timedelta(days=int(rng.integers(0, 183)))
            else:
                launch = hist_start + timedelta(days=int(rng.integers(183, 366)))
            cost, msrp, sale = compute_pricing(float(p.BasePrice), rng)
            prod_rows.append(
                (
                    pid,
                    p.ProductName,
                    chosen.Brand,
                    chosen.Company,
                    p.Department,
                    p.Category,
                    p.Subcategory,
                    cost,
                    msrp,
                    sale,
                    refrigerated,
                    launch,
                    taxability,
                    tags,
                )
            )
    out["dim_products"] = spark.createDataFrame(prod_rows, spark_schema("dim_products"))
    return out


def generate_dim_date(spark: SparkSession, start: date, end: date) -> DataFrame:
    """Exact port of 02-historical-data-load's dim_date (fiscal year starts July)."""
    dates = date_grid(spark, start, end, "date")
    month = F.month("date")
    iso_day = F.pmod(F.dayofweek("date") + F.lit(5), F.lit(7)) + F.lit(1)
    week_of_year = F.floor((F.dayofyear("date") + F.lit(7) - F.dayofweek("date")) / F.lit(7))
    return dates.select(
        F.date_format("date", "yyyyMMdd").cast("long").alias("date_key"),
        "date",
        F.year("date").cast("long").alias("year"),
        F.quarter("date").cast("long").alias("quarter"),
        month.cast("long").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.dayofmonth("date").cast("long").alias("day"),
        iso_day.cast("long").alias("day_of_week"),
        F.date_format("date", "EEEE").alias("day_name"),
        week_of_year.cast("long").alias("week_of_year"),
        F.when(iso_day >= 6, 1).otherwise(0).cast("long").alias("is_weekend"),
        F.when(month >= 7, F.year("date"))
        .otherwise(F.year("date") - 1)
        .cast("long")
        .alias("fiscal_year"),
        (F.floor(F.pmod(month - F.lit(7), F.lit(12)) / F.lit(3)) + F.lit(1))
        .cast("long")
        .alias("fiscal_quarter"),
    )

# --- retail_setup/generation/receipts.py ---
"""Receipts fact group, Spark-native.

Randomness: every stochastic decision derives a uniform double from
xxhash64(key columns, salt) via `runtime.seeded_draws` — partition-
arrangement-independent, so output is deterministic for a (config, seed) pair
regardless of cluster shape. The seed is folded into the salt delimiter inside
`seeded_draws`, keeping generators decoupled from one another.

Visits precede purchase decisions. Negative-binomial baskets combine shopping
missions and persistent budgets, calibrated against the eligible catalog.

Money is integer cents end-to-end; tax replicates datagen's basis-point
integer formula exactly:
    rate_bps = round(rate * 10000); mult = 100/50/0 by taxability;
    tax = (ext_cents * rate_bps * mult + 500_000) // 1_000_000
implemented with Spark integer `DIV` so no float rounding is involved.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# (method, mix weight, decline multiplier, processing_ms lo, processing_ms hi)
TENDERS = [
    ("CREDIT_CARD", 0.4, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.3, 0.8, 1200, 3500),
    ("CASH", 0.2, 0.0, 500, 2000),
    ("MOBILE_PAY", 0.1, 1.2, 800, 2500),
]
BASE_DECLINE = 0.025
DECLINE_REASONS = [
    "INSUFFICIENT_FUNDS",
    "CARD_EXPIRED",
    "INVALID_CVV",
    "NETWORK_ERROR",
    "FRAUD_SUSPECTED",
    "CARD_BLOCKED",
    "LIMIT_EXCEEDED",
]

# Share of in-store receipts whose customer lives in the store's geography
# (datagen home-store locality). The rest draw a network-wide customer.
GEO_AFFINITY = 0.70
# A deterministic cohort stops receiving purchases at staggered dates. This
# creates realistic forward churn labels throughout long histories while
# preserving active customers and pre-churn behavior.
CHURN_COHORT_MODULUS = 10
CHURN_CUTOFF_MAX_MARGIN_DAYS = 60

# Per-department seasonal demand multipliers by month (Jan..Dec), keyed by a
# lowercase token matched as a substring of the department name so the one
# table works across store types (datagen SeasonalPatterns category effects).
SEASONAL_LIFT: dict[str, list[float]] = {
    "electronics": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 2.5, 2.0],
    "seasonal": [0.6, 0.7, 0.9, 1.0, 1.3, 1.5, 1.5, 1.2, 1.0, 1.4, 2.0, 2.5],
    "garden": [0.5, 0.6, 1.3, 1.8, 2.0, 1.8, 1.4, 1.0, 0.8, 0.6, 0.5, 0.5],
    "home": [0.8, 0.8, 1.2, 1.5, 1.6, 1.4, 1.2, 1.0, 0.9, 0.8, 1.0, 1.1],
    "sport": [0.9, 0.9, 1.1, 1.3, 1.5, 1.6, 1.6, 1.3, 1.1, 1.0, 1.1, 1.2],
    "clothing": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "apparel": [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "office": [1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.6, 1.4, 1.0, 1.0, 1.1],
    "toys": [0.8, 0.8, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.1, 2.0, 3.0],
    "baby": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2],
    "grocery": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.3, 1.4],
    "fresh": [1.0, 1.0, 1.0, 1.0, 1.05, 1.1, 1.1, 1.05, 1.0, 1.05, 1.3, 1.4],
    "health": [1.3, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.1],
    "beauty": [1.2, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.2, 1.5],
    "automotive": [1.0, 1.0, 1.05, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0],
}

# Named promo catalog (datagen promotion_utils parity): seasonal codes apply
# only in their eligible months; evergreen SAVE/CLEARANCE codes fill in
# otherwise. Some codes carry a minimum-purchase threshold; BOGO is a qty-based
# buy-one-get-one. (code, discount_pct, eligible_months | None, min_cents, kind)
PROMO_CATALOG: list[tuple[str, int, list[int] | None, int, str]] = [
    ("SAVE10", 10, None, 0, "PCT"),
    ("SAVE15", 15, None, 2500, "PCT"),  # min $25
    ("SAVE20", 20, None, 5000, "PCT"),  # min $50
    ("SAVE25", 25, None, 7500, "PCT"),  # min $75
    ("CLEARANCE30", 30, None, 0, "PCT"),
    ("BOGO50", 50, None, 0, "BOGO"),  # buy one get one 50% off (qty >= 2)
    ("NEWYEAR15", 15, [1, 2], 0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0, "PCT"),
    ("BFRIDAY30", 30, [11], 0, "PCT"),
    ("BFRIDAY40", 40, [11], 10000, "PCT"),  # min $100
    ("HOLIDAY20", 20, [12], 0, "PCT"),
]
# Evergreen fallback: always eligible, no minimum, percentage only.
EVERGREEN: list[tuple[str, int]] = [
    ("SAVE10", 10),
    ("CLEARANCE30", 30),
    ("DEAL15", 15),
]

# Per-customer shopping segments (datagen CustomerJourney segment distribution).
SEGMENT_WEIGHTS: list[tuple[str, float]] = [
    ("BUDGET", 0.35),
    ("CONVENIENCE", 0.25),
    ("QUALITY", 0.20),
    ("BRAND_LOYAL", 0.20),
]


def _segment_price_skew(u: Column, seg: Column) -> Column:
    """Skew a uniform draw toward cheaper/pricier products by customer segment.

    BUDGET biases to the low (cheap) end of a department's price-ranked
    catalog, QUALITY/BRAND_LOYAL to the high end, CONVENIENCE is neutral.
    """
    return (
        F.when(seg == "BUDGET", u * u)
        .when(seg == "QUALITY", F.lit(1.0) - (F.lit(1.0) - u) * (F.lit(1.0) - u))
        .when(seg == "BRAND_LOYAL", F.pow(u, F.lit(0.85)))
        .otherwise(u)
    )


# Weather simulation (datagen EventPatterns): a per-store-day weather state
# scales foot traffic. Seasonal odds make winter snow/storm and summer sun more
# likely. Only the traffic multiplier surfaces (there is no weather column).
WEATHER_MULTS = [1.1, 1.0, 0.7, 0.6, 0.5]  # sunny, cloudy, rainy, snowy, stormy
WEATHER_P_WINTER = [0.25, 0.30, 0.15, 0.20, 0.10]
WEATHER_P_SUMMER = [0.55, 0.25, 0.15, 0.00, 0.05]
WEATHER_P_SHOULDER = [0.40, 0.30, 0.20, 0.05, 0.05]

# Shopping trip archetypes (datagen ShoppingBehaviorType): each trip is a quick
# run / normal run / family shop / bulk stock-up, giving multi-modal basket
# sizes. Multipliers scale the store-type basket_lambda; weighted mean ~1.0 so
# overall item volume is preserved.
TRIP_TYPES: list[tuple[str, float, float]] = [  # (name, weight, basket multiplier)
    ("QUICK_TRIP", 0.30, 0.30),
    ("GROCERY_RUN", 0.40, 0.90),
    ("FAMILY_SHOPPING", 0.20, 1.60),
    ("BULK_SHOPPING", 0.10, 2.50),
]


def _cdf_pick_lit(u: Column, probs: list[float], values: list[float]) -> Column:
    """Inverse-CDF pick returning the chosen literal value."""
    acc = 0.0
    expr: Column | None = None
    for p, v in list(zip(probs, values))[:-1]:
        acc += p
        cond = u < F.lit(acc)
        expr = F.when(cond, F.lit(v)) if expr is None else expr.when(cond, F.lit(v))
    return expr.otherwise(F.lit(values[-1])) if expr is not None else F.lit(values[0])


def _weather_mult(u: Column, month: Column) -> Column:
    """Per-store-day weather traffic multiplier with seasonal weather odds."""
    return (
        F.when(month.isin(12, 1, 2), _cdf_pick_lit(u, WEATHER_P_WINTER, WEATHER_MULTS))
        .when(month.isin(6, 7, 8), _cdf_pick_lit(u, WEATHER_P_SUMMER, WEATHER_MULTS))
        .otherwise(_cdf_pick_lit(u, WEATHER_P_SHOULDER, WEATHER_MULTS))
    )


def _trip_basket_mult(u: Column) -> Column:
    """Pick a shopping-trip archetype's basket-size multiplier (inverse-CDF)."""
    return _cdf_pick_lit(u, [w for _, w, _ in TRIP_TYPES], [m for _, _, m in TRIP_TYPES])


def _seasonal_factor(dept_name: str, month_col: Column) -> Column:
    """Monthly demand multiplier for a department (1.0 if no token matches)."""
    key = dept_name.lower()
    lifts = next((m for token, m in SEASONAL_LIFT.items() if token in key), None)
    if lifts is None:
        return F.lit(1.0)
    expr: Column | None = None
    for m in range(1, 13):
        cond = month_col == F.lit(m)
        expr = (
            F.when(cond, F.lit(lifts[m - 1]))
            if expr is None
            else expr.when(cond, F.lit(lifts[m - 1]))
        )
    return expr.otherwise(F.lit(1.0))


def _with_seasonal_department(
    df: DataFrame, u_col: Column, dept_weights: dict[str, float]
) -> DataFrame:
    """Pick a department per row via inverse-CDF over base weights scaled by the
    seasonal lift for the row's month.

    Intermediate per-department weight (``_w*``) and cumulative-CDF (``_c*``)
    columns are materialized so Catalyst codegen stays small even for store
    types with many departments (an inline single expression overflows the
    64 KB JVM method limit and forces interpreted fallback). They are dropped
    before returning.
    """
    depts = list(dept_weights)
    month = F.month(F.col("event_date"))
    out = df
    for i, dn in enumerate(depts):
        out = out.withColumn(f"_w{i}", F.lit(float(dept_weights[dn])) * _seasonal_factor(dn, month))
    wt = F.col("_w0")
    for i in range(1, len(depts)):
        wt = wt + F.col(f"_w{i}")
    out = out.withColumn("_wt", wt)
    prev: str | None = None
    for i in range(len(depts) - 1):
        term = F.col(f"_w{i}") / F.col("_wt")
        out = out.withColumn(f"_c{i}", term if prev is None else F.col(prev) + term)
        prev = f"_c{i}"
    expr: Column | None = None
    for i, dn in enumerate(depts[:-1]):
        cond = u_col < F.col(f"_c{i}")
        expr = F.when(cond, F.lit(dn)) if expr is None else expr.when(cond, F.lit(dn))
    dept_col = expr.otherwise(F.lit(depts[-1])) if expr is not None else F.lit(depts[0])
    out = out.withColumn("department", dept_col)
    drop_cols = (
        [f"_w{i}" for i in range(len(depts))] + ["_wt"] + [f"_c{i}" for i in range(len(depts) - 1)]
    )
    return out.drop(*drop_cols)


def _promo_eligible_expr(
    idx_col: Column, month_col: Column, ext_before: Column, qty: Column
) -> Column:
    """True when the catalog entry at idx_col is eligible: in-month, line value
    meets any minimum, and BOGO needs qty >= 2."""
    expr: Column | None = None
    for i, (_, _, months, min_cents, kind) in enumerate(PROMO_CATALOG):
        e = F.lit(True) if months is None else month_col.isin(*months)
        if min_cents:
            e = e & (ext_before >= F.lit(min_cents))
        if kind == "BOGO":
            e = e & (qty >= F.lit(2))
        cond = idx_col == F.lit(i)
        expr = F.when(cond, e) if expr is None else expr.when(cond, e)
    return expr.otherwise(F.lit(False))


def _apply_churn_lifecycle(
    receipts: DataFrame,
    d: seeded_draws,
    cfg: GenerationConfig,
) -> DataFrame:
    """Stop assigning purchases to a seeded customer cohort after its cutoff."""

    customer_count = int(cfg.customer_count)
    history_days = (cfg.end_date - cfg.start_date).days + 1
    cutoff_margin = min(
        CHURN_CUTOFF_MAX_MARGIN_DAYS,
        max(1, history_days // 4),
    )
    cutoff_span = max(1, history_days - (2 * cutoff_margin))
    cutoff_offset = (
        F.lit(cutoff_margin)
        + F.pmod(
            d.h64(["customer_id"], "churn_cutoff"),
            F.lit(cutoff_span),
        )
    ).cast("int")
    cutoff_date = F.date_add(F.lit(cfg.start_date), cutoff_offset)

    replacement_base = (
        F.pmod(
            d.h64(["receipt_id_ext"], "active_customer"),
            F.lit(customer_count),
        )
        + F.lit(1)
    ).cast("long")
    replacement_customer = F.when(
        F.pmod(replacement_base, F.lit(CHURN_COHORT_MODULUS)) == 0,
        F.pmod(replacement_base, F.lit(customer_count)) + F.lit(1),
    ).otherwise(replacement_base)
    is_post_churn = (F.pmod(F.col("customer_id"), F.lit(CHURN_COHORT_MODULUS)) == 0) & (
        F.col("event_date") >= cutoff_date
    )
    return receipts.withColumn(
        "customer_id",
        F.when(is_post_churn, replacement_customer).otherwise(F.col("customer_id")),
    )


def _assign_customers(
    receipts: DataFrame, dims: dict[str, DataFrame], d: seeded_draws, cfg: GenerationConfig
) -> DataFrame:
    """Resolve each receipt's customer_id with store-geography affinity.

    With probability ``GEO_AFFINITY`` the customer is drawn from those living
    in the store's geography (resolved by a deterministic within-geography
    rank), otherwise a network-wide customer is used. Falls back to the
    network-wide pick when the store's geography has no customers.
    """
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_id"), F.col("GeographyID").alias("cust_geo_id")
    )
    # Replicated selection tickets give persistent, long-tailed visit propensity.
    # This dimension-sized expansion avoids receipt x customer inequality joins.
    customers = customers.withColumn(
        "_ticket",
        F.explode(F.sequence(F.lit(1), customer_visit_tickets(d, F.col("customer_id")))),
    )
    geo_sizes = customers.groupBy("cust_geo_id").agg(F.count("*").alias("geo_cust_count"))
    rank_w = Window.partitionBy("cust_geo_id").orderBy("customer_id", "_ticket")
    cust_ranked = customers.withColumn("local_rank", F.row_number().over(rank_w))
    global_customers = customers.withColumn(
        "_global_rank", F.row_number().over(Window.orderBy("customer_id", "_ticket"))
    ).select("_global_rank", F.col("customer_id").alias("global_customer"))
    global_size = customers.agg(F.count("*").alias("_global_size"))

    r = (
        receipts.join(geo_sizes, receipts["store_geo_id"] == geo_sizes["cust_geo_id"], "left")
        .drop("cust_geo_id")
        .withColumn("geo_cust_count", F.coalesce(F.col("geo_cust_count"), F.lit(0)))
        .crossJoin(global_size)
        .withColumn(
            "_global_rank",
            (d.h64(["receipt_id_ext"], "cust") % F.col("_global_size") + 1).cast("int"),
        )
        .join(global_customers, "_global_rank")
        .withColumn(
            "want_local",
            (d.u(["receipt_id_ext"], "affinity") < F.lit(GEO_AFFINITY))
            & (F.col("geo_cust_count") > 0),
        )
        .withColumn(
            "local_rank_target",
            F.when(
                F.col("geo_cust_count") > 0,
                (d.h64(["receipt_id_ext"], "localcust") % F.col("geo_cust_count") + 1),
            ).cast("long"),
        )
    )
    local = cust_ranked.select(
        F.col("cust_geo_id").alias("_lgeo"),
        F.col("local_rank").alias("_lrank"),
        F.col("customer_id").alias("local_customer"),
    )
    assigned = (
        r.join(
            local,
            (F.col("store_geo_id") == F.col("_lgeo"))
            & (F.col("local_rank_target") == F.col("_lrank")),
            "left",
        )
        .withColumn(
            "customer_id",
            F.when(
                F.col("want_local") & F.col("local_customer").isNotNull(), F.col("local_customer")
            ).otherwise(F.col("global_customer")),
        )
        .drop(
            "_lgeo",
            "_lrank",
            "local_customer",
            "global_customer",
            "want_local",
            "local_rank_target",
            "geo_cust_count",
            "_global_rank",
            "_global_size",
        )
    )
    return _apply_churn_lifecycle(assigned, d, cfg)


def _fmt(cents_col: Column) -> Column:
    """Integer cents -> 'XX.XX' string (no thousands separators)."""
    return F.format_string("%.2f", cents_col / F.lit(100.0))


def _open_close(operating_hours: str) -> tuple[int, int]:
    """Parse an ``operating_hours`` string into ``[open, close)`` integer hours."""
    if operating_hours == "24h":
        return 0, 24
    lo, hi = operating_hours.split("-")
    return int(lo), int(hi)


def _open_close_cols(operating_hours: Column) -> tuple[Column, Column]:
    """Column form of :func:`_open_close` for ``[open, close)`` hour bounds.

    ``"24h"`` maps to ``[0, 24)``; ``"7-22"`` maps to ``[7, 22)``. The close hour
    is exclusive, so a ``"7-22"`` store transacts in hours 7..21 inclusive.
    """
    is_24 = operating_hours == "24h"
    parts = F.split(operating_hours, "-")
    open_h = F.when(is_24, F.lit(0)).otherwise(parts.getItem(0).cast("int"))
    close_h = F.when(is_24, F.lit(24)).otherwise(parts.getItem(1).cast("int"))
    return open_h, close_h


def _pick_hour_static(u: Column, hourly_weights: list[float], open_h: int, close_h: int) -> Column:
    """Inverse-CDF hour pick for a single, statically known operating window.

    Weights outside ``[open_h, close_h)`` are zeroed and the CDF renormalised
    over the open hours, so the pick can never land on a closed hour. The
    cumulative fractions are Python floats (not column arithmetic), keeping the
    generated expression small.
    """
    masked = [w if open_h <= h < close_h else 0.0 for h, w in enumerate(hourly_weights)]
    total = sum(masked) or 1.0
    chain: Column | None = None
    acc = 0.0
    for h in range(close_h - 1):
        acc += masked[h]
        cond = u < F.lit(acc / total)
        chain = F.when(cond, F.lit(h)) if chain is None else chain.when(cond, F.lit(h))
    last_open = close_h - 1
    return F.lit(last_open) if chain is None else chain.otherwise(F.lit(last_open))


def _pick_hour(
    u: Column, hourly_weights: list[float], operating_hours: Column, patterns: list[str]
) -> Column:
    """Masked inverse-CDF hour pick that respects each store's operating window.

    Operating hours come from a small fixed set of patterns, so the per-pattern
    static CDF (cheap Python-float chain) is selected by the store's
    ``operating_hours`` string — avoiding a per-row column CDF that would
    explode the Catalyst expression tree.
    """
    expr: Column | None = None
    for pat in patterns:
        open_h, close_h = _open_close(pat)
        branch = _pick_hour_static(u, hourly_weights, open_h, close_h)
        cond = operating_hours == F.lit(pat)
        expr = F.when(cond, branch) if expr is None else expr.when(cond, branch)
    # Every store carries one of ``patterns``; the fallback only guards nulls.
    return F.lit(12) if expr is None else expr.otherwise(F.lit(12))


def generate_receipts_group(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_receipts, fact_receipt_lines, fact_payments (in-store only)."""

    d = seeded_draws(cfg.seed)

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"),
        "tax_rate",
        "daily_traffic_multiplier",
        "operating_hours",
        F.col("GeographyID").alias("store_geo_id"),
    )
    grid = generate_hourly_visits(spark, dims["dim_stores"], profile, cfg).join(stores, "store_id")
    earlier_hours = (
        Window.partitionBy("store_id", "day")
        .orderBy("hour")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    grid = grid.withColumn(
        "_prior_receipts", F.coalesce(F.sum("n_receipts").over(earlier_hours), F.lit(0))
    )
    receipts = (
        grid.filter(F.col("n_receipts") > 0)
        .withColumn("_hour_seq", F.explode(F.sequence(F.lit(1), F.col("n_receipts"))))
        .withColumn("seq", F.col("_prior_receipts") + F.col("_hour_seq"))
        .withColumn("minute", (d.h64(["store_id", "day", "seq"], "min") % 60).cast("int"))
        .withColumn("second", (d.h64(["store_id", "day", "seq"], "sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # RCP(3) + yyyyMMddHHmm(12) + store(4) + seq(6) = 25 chars, unique by
        # construction (store_id, day, seq) is a key
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RCP"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "receipt store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily receipt sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
    )

    # geography affinity: resolve each receipt's customer (local vs network-wide)
    receipts = _assign_customers(receipts, dims, d, cfg)

    receipts = receipts.withColumn(
        "_offer", d.u(["receipt_id_ext"], "offer") < F.lit(profile.promo_rate)
    ).withColumn(
        "tender_type",
        d.pick_by_weights(["receipt_id_ext"], "tender", [(n, w) for n, w, _, _, _ in TENDERS]),
    )

    # --- lines: explode baskets, weighted department -> price-tiered product
    # within department, restricted to products already launched on the sale day
    # (IMP-010 no-pre-launch-sales invariant).
    products = dims["dim_products"].select(
        F.col("ID").alias("product_id"),
        F.col("SalePrice"),
        F.col("taxability"),
        F.col("Department").alias("department"),
        F.to_date("LaunchDate").alias("launch_date"),
    )

    product_departments = {r.department for r in products.select("department").distinct().collect()}
    unknown = set(profile.department_weights) - product_departments
    if unknown:
        raise ValueError(
            f"profile department_weights reference departments missing from the catalog: {sorted(unknown)}"
        )

    sale_dates = receipts.select("event_date").distinct()
    catalog = retail_catalog(products, sale_dates, d)
    department_weight = sum(
        (
            F.when(
                F.col("department") == name,
                F.lit(weight) * _seasonal_factor(name, F.month("event_date")),
            ).otherwise(0.0)
            for name, weight in profile.department_weights.items()
        ),
        F.lit(0.0),
    )
    calibration = (
        catalog.withColumn("_weight", department_weight)
        .groupBy("event_date")
        .agg(
            (F.sum(F.col("_weight") * F.col("_expected_line_value")) / F.sum("_weight")).alias(
                "_line_value"
            ),
            (F.sum(F.col("_weight") * F.col("_expected_taxable_value")) / F.sum("_weight")).alias(
                "_taxable_value"
            ),
        )
    )
    receipts = (
        receipts.join(calibration, "event_date")
        .withColumn(
            "_basket_mean",
            calibrated_basket_mean(
                F.col("_line_value"),
                F.col("_taxable_value"),
                F.col("tax_rate"),
                profile.avg_ticket_target,
                profile.promo_rate,
                profile.basket_lambda,
            ),
        )
        .withColumn(
            "basket_n",
            retail_basket_size(
                d,
                ["receipt_id_ext"],
                F.col("customer_id"),
                F.col("_basket_mean"),
                F.col("_offer"),
                profile.promo_rate,
            ),
        )
    )

    exploded = receipts.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "store_id",
        "tax_rate",
        "basket_n",
        "_offer",
        "customer_id",
    ).withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    exploded = _with_seasonal_department(
        exploded,
        F.when(
            d.u(["receipt_id_ext", "line_num"], "mission-coherence") < 0.60,
            F.when(
                d.u(["receipt_id_ext"], "favorite-mission") < 0.50,
                d.u(["customer_id"], "favorite-department"),
            ).otherwise(d.u(["receipt_id_ext"], "mission-department")),
        ).otherwise(d.u(["receipt_id_ext", "line_num"], "dept")),
        profile.department_weights,
    )

    lines = (
        exploded.join(catalog, ["event_date", "department"])
        .withColumn(
            "_product",
            pick_retail_product(F.col("_catalog"), d.u(["receipt_id_ext", "line_num"], "prod")),
        )
        .select("*", "_product.product_id", "_product.SalePrice", "_product.taxability")
        .drop("_product", "_catalog", "_expected_line_value", "_expected_taxable_value")
        .withColumn(
            "quantity", retail_quantity(d, ["receipt_id_ext", "line_num"], F.col("department"))
        )
        .withColumn("unit_cents", F.round(F.col("SalePrice") * 100).cast("long"))
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn("has_promo", F.col("_offer"))
        # named promo code + matching discount (datagen promotion_utils parity):
        # a seasonal/min-purchase/BOGO code if eligible, else an evergreen code.
        .withColumn(
            "_pidx",
            (d.h64(["receipt_id_ext", "line_num"], "pcode") % F.lit(len(PROMO_CATALOG))).cast(
                "int"
            ),
        )
        .withColumn(
            "_pcode",
            F.element_at(F.array(*[F.lit(t[0]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_ppct",
            F.element_at(F.array(*[F.lit(t[1]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_pkind",
            F.element_at(F.array(*[F.lit(t[4]) for t in PROMO_CATALOG]), F.col("_pidx") + 1),
        )
        .withColumn(
            "_pelig",
            _promo_eligible_expr(
                F.col("_pidx"), F.month(F.col("event_date")), F.col("ext_before"), F.col("quantity")
            ),
        )
        .withColumn(
            "_evidx",
            (d.h64(["receipt_id_ext", "line_num"], "pcodeev") % F.lit(len(EVERGREEN))).cast("int"),
        )
        .withColumn(
            "_evcode", F.element_at(F.array(*[F.lit(c) for c, _ in EVERGREEN]), F.col("_evidx") + 1)
        )
        .withColumn(
            "_evpct", F.element_at(F.array(*[F.lit(p) for _, p in EVERGREEN]), F.col("_evidx") + 1)
        )
        .withColumn(
            "promo_code",
            F.when(
                F.col("has_promo"),
                F.when(F.col("_pelig"), F.col("_pcode")).otherwise(F.col("_evcode")),
            ),
        )
        .withColumn("_disc_pct", F.when(F.col("_pelig"), F.col("_ppct")).otherwise(F.col("_evpct")))
        .withColumn("_disc_kind", F.when(F.col("_pelig"), F.col("_pkind")).otherwise(F.lit("PCT")))
        .withColumn(
            "discount_cents",
            F.when(
                F.col("has_promo"),
                F.when(
                    F.col("_disc_kind") == "BOGO",
                    # buy-one-get-one: every 2nd item discounted at _disc_pct
                    F.floor(
                        F.floor(F.col("quantity") / F.lit(2))
                        * F.col("unit_cents")
                        * F.col("_disc_pct")
                        / F.lit(100.0)
                        + F.lit(0.5)
                    ).cast("long"),
                ).otherwise(
                    F.floor(
                        F.col("ext_before") * F.col("_disc_pct") / F.lit(100.0) + F.lit(0.5)
                    ).cast("long")
                ),
            ).otherwise(F.lit(0).cast("long")),
        )
        .withColumn(
            "ext_cents",
            F.greatest(F.lit(0).cast("long"), F.col("ext_before") - F.col("discount_cents")),
        )
        # tax: integer basis-point math, replicating datagen _tax_cents exactly
        .withColumn("rate_bps", F.round(F.col("tax_rate") * 10000).cast("long"))
        .withColumn(
            "tax_mult",
            F.when(F.col("taxability") == "TAXABLE", 100)
            .when(F.col("taxability") == "REDUCED_RATE", 50)
            .otherwise(0)
            .cast("long"),
        )
        .withColumn(
            "line_tax_cents",
            F.floor(
                (F.col("ext_cents") * F.col("rate_bps") * F.col("tax_mult") + F.lit(500_000))
                / F.lit(1_000_000)
            ).cast("long"),
        )
    )

    fact_receipt_lines = lines.select(
        "receipt_id_ext",
        "event_ts",
        "event_date",
        "line_num",
        "product_id",
        "quantity",
        _fmt(F.col("unit_cents")).alias("unit_price"),
        "unit_cents",
        _fmt(F.col("ext_cents")).alias("ext_price"),
        "ext_cents",
        "promo_code",
    ).select(*column_names("fact_receipt_lines"))

    # --- header rollup
    hdr = lines.groupBy("receipt_id_ext").agg(
        F.sum("ext_cents").alias("subtotal_cents"),
        F.sum("discount_cents").alias("discount_cents"),
        F.sum("line_tax_cents").alias("tax_cents"),
    )
    fact_receipts = (
        receipts.join(hdr, "receipt_id_ext")
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("receipt_type", F.lit("SALE"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: discount is applied before tax, so subtotal_cents (kept as
        # the existing net subtotal) plus discount_cents recovers the
        # pre-discount gross subtotal. attribution fields default NULL here;
        # attribution.py enriches the ~5% of receipts selected for attribution.
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .select(
            "receipt_id_ext",
            "trace_id",
            "event_ts",
            "event_date",
            "store_id",
            "customer_id",
            "receipt_type",
            "tender_type",
            "subtotal_cents",
            _fmt(F.col("discount_cents")).alias("discount_amount"),
            "tax_cents",
            "total_cents",
            _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
            _fmt(F.col("tax_cents")).alias("tax_amount"),
            _fmt(F.col("total_cents")).alias("total_amount"),
            "payment_method",
            # Legacy semantic-model column (sourceColumn: Subtotal); same value
            # as subtotal_amount per the TMDL contract.
            _fmt(F.col("subtotal_cents")).alias("Subtotal"),
            "gross_subtotal_cents",
            "discount_cents",
            "attribution_journey_id",
            "campaign_id",
            "impression_id_ext",
        )
        .select(*column_names("fact_receipts"))
    )

    # --- payments (one per receipt; in-store, so order_id_ext is NULL)
    u_dec = d.u(["receipt_id_ext"], "decline")
    decline_p: Column = F.lit(0.0)
    for name, _, mult, _, _ in TENDERS:
        decline_p = F.when(F.col("payment_method") == name, F.lit(BASE_DECLINE * mult)).otherwise(
            decline_p
        )
    proc_lo: Column = F.lit(TENDERS[-1][3])
    proc_hi: Column = F.lit(TENDERS[-1][4])
    for name, _, _, lo, hi in TENDERS[:-1]:
        proc_lo = F.when(F.col("payment_method") == name, F.lit(lo)).otherwise(proc_lo)
        proc_hi = F.when(F.col("payment_method") == name, F.lit(hi)).otherwise(proc_hi)
    reason_idx = (d.h64(["receipt_id_ext"], "reason") % len(DECLINE_REASONS)).cast("int")
    fact_payments = (
        fact_receipts.withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TXN_"),
                F.unix_timestamp("event_ts").cast("string"),
                F.lit("_"),
                fixed_width_decimal(
                    d.h64(["receipt_id_ext"], "txn") % 1_000_000,
                    6,
                    "payment transaction suffix",
                ),
            ),
        )
        .withColumn("status", F.when(u_dec < decline_p, "DECLINED").otherwise("APPROVED"))
        .withColumn(
            "decline_reason",
            F.when(
                F.col("status") == "DECLINED",
                F.element_at(F.array(*[F.lit(r) for r in DECLINE_REASONS]), reason_idx + 1),
            ),
        )
        .withColumn(
            "processing_time_ms",
            (proc_lo + d.u(["receipt_id_ext"], "proc") * (proc_hi - proc_lo)).cast("long"),
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/returns.py ---
"""Returns fact group, Spark-native (unioned into the receipts contract).

Semantics (datagen utils_mixin): sample ~``cfg.return_rate`` of SALE receipts
per day — Dec 26 spikes 6x, capped at 10% of the day's receipts. Each return
posts 1..``RETURN_WINDOW_DAYS`` days *after* its originating sale (IMP-010: no
same-day returns), clamped to ``cfg.end_date``. The return header gets a new
``receipt_id_ext`` with the same 25-char layout as sales (``RET`` +
yyyyMMddHHmm + store4 + seq6), ``receipt_type='RETURN'``, noon ``event_ts`` on
the return day, NULL ``customer_id``, CREDIT_CARD tender, and negated cents.
Lines mirror the original receipt's lines with negative quantity / ext_cents
(``promo_code='RETURN'``); each return gets one negative, APPROVED payment —
refunds don't decline.

All randomness comes from `runtime.seeded_draws` keyed on the original
receipt id, so output is deterministic for a (config, seed) pair.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# CREDIT_CARD processing-time bounds (ms), matching the sales TENDERS table.
_PROC_MS_LO = 1500
_PROC_MS_HI = 4000

# Returns post 1..RETURN_WINDOW_DAYS days after the originating sale.
RETURN_WINDOW_DAYS = 14


def build_return_headers(
    sales_group: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Sample SALE receipts for return and date each strictly after its sale.

    Returns the RETURN header DataFrame, which retains ``orig_receipt_id_ext``
    and ``orig_event_date`` (the originating sale day) alongside the return
    ``event_date``/``event_ts``. Exposing the originating sale day lets callers
    and tests verify the IMP-010 no-same-day-returns guarantee, which the final
    ``fact_receipts`` contract can't carry. Deterministic per (config, seed).
    """
    d = seeded_draws(cfg.seed)

    sales = sales_group["fact_receipts"].filter(
        (F.col("receipt_type") == "SALE")
        & (F.col("event_date") < F.lit(cfg.end_date))
        & ~((F.month("event_date") == 12) & (F.dayofmonth("event_date") == 25))
    )

    # --- sample: per-day rate, 6x on Dec 26, hard-capped at 10%.
    # Accumulate fractional daily quotas before rounding so low-volume days do
    # not each lose their fraction independently. The lowest stable hashes pick
    # the rows within each day.
    day_rate = F.least(
        F.lit(0.10),
        F.lit(cfg.return_rate)
        * F.when(
            (F.month("event_date") == 12) & (F.dayofmonth("event_date") == 26),
            F.lit(6.0),
        ).otherwise(F.lit(1.0)),
    )
    daily = (
        sales.groupBy("event_date")
        .agg(F.count("*").alias("_day_count"))
        .withColumn("_expected", F.col("_day_count") * day_rate)
    )
    cumulative = Window.orderBy("event_date").rowsBetween(
        Window.unboundedPreceding, Window.currentRow
    )
    prior = Window.orderBy("event_date").rowsBetween(Window.unboundedPreceding, -1)
    quotas = (
        daily.withColumn("_cumulative_expected", F.sum("_expected").over(cumulative))
        .withColumn(
            "_prior_expected",
            F.coalesce(F.sum("_expected").over(prior), F.lit(0.0)),
        )
        .withColumn(
            "_n_keep",
            (
                F.floor(F.col("_cumulative_expected") + F.lit(1e-9))
                - F.floor(F.col("_prior_expected") + F.lit(1e-9))
            ).cast("long"),
        )
        .select("event_date", "_n_keep")
    )
    u_ret = d.u(["receipt_id_ext"], "return")
    day_w = Window.partitionBy("event_date")
    sampled = (
        sales.join(quotas, "event_date")
        .withColumn("_ret_rank", F.row_number().over(day_w.orderBy(u_ret, "receipt_id_ext")))
        .filter(F.col("_ret_rank") <= F.col("_n_keep"))
        .drop("_n_keep", "_ret_rank")
    )

    # --- header: new RET id (RET(3)+yyyyMMddHHmm(12)+store4+seq6 = 25 chars,
    # unique because seq is a row_number per (store_id, return day) at noon)
    sequenced = (
        sampled.withColumnRenamed("receipt_id_ext", "orig_receipt_id_ext")
        .withColumnRenamed("event_date", "orig_event_date")
        .withColumn(
            "_max_delay",
            F.least(
                F.lit(RETURN_WINDOW_DAYS),
                F.datediff(F.lit(cfg.end_date), F.col("orig_event_date")),
            ).cast("int"),
        )
        .filter(F.col("_max_delay") >= 1)
        .withColumn(
            "_return_delay",
            (
                F.lit(1)
                + F.floor(d.u(["orig_receipt_id_ext"], "return_delay") * F.col("_max_delay"))
            ).cast("int"),
        )
        .withColumn(
            "_candidate_date",
            F.date_add("orig_event_date", F.col("_return_delay")),
        )
        .withColumn(
            "_latest_return_date",
            F.date_add("orig_event_date", F.col("_max_delay")),
        )
        .withColumn(
            "event_date",
            F.when(
                (F.month("_candidate_date") == 12) & (F.dayofmonth("_candidate_date") == 25),
                F.when(
                    F.date_add("_candidate_date", 1) <= F.col("_latest_return_date"),
                    F.date_add("_candidate_date", 1),
                ).otherwise(F.date_add("_candidate_date", -1)),
            ).otherwise(F.col("_candidate_date")),
        )
        .filter(
            (F.col("event_date") > F.col("orig_event_date"))
            & (F.col("event_date") <= F.col("_latest_return_date"))
        )
        .withColumn(
            "event_ts",
            F.to_timestamp(F.concat(F.col("event_date").cast("string"), F.lit(" 12:00:00"))),
        )
    )
    seq_w = Window.partitionBy("store_id", "event_date").orderBy("orig_receipt_id_ext")
    return (
        sequenced.withColumn("seq", F.row_number().over(seq_w))
        .withColumn(
            "receipt_id_ext",
            F.concat(
                F.lit("RET"),
                F.date_format("event_ts", "yyyyMMddHHmm"),
                fixed_width_decimal(F.col("store_id"), 4, "return store ID"),
                fixed_width_decimal(F.col("seq"), 6, "daily return sequence"),
            ),
        )
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
        .withColumn("customer_id", F.lit(None).cast("long"))
        .withColumn("receipt_type", F.lit("RETURN"))
        .withColumn("tender_type", F.lit("CREDIT_CARD"))
        .withColumn("payment_method", F.lit("CREDIT_CARD"))
        .withColumn("subtotal_cents", -F.col("subtotal_cents"))
        .withColumn("tax_cents", -F.col("tax_cents"))
        .withColumn("total_cents", -F.col("total_cents"))
        # IMP-007: negate the original discount alongside subtotal/tax so
        # gross - discount = net continues to hold for the return row.
        # Returns are never attribution candidates (SALE-only per contract).
        .withColumn("discount_cents", -F.col("discount_cents"))
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
    )


def generate_returns(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    return_headers: DataFrame | None = None,
) -> dict[str, DataFrame]:
    """Generate refunds, optionally retaining a pre-fulfillment sampling plan.

    Planned dates/IDs survive fulfillment; financial values are refreshed from
    accepted receipts and wholly unfulfilled purchases disappear in the join.
    """

    d = seeded_draws(cfg.seed)

    header = build_return_headers(sales_group, cfg) if return_headers is None else return_headers
    money = ["subtotal_cents", "discount_cents", "tax_cents", "total_cents"]
    header = (
        header.drop(*money, "gross_subtotal_cents")
        .join(
            sales_group["fact_receipts"].select(
                F.col("receipt_id_ext").alias("orig_receipt_id_ext"),
                *[(-F.col(name)).alias(name) for name in money],
            ),
            "orig_receipt_id_ext",
        )
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
    )

    fact_receipts = header.select(
        "receipt_id_ext",
        "trace_id",
        "event_ts",
        "event_date",
        "store_id",
        "customer_id",
        "receipt_type",
        "tender_type",
        "subtotal_cents",
        _fmt(F.col("discount_cents")).alias("discount_amount"),
        "tax_cents",
        "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"),
        "payment_method",
        # Legacy semantic-model column mirrors subtotal_amount (TMDL contract).
        _fmt(F.col("subtotal_cents")).alias("Subtotal"),
        "gross_subtotal_cents",
        "discount_cents",
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    ).select(*column_names("fact_receipts"))

    # --- lines: mirror the original receipt's lines, negated
    orig_lines = sales_group["fact_receipt_lines"].select(
        F.col("receipt_id_ext").alias("orig_receipt_id_ext"),
        "line_num",
        "product_id",
        "quantity",
        "unit_cents",
        "ext_cents",
    )
    fact_receipt_lines = (
        header.select("orig_receipt_id_ext", "receipt_id_ext", "event_ts", "event_date")
        .join(orig_lines, "orig_receipt_id_ext")
        .withColumn("quantity", (-F.col("quantity")).cast("int"))
        .withColumn("ext_cents", -F.col("ext_cents"))
        .withColumn("promo_code", F.lit("RETURN"))
        .select(
            "receipt_id_ext",
            "event_ts",
            "event_date",
            "line_num",
            "product_id",
            "quantity",
            _fmt(F.col("unit_cents")).alias("unit_price"),
            "unit_cents",
            _fmt(F.col("ext_cents")).alias("ext_price"),
            "ext_cents",
            "promo_code",
        )
        .select(*column_names("fact_receipt_lines"))
    )

    # --- payments: one negative APPROVED CREDIT_CARD refund per return
    fact_payments = (
        fact_receipts.withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TXN_"),
                F.unix_timestamp("event_ts").cast("string"),
                F.lit("_"),
                fixed_width_decimal(
                    d.h64(["receipt_id_ext"], "txn") % 1_000_000,
                    6,
                    "return transaction suffix",
                ),
            ),
        )
        .withColumn("status", F.lit("APPROVED"))
        .withColumn("decline_reason", F.lit(None).cast("string"))
        .withColumn(
            "processing_time_ms",
            (
                F.lit(_PROC_MS_LO)
                + d.u(["receipt_id_ext"], "proc") * F.lit(_PROC_MS_HI - _PROC_MS_LO)
            ).cast("long"),
        )
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/inventory_policy.py ---
"""Causal, streaming inventory policy, independent of Spark.

Each SKU learns an EWMA of attempted daily demand (including lost sales).
There are no backorders: rejected whole lines have no stock commitment.
Inventory position includes outstanding deliveries and accepted restock returns.
SKU-specific shipments use deterministic 1-3 day service, capacity-bounded
legs, and unlimited just-in-time supplier replenishment at the DC. This is a
stock/service simulation, not a vehicle-routing or supplier-capacity model.
"""

from dataclasses import dataclass
from datetime import date, datetime, timedelta
from datetime import time as inventory_time
from hashlib import sha256
from heapq import heappop, heappush
from math import ceil
from typing import Iterable, Iterator

CASE_PACK = 12


def stable_int(seed: int, *keys: object) -> int:
    """Stable across processes, Python hash seeds, and Spark partition counts."""
    value = "|".join(str(key) for key in (seed, *keys))
    return int.from_bytes(sha256(value.encode("utf-8")).digest()[:8], "big")


def opening_stock(seed: int, store_id: int, product_id: int) -> int:
    """History-independent opening state; never inspect future demand."""
    return 40 + stable_int(seed, store_id, product_id, "opening") % 81


@dataclass(frozen=True)
class Demand:
    event_ts: datetime
    receipt_id: str
    line_num: int
    quantity: int
    return_ts: datetime | None = None
    return_id: str | None = None

    @property
    def trace_id(self) -> str:
        return f"TRC-INV-{self.receipt_id}-{self.line_num}"


@dataclass
class InventoryEvent:
    store_id: int
    product_id: int
    event_ts: datetime
    trace_id: str
    kind: str
    quantity: int
    source: str
    balance: int = 0
    receipt_id: str | None = None
    line_num: int | None = None
    requested_quantity: int = 0
    unfulfilled_quantity: int = 0
    reorder_point: int | None = None
    priority: str | None = None
    shipment_id: str | None = None
    leg: int | None = None
    scheduled_ts: datetime | None = None
    loading_ts: datetime | None = None
    depart_ts: datetime | None = None
    eta: datetime | None = None
    unloading_ts: datetime | None = None
    etd: datetime | None = None


def shipment_events(
    seed: int,
    store_id: int,
    product_id: int,
    day: date,
    quantity: int,
    capacity: int,
) -> Iterator[InventoryEvent]:
    """One SKU per shipment leg; the ledger consumes these exact timestamps."""
    draw = stable_int(seed, store_id, product_id, day, "service")
    service_day = day + timedelta(days=1 + draw % 3)
    for leg, offset in enumerate(range(0, quantity, capacity)):
        stagger = timedelta(minutes=30 * leg)
        loading = datetime.combine(service_day, inventory_time(4)) + stagger
        departure = loading + timedelta(hours=2)
        arrival = departure + timedelta(minutes=120 + (draw // 3) % 601)
        unloading = arrival + timedelta(minutes=15)
        completed = arrival + timedelta(minutes=30 + (draw // 1803) % 91)
        shipment_id = f"SHIP-{day:%Y%m%d}-{store_id}-{product_id}-{leg}"
        yield InventoryEvent(
            store_id,
            product_id,
            datetime.combine(day, inventory_time(23, 59, 59)),
            f"TRC-PLAN-{shipment_id}",
            "SHIPMENT",
            min(capacity, quantity - offset),
            shipment_id,
            shipment_id=shipment_id,
            leg=leg,
            scheduled_ts=datetime.combine(day + timedelta(days=1), inventory_time()),
            loading_ts=loading,
            depart_ts=departure,
            eta=arrival,
            unloading_ts=unloading,
            etd=completed,
        )


def simulate_sku(
    demand: Iterable[Demand],
    *,
    store_id: int,
    product_id: int,
    start_date: date,
    end_date: date,
    seed: int,
    capacity: int,
    daily_prior: float = 1.0,
) -> Iterator[InventoryEvent]:
    """Consume sorted demand with O(one SKU + pending deliveries/returns) state."""
    if capacity < 1 or daily_prior < 0 or end_date < start_date:
        raise ValueError("invalid inventory simulation bounds")
    on_hand = opening_stock(seed, store_id, product_id)
    yield InventoryEvent(
        store_id,
        product_id,
        datetime.combine(start_date, inventory_time()),
        f"TRC-INIT-ST-{store_id}-{product_id}",
        "INITIAL",
        on_hand,
        "SEED",
        balance=on_hand,
    )
    pending: list[tuple[datetime, str, InventoryEvent]] = []
    inbound = 0
    expected_returns = 0
    rate = daily_prior
    iterator = iter(demand)
    candidate = next(iterator, None)
    previous: tuple[datetime, str] | None = None
    day = start_date
    while day <= end_date:
        closing = datetime.combine(day, inventory_time(23, 59, 59))
        attempted = 0
        while candidate is not None and candidate.event_ts <= closing:
            key = (candidate.event_ts, candidate.trace_id)
            if candidate.event_ts.date() < start_date or (previous and key <= previous):
                raise ValueError("demand must have unique, chronological line keys")
            if candidate.quantity <= 0:
                raise ValueError("candidate demand quantity must be positive")
            previous = key
            while pending and pending[0][:2] <= key:
                _, _, event = heappop(pending)
                on_hand += event.quantity
                if event.kind == "RETURN":
                    expected_returns -= event.quantity
                else:
                    inbound -= event.quantity
                event.balance = on_hand
                yield event
            attempted += candidate.quantity
            fulfilled = candidate.quantity <= on_hand
            if fulfilled:
                on_hand -= candidate.quantity
            yield InventoryEvent(
                store_id,
                product_id,
                candidate.event_ts,
                candidate.trace_id,
                "SALE" if fulfilled else "LOST_SALE",
                -candidate.quantity if fulfilled else 0,
                "CUSTOMER_PURCHASE",
                balance=on_hand,
                receipt_id=candidate.receipt_id,
                line_num=candidate.line_num,
                requested_quantity=candidate.quantity,
                unfulfilled_quantity=0 if fulfilled else candidate.quantity,
            )
            if fulfilled and candidate.return_ts is not None:
                if candidate.return_ts <= candidate.event_ts or not candidate.return_id:
                    raise ValueError("restock returns must identify a later return")
                returned = InventoryEvent(
                    store_id,
                    product_id,
                    candidate.return_ts,
                    f"TRC-INV-{candidate.return_id}-{candidate.line_num}",
                    "RETURN",
                    candidate.quantity,
                    candidate.return_id,
                )
                heappush(pending, (returned.event_ts, returned.trace_id, returned))
                expected_returns += returned.quantity
            candidate = next(iterator, None)
        while pending and pending[0][0] <= closing:
            _, _, event = heappop(pending)
            on_hand += event.quantity
            if event.kind == "RETURN":
                expected_returns -= event.quantity
            else:
                inbound -= event.quantity
            event.balance = on_hand
            yield event
        rate = 0.75 * rate + 0.25 * attempted
        point = max(CASE_PACK, ceil(rate * 4))
        target = max(point + CASE_PACK, ceil(rate * 8))
        # Only near-term returns reduce the order. A distant return cannot
        # cover demand before the service lead time has elapsed.
        near_returns = (
            sum(
                event.quantity
                for ts, _, event in pending
                if event.kind == "RETURN" and ts <= closing + timedelta(days=4)
            )
            if expected_returns
            else 0
        )
        position = on_hand + inbound + near_returns
        if position <= point:
            quantity = ceil((target - position) / CASE_PACK) * CASE_PACK
            priority = "URGENT" if on_hand == 0 else ("HIGH" if on_hand < rate * 2 else "NORMAL")
            yield InventoryEvent(
                store_id,
                product_id,
                closing,
                f"TRC-RO-{store_id}-{day}-{product_id}",
                "REORDER",
                quantity,
                "POLICY",
                balance=on_hand,
                reorder_point=point,
                priority=priority,
            )
            for shipment in shipment_events(seed, store_id, product_id, day, quantity, capacity):
                yield shipment
                delivery = InventoryEvent(
                    store_id,
                    product_id,
                    shipment.unloading_ts,
                    f"TRC-SI-{shipment.shipment_id}-{product_id}",
                    "INBOUND_SHIPMENT",
                    shipment.quantity,
                    shipment.shipment_id,
                )
                heappush(pending, (delivery.event_ts, delivery.trace_id, delivery))
                inbound += delivery.quantity
        day += timedelta(days=1)
    if candidate is not None:
        raise ValueError("candidate demand falls outside the simulation history")

# --- retail_setup/generation/fulfillment.py ---
"""Spark adapter for the causal SKU policy and fulfilled receipt finances."""

from dataclasses import dataclass, fields
from datetime import date
from functools import partial
from itertools import groupby
from operator import attrgetter
from typing import Any, Iterator

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


_EVENT_SCHEMA = """
    store_id long, product_id long, event_ts timestamp, trace_id string,
    kind string, quantity long, source string, balance long,
    receipt_id string, line_num long, requested_quantity long,
    unfulfilled_quantity long, reorder_point long, priority string,
    shipment_id string, leg long, scheduled_ts timestamp, loading_ts timestamp,
    depart_ts timestamp, eta timestamp, unloading_ts timestamp, etd timestamp
"""
_FINANCE_COLUMNS = [
    "subtotal_cents",
    "discount_cents",
    "gross_subtotal_cents",
    "tax_cents",
    "total_cents",
    "subtotal_amount",
    "discount_amount",
    "tax_amount",
    "total_amount",
    "Subtotal",
]


@dataclass
class FulfillmentResult:
    sales: dict[str, DataFrame]
    return_headers: DataFrame
    events: DataFrame


_EVENT_FIELDS = tuple(field.name for field in fields(InventoryEvent))
_EVENT_TYPES = dict(
    (name, kind)
    for name, kind in (column.split() for column in _EVENT_SCHEMA.replace("\n", " ").split(","))
)
_DEMAND_COLUMNS = (
    "store_id",
    "product_id",
    "event_ts",
    "receipt_id_ext",
    "line_num",
    "quantity",
    "return_ts",
    "return_id",
)
# Bounded output batches keep Python worker memory flat for very busy SKUs.
_ARROW_BATCH_ROWS = 10_000


def _arrow_event_schema(time_zone: str) -> Any:
    import pyarrow as pa

    arrow_types = {
        "long": pa.int64(),
        "string": pa.string(),
        "timestamp": pa.timestamp("us", tz=time_zone),
    }
    return pa.schema([(name, arrow_types[_EVENT_TYPES[name]]) for name in _EVENT_FIELDS])


def _simulate_arrow_partition(
    batches: Iterator[Any],
    *,
    start_date: date,
    end_date: date,
    seed: int,
    capacity: int,
    daily_prior: float,
    time_zone: str,
) -> Iterator[Any]:
    """Stream sorted (store, SKU) demand through the policy for ``mapInArrow``.

    Arrow moves rows between the JVM and Python in column batches instead of
    pickling one Row/dict per demand line and inventory event, which dominated
    this stage. Timestamps cross the boundary as UTC instants and are handled
    as naive UTC datetimes, exactly as the policy expects.
    """
    import pyarrow as pa

    schema = _arrow_event_schema(time_zone)
    naive = pa.timestamp("us")
    values = attrgetter(*_EVENT_FIELDS)

    def demand_rows() -> Iterator[tuple]:
        for batch in batches:
            columns = []
            for name in _DEMAND_COLUMNS:
                column = batch.column(batch.schema.get_field_index(name))
                if pa.types.is_timestamp(column.type):
                    column = column.cast(naive)
                columns.append(column.to_pylist())
            yield from zip(*columns)

    def flush(rows: list[tuple]) -> Any:
        return pa.RecordBatch.from_arrays(
            [
                pa.array(column, type=schema.field(index).type)
                for index, column in enumerate(zip(*rows))
            ],
            schema=schema,
        )

    pending: list[tuple] = []
    for (store_id, product_id), sku_rows in groupby(
        demand_rows(),
        key=lambda row: (row[0], row[1]),
    ):
        demand = (
            Demand(event_ts, receipt_id, line_num, quantity, return_ts, return_id)
            for _, _, event_ts, receipt_id, line_num, quantity, return_ts, return_id in sku_rows
            if receipt_id is not None
        )
        for event in simulate_sku(
            demand,
            store_id=store_id,
            product_id=product_id,
            start_date=start_date,
            end_date=end_date,
            seed=seed,
            capacity=capacity,
            daily_prior=daily_prior,
        ):
            pending.append(values(event))
            if len(pending) >= _ARROW_BATCH_ROWS:
                yield flush(pending)
                pending = []
    if pending:
        yield flush(pending)


def _fulfilled_finances(
    sales: dict[str, DataFrame],
    events: DataFrame,
    dims: dict[str, DataFrame],
) -> dict[str, DataFrame]:
    accepted = events.filter(F.col("kind") == "SALE").select(
        F.col("receipt_id").alias("receipt_id_ext"),
        "line_num",
    )
    lines = sales["fact_receipt_lines"].join(
        accepted,
        ["receipt_id_ext", "line_num"],
        "left_semi",
    )
    finance_lines = (
        lines.join(
            sales["fact_receipts"].select("receipt_id_ext", "store_id"),
            "receipt_id_ext",
        )
        .join(dims["dim_stores"].select(F.col("ID").alias("store_id"), "tax_rate"), "store_id")
        .join(
            dims["dim_products"].select(F.col("ID").alias("product_id"), "taxability"),
            "product_id",
        )
        .withColumn("_gross", F.col("unit_cents") * F.col("quantity"))
        .withColumn("_bps", F.round(F.col("tax_rate") * 10000).cast("long"))
        .withColumn(
            "_tax_mult",
            F.when(F.col("taxability") == "TAXABLE", 100)
            .when(F.col("taxability") == "REDUCED_RATE", 50)
            .otherwise(0),
        )
        .withColumn(
            "_tax",
            F.floor(
                (
                    F.col("ext_cents").cast("decimal(24,0)") * F.col("_bps") * F.col("_tax_mult")
                    + F.lit(500_000)
                )
                / F.lit(1_000_000),
            ).cast("long"),
        )
    )
    totals = finance_lines.groupBy("receipt_id_ext").agg(
        F.sum("ext_cents").alias("subtotal_cents"),
        F.sum("_gross").alias("gross_subtotal_cents"),
        F.sum("_tax").alias("tax_cents"),
    )
    header = (
        sales["fact_receipts"]
        .drop(*_FINANCE_COLUMNS)
        .join(totals, "receipt_id_ext")
        .withColumn("discount_cents", F.col("gross_subtotal_cents") - F.col("subtotal_cents"))
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
    )
    for name in ("subtotal", "discount", "tax", "total"):
        header = header.withColumn(f"{name}_amount", _fmt(F.col(f"{name}_cents")))
    header = header.withColumn("Subtotal", F.col("subtotal_amount")).select(
        *column_names("fact_receipts"),
    )
    payments = (
        sales["fact_payments"]
        .drop("amount_cents", "amount")
        .join(
            header.select("receipt_id_ext", F.col("total_cents").alias("amount_cents")),
            "receipt_id_ext",
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )
    return {
        "fact_receipts": header,
        "fact_receipt_lines": lines.select(*column_names("fact_receipt_lines")),
        "fact_payments": payments,
    }


def resolve_fulfillment(
    sales_group: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    return_headers: DataFrame | None = None,
) -> FulfillmentResult:
    """Resolve candidate demand before any receipt-dependent fact generation.

    Only small dimension metadata is counted on the driver. Sales are shuffled
    by (store, SKU), sorted, and streamed on workers, never collected/grouped
    into Python lists. ``events`` is cached because all output ledgers and the
    fulfilled receipt filter must consume the exact same policy execution.
    """
    headers = (
        build_return_headers(sales_group, cfg) if return_headers is None else return_headers
    ).cache()
    returns = headers.select(
        F.col("orig_receipt_id_ext").alias("receipt_id_ext"),
        F.col("receipt_id_ext").alias("return_id"),
        F.col("event_ts").alias("return_ts"),
    )
    demand = (
        sales_group["fact_receipt_lines"]
        .join(sales_group["fact_receipts"].select("receipt_id_ext", "store_id"), "receipt_id_ext")
        .join(returns, "receipt_id_ext", "left")
    )
    restock = (
        seeded_draws(cfg.seed).u(
            [F.col("return_id"), F.col("line_num").cast("string")],
            "restock",
        )
        < 0.5
    )
    demand = demand.select(
        "store_id",
        "product_id",
        "event_ts",
        "receipt_id_ext",
        "line_num",
        "quantity",
        F.when(restock, F.col("return_ts")).alias("return_ts"),
        "return_id",
        F.concat(F.lit("TRC-INV-"), "receipt_id_ext", F.lit("-"), "line_num").alias("_sort"),
    )
    # Seed every catalog SKU, not only products that happen to sell in this window.
    seeds = (
        dims["dim_stores"]
        .select(F.col("ID").alias("store_id"))
        .crossJoin(dims["dim_products"].select(F.col("ID").alias("product_id")))
        .select(
            "store_id",
            "product_id",
            F.to_timestamp(F.lit(f"{cfg.start_date} 00:00:00")).alias("event_ts"),
            F.lit(None).cast("string").alias("receipt_id_ext"),
            F.lit(0).alias("line_num"),
            F.lit(0).alias("quantity"),
            F.lit(None).cast("timestamp").alias("return_ts"),
            F.lit(None).cast("string").alias("return_id"),
            F.lit("").alias("_sort"),
        )
    )
    n_products = dims["dim_products"].count()
    if n_products < 1:
        raise ValueError("fulfillment requires a nonempty product catalog")
    ordered = (
        demand.unionByName(seeds)
        .repartition("store_id", "product_id")
        .sortWithinPartitions(
            "store_id",
            "product_id",
            "event_ts",
            "_sort",
        )
    )
    # A theoretical basket prior, not a look-ahead at generated demand.
    spark = sales_group["fact_receipts"].sparkSession
    worker = partial(
        _simulate_arrow_partition,
        start_date=cfg.start_date,
        end_date=cfg.end_date,
        seed=cfg.seed,
        capacity=cfg.truck_capacity,
        daily_prior=cfg.transactions_per_store_day * 10.0 / n_products,
        time_zone=spark.conf.get("spark.sql.session.timeZone"),
    )
    events = (
        ordered.select(*_DEMAND_COLUMNS)
        .mapInArrow(worker, _EVENT_SCHEMA)
        .withColumn(
            "event_date",
            F.to_date("event_ts"),
        )
        .cache()
    )
    return FulfillmentResult(_fulfilled_finances(sales_group, events, dims), headers, events)

# --- retail_setup/generation/store_activity.py ---
"""Store ops + foot traffic facts, Spark-native.

fact_store_ops: exactly two rows (opened/closed) per store-day, with the
open/close hours parsed from `dim_stores.operating_hours` over the four
known formats ("6-22", "7-22", "7-23", "24h"). Dec 25 is skipped entirely
(stores closed for Christmas, any year).

Footfall uses the same independently generated visitors as receipt conversion.
Zone counts conserve those visitors exactly, including non-purchasing visits.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# operating_hours literal -> (open_hour, close_hour); "24h" maps to 0/24
OPERATING_HOURS_MAP = {"6-22": (6, 22), "7-22": (7, 22), "7-23": (7, 23), "24h": (0, 24)}

BASE_CONVERSION = 0.20
PEAK_HOURS = (12, 13, 17, 18, 19)
PEAK_MULTIPLIER = 1.3
WEEKEND_MULTIPLIER = 0.9
# Baseline browsers per store-hour (scaled by store size + hour-of-day) so that
# open hours with zero receipts still emit foot traffic.
BASE_HOURLY_BROWSERS = 8

# zone -> (dwell_lo_seconds, dwell_hi_seconds)
ZONE_DWELL = {
    "ENTRANCE_MAIN": (45, 90),
    "ENTRANCE_SIDE": (30, 75),
    "AISLES_A": (180, 420),
    "AISLES_B": (120, 300),
    "CHECKOUT": (90, 240),
}
ZONES = list(ZONE_DWELL)

# store_format -> zone share, ordered as ZONES
FORMAT_ZONE_SHARES = {
    "hypermarket": [0.20, 0.10, 0.35, 0.25, 0.10],
    "superstore": [0.25, 0.10, 0.30, 0.25, 0.10],
    "standard": [0.30, 0.15, 0.25, 0.15, 0.15],
    "neighborhood": [0.35, 0.15, 0.20, 0.10, 0.20],
    "express": [0.45, 0.15, 0.10, 0.05, 0.25],
}


def generate_store_ops(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Two rows (opened/closed) per store-day, skipping Dec 25 of any year."""

    stores = dims["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
    store_ids = [r.store_id for r in stores.select("store_id").collect()]

    grid = (
        store_day_grid(spark, store_ids, cfg.start_date, cfg.end_date, cfg.seed, "store_ops")
        .filter(~((F.month("day") == 12) & (F.dayofmonth("day") == 25)))
        .join(stores, "store_id")
    )

    # operating_hours -> open/close hour via a F.when chain over known formats
    open_hour, close_hour = None, None
    for literal, (o, c) in OPERATING_HOURS_MAP.items():
        cond = F.col("operating_hours") == literal
        open_hour = F.when(cond, o) if open_hour is None else open_hour.when(cond, o)
        close_hour = F.when(cond, c) if close_hour is None else close_hour.when(cond, c)
    # unknown formats fall back to standard hours instead of silent NULL event_ts
    grid = grid.withColumn("open_hour", open_hour.otherwise(6)).withColumn(
        "close_hour", close_hour.otherwise(22)
    )

    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    open_ts = F.timestamp_seconds(day_ts + F.col("open_hour") * 3600)
    # 24h stores "close" at 23:59:00 the same day; others at close_hour:00
    close_secs = F.when(F.col("close_hour") == 24, F.lit(23 * 3600 + 59 * 60)).otherwise(
        F.col("close_hour") * 3600
    )
    close_ts = F.timestamp_seconds(day_ts + close_secs)

    ops = grid.select(
        "store_id",
        F.col("day").alias("event_date"),
        F.explode(
            F.array(
                F.struct(F.lit("opened").alias("operation_type"), open_ts.alias("event_ts")),
                F.struct(F.lit("closed").alias("operation_type"), close_ts.alias("event_ts")),
            )
        ).alias("op"),
    ).select(
        F.col("op.event_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-OPS-"),
            F.col("store_id"),
            F.lit("-"),
            F.col("event_date"),
            F.lit("-"),
            F.col("op.operation_type"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("op.operation_type").alias("operation_type"),
        "event_date",
    )
    return ops.select(*column_names("fact_store_ops"))


def generate_foot_traffic(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Per store-hour-zone sensor counts generated before purchase decisions."""

    d = seeded_draws(cfg.seed)

    del receipts
    profile = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type).profile
    grid = generate_hourly_visits(spark, dims["dim_stores"], profile, cfg)
    store_attrs = dims["dim_stores"].select(F.col("ID").alias("store_id"), "store_format")
    hourly = grid.withColumnRenamed("visitors", "total_visitors").join(store_attrs, "store_id")

    # per-format share column for each zone, then explode the 5-zone structs
    zone_structs = []
    for i, zone in enumerate(ZONES):
        lower = None
        upper = None
        for fmt, shares in FORMAT_ZONE_SHARES.items():
            cond = F.col("store_format") == fmt
            lo_share, hi_share = sum(shares[:i]), sum(shares[: i + 1])
            lower = F.when(cond, lo_share) if lower is None else lower.when(cond, lo_share)
            upper = F.when(cond, hi_share) if upper is None else upper.when(cond, hi_share)
        lo, hi = ZONE_DWELL[zone]
        zone_structs.append(
            F.struct(
                F.lit(zone).alias("zone"),
                lower.alias("lower"),
                upper.alias("upper"),
                F.lit(lo).alias("dwell_lo"),
                F.lit(hi).alias("dwell_hi"),
            )
        )

    ft = hourly.select(
        "store_id",
        "hour_ts",
        "total_visitors",
        F.explode(F.array(*zone_structs)).alias("z"),
    )

    dwell_u = d.u([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_dwell")
    ft = ft.select(
        F.col("hour_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-FT-"),
            F.col("store_id"),
            F.lit("-"),
            F.date_format("hour_ts", "yyyyMMddHH"),
            F.lit("-"),
            F.col("z.zone"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.format_string("SENSOR_%03d_%s", F.col("store_id"), F.col("z.zone")).alias("sensor_id"),
        F.col("z.zone").alias("zone"),
        F.floor(F.col("z.dwell_lo") + dwell_u * (F.col("z.dwell_hi") - F.col("z.dwell_lo") + 1))
        .cast("long")
        .alias("dwell_seconds"),
        (
            F.round(F.col("total_visitors") * F.col("z.upper"))
            - F.round(F.col("total_visitors") * F.col("z.lower"))
        )
        .cast("long")
        .alias("count"),
        F.to_date("hour_ts").alias("event_date"),
    )
    return ft.select(*column_names("fact_foot_traffic"))

# --- retail_setup/generation/online_orders.py ---
"""Online orders fact group (headers, lines, payments stream), Spark-native.

Mirrors `receipts.py`: all randomness flows through `runtime.seeded_draws`
(xxhash64-based, partition-arrangement independent), money is integer cents,
and daily volume is negative-binomial, allowing overdispersed counts and zero days.

Documented decisions (per plan 2b, Task 5):
- Tax uses the exact integer basis-point formula from receipts. Store/BOPIS
  lines use the fulfilment store's rate; DC-shipped lines use the customer's
  geography rate, with the network mean as fallback.
- Backorders: a small fraction of non-BOPIS lines ship but are not yet
  delivered (fulfillment_status SHIPPED, delivered_ts NULL), mirroring
  datagen's ``"DELIVERED" if not is_backordered else "SHIPPED"``.
- Product picks use the shared launch-eligible popularity model and seasonal
  department weights.
- CANCELLED orders get NO payment row (no pay-then-refund), which preserves
  the `payment.amount == header.total` invariant.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F


# Online tender mix per plan: 60% CC / 25% DC / 10% PAYPAL / 5% OTHER.
# (method, mix weight, decline multiplier, processing_ms lo, processing_ms hi)
# CC/DC constants carried from the Plan-2a TENDERS table; PAYPAL/OTHER are the
# online-only additions (PAYPAL x1.1 decline, OTHER x0.5 decline).
ONLINE_TENDERS = [
    ("CREDIT_CARD", 0.60, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.25, 0.8, 1200, 3500),
    ("PAYPAL", 0.10, 1.1, 2000, 5000),
    ("OTHER", 0.05, 0.5, 1500, 4000),
]

CANCEL_RATE = 0.02
# Fraction of shippable (non-BOPIS) lines that backorder: ship but not yet
# delivered. Approximates datagen's inventory-driven backorder check, which
# utility cannot evaluate here (the inventory chain is generated afterwards).
BACKORDER_RATE = 0.08

# basket-size buckets: 60% -> 1-3 lines, 30% -> 2-5, 10% -> 5-8
BASKET_BUCKETS = [("S", 0.60, 1, 3), ("M", 0.30, 2, 5), ("L", 0.10, 5, 8)]

# promo codes with matching percentage discounts (5/10/20%)
PROMOS = [("PROMO05", 5), ("PROMO10", 10), ("PROMO20", 20)]


def generate_online_orders(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_online_order_headers, fact_online_order_lines, payments.

    Payments are returned under the "payments" key for the orchestrator to
    union with the in-store stream before writing fact_payments.
    """
    d = seeded_draws(cfg.seed)

    # Destination-based tax: per-customer-geography rate (mean store tax_rate in
    # that geography), with the network mean as the fallback for geographies
    # that have no store.
    mean_rate = dims["dim_stores"].agg(F.avg("tax_rate")).first()[0]
    mean_bps = int(round(float(mean_rate) * 10000))
    geo_rate = dims["dim_stores"].groupBy("GeographyID").agg(F.avg("tax_rate").alias("geo_rate"))
    cust_rate = (
        dims["dim_customers"]
        .select(F.col("ID").alias("customer_id"), "GeographyID")
        .join(geo_rate, "GeographyID", "left")
        .select(
            "customer_id",
            F.coalesce(F.round(F.col("geo_rate") * 10000), F.lit(mean_bps))
            .cast("long")
            .alias("rate_bps"),
        )
    )

    dc_ids = sorted(r.ID for r in dims["dim_distribution_centers"].select("ID").collect())
    store_ids = sorted(r.ID for r in dims["dim_stores"].select("ID").collect())
    dc_arr = F.array(*[F.lit(int(i)).cast("long") for i in dc_ids])
    st_arr = F.array(*[F.lit(int(i)).cast("long") for i in store_ids])

    # --- day grid: network-wide daily volume, monthly-weight scaled, clamped normal
    mw = profile.monthly_weights
    m_mean = sum(mw) / 12.0
    days = date_grid(spark, cfg.start_date, cfg.end_date)
    monthly_w = F.element_at(F.array(*[F.lit(w / m_mean) for w in mw]), F.month("day"))
    lam = F.lit(float(cfg.online_orders_per_day)) * monthly_w
    n_orders = d.negative_binomial(["day"], "onl_n", lam, 24).cast("int")

    # --- orders: ids, timestamps, customer, tender, cancellation, basket size
    bucket = d.pick_by_weights(
        ["day", "seq"], "onl_bucket", [(name, w) for name, w, _, _ in BASKET_BUCKETS]
    )
    basket_n: Column = F.lit(None).cast("int")
    for name, _, lo, hi in BASKET_BUCKETS:
        basket_n = F.when(
            bucket == name,
            (d.h64(["day", "seq"], f"onl_size_{name}") % (hi - lo + 1) + lo).cast("int"),
        ).otherwise(basket_n)

    orders = (
        days.withColumn("n_orders", n_orders)
        .filter(F.col("n_orders") > 0)
        .withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_orders"))))
        .withColumn("hour", (d.h64(["day", "seq"], "onl_hour") % 24).cast("int"))
        .withColumn("minute", (d.h64(["day", "seq"], "onl_min") % 60).cast("int"))
        .withColumn("second", (d.h64(["day", "seq"], "onl_sec") % 60).cast("int"))
        .withColumn(
            "event_ts",
            F.make_timestamp(
                F.year("day"),
                F.month("day"),
                F.dayofmonth("day"),
                F.col("hour"),
                F.col("minute"),
                F.col("second"),
            ),
        )
        .withColumn("event_date", F.col("day"))
        # ONL + yyyyMMdd + 5-digit seq + 3-digit draw; unique because (day, seq)
        # is a key within the grid.
        .withColumn(
            "order_id_ext",
            F.concat(
                F.lit("ONL"),
                F.date_format("day", "yyyyMMdd"),
                fixed_width_decimal(F.col("seq"), 5, "daily online-order sequence"),
                fixed_width_decimal(
                    d.h64(["day", "seq"], "onl_rand") % 1000,
                    3,
                    "online-order random suffix",
                ),
            ),
        )
        .withColumn(
            "customer_id",
            (d.h64(["order_id_ext"], "onl_cust") % F.lit(cfg.customer_count) + 1).cast("long"),
        )
        .withColumn(
            "tender_type",
            d.pick_by_weights(
                ["order_id_ext"], "onl_tender", [(n, w) for n, w, _, _, _ in ONLINE_TENDERS]
            ),
        )
        .withColumn("is_cancelled", d.u(["order_id_ext"], "onl_cancel") < F.lit(CANCEL_RATE))
        # per-order promo intensity: 10-30% of its lines carry a promo
        .withColumn("promo_rate", d.u(["order_id_ext"], "onl_prate") * F.lit(0.20) + F.lit(0.10))
        .withColumn("basket_n", basket_n)
        # destination-based tax rate for this order's customer
        .join(cust_rate, "customer_id", "left")
        .withColumn("rate_bps", F.coalesce(F.col("rate_bps"), F.lit(mean_bps)))
    )

    # --- lines: shared popularity over products launched by the order date
    products = dims["dim_products"].select(
        F.col("ID").alias("product_id"),
        F.col("SalePrice"),
        F.col("taxability"),
        F.col("Department").alias("department"),
        F.to_date("LaunchDate").alias("launch_date"),
    )
    order_dates = orders.select("event_date").distinct()
    catalog = retail_catalog(products, order_dates, d)

    qty = retail_quantity(d, ["order_id_ext", "line_num"], F.col("department"))

    promo_idx = (d.h64(["order_id_ext", "line_num"], "onl_pidx") % len(PROMOS)).cast("int")
    promo_code_arr = F.array(*[F.lit(c) for c, _ in PROMOS])
    promo_pct_arr = F.array(*[F.lit(p).cast("long") for _, p in PROMOS])

    # per-line fulfillment: 60% SHIP_FROM_DC / 30% SHIP_FROM_STORE / 10% BOPIS
    mode = d.pick_by_weights(
        ["order_id_ext", "line_num"],
        "onl_mode",
        [("SHIP_FROM_DC", 0.60), ("SHIP_FROM_STORE", 0.30), ("BOPIS", 0.10)],
    )

    # store tax rate (bps) for destination-based tax on store-fulfilled lines
    store_bps = dims["dim_stores"].select(
        F.col("ID").alias("_snid"),
        F.round(F.col("tax_rate") * 10000).cast("long").alias("_store_bps"),
    )

    candidates = orders.select(
        "order_id_ext",
        "event_ts",
        "event_date",
        "is_cancelled",
        "promo_rate",
        "basket_n",
        "rate_bps",
    ).withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    candidates = _with_seasonal_department(
        candidates,
        d.u(["order_id_ext", "line_num"], "onl-department"),
        profile.department_weights,
    )
    lines = (
        candidates.join(catalog, ["event_date", "department"])
        .withColumn(
            "_product",
            pick_retail_product(F.col("_catalog"), d.u(["order_id_ext", "line_num"], "onl_prod")),
        )
        .select("*", "_product.product_id", "_product.SalePrice", "_product.taxability")
        .drop("_product", "_catalog")
        .withColumn("quantity", qty.cast("long"))
        .withColumn("unit_cents", F.round(F.col("SalePrice") * 100).cast("long"))
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn(
            "has_promo", d.u(["order_id_ext", "line_num"], "onl_promo") < F.col("promo_rate")
        )
        .withColumn(
            "promo_code", F.when(F.col("has_promo"), F.element_at(promo_code_arr, promo_idx + 1))
        )
        .withColumn(
            "promo_pct",
            F.when(F.col("has_promo"), F.element_at(promo_pct_arr, promo_idx + 1)).otherwise(
                F.lit(0).cast("long")
            ),
        )
        # matching 5/10/20% discount, rounded half-up in integer cents
        .withColumn(
            "discount_cents",
            F.floor((F.col("ext_before") * F.col("promo_pct") + F.lit(50)) / F.lit(100)).cast(
                "long"
            ),
        )
        .withColumn("ext_cents", F.col("ext_before") - F.col("discount_cents"))
        # fulfillment + node resolved before tax so the destination rate is known
        .withColumn("fulfillment_mode", mode)
        .withColumn(
            "node_type",
            F.when(F.col("fulfillment_mode") == "SHIP_FROM_DC", "DC").otherwise("STORE"),
        )
        .withColumn(
            "node_id",
            F.when(
                F.col("node_type") == "DC",
                F.element_at(
                    dc_arr,
                    (d.h64(["order_id_ext", "line_num"], "onl_node") % len(dc_ids) + 1).cast("int"),
                ),
            ).otherwise(
                F.element_at(
                    st_arr,
                    (d.h64(["order_id_ext", "line_num"], "onl_node") % len(store_ids) + 1).cast(
                        "int"
                    ),
                )
            ),
        )
        # destination-based tax (datagen parity): STORE/BOPIS lines use the
        # fulfilling store's rate; DC lines use the customer-geography rate.
        .join(store_bps, F.col("node_id") == F.col("_snid"), "left")
        .withColumn(
            "line_rate_bps",
            F.when(
                F.col("node_type") == "STORE", F.coalesce(F.col("_store_bps"), F.col("rate_bps"))
            ).otherwise(F.col("rate_bps")),
        )
        .withColumn(
            "tax_mult",
            F.when(F.col("taxability") == "TAXABLE", 100)
            .when(F.col("taxability") == "REDUCED_RATE", 50)
            .otherwise(0)
            .cast("long"),
        )
        .withColumn(
            "line_tax_cents",
            F.floor(
                (F.col("ext_cents") * F.col("line_rate_bps") * F.col("tax_mult") + F.lit(500_000))
                / F.lit(1_000_000)
            ).cast("long"),
        )
        # Backorder a small fraction of shippable (non-BOPIS, non-cancelled)
        # lines: they ship but are not yet delivered (datagen SHIPPED state).
        .withColumn(
            "is_backordered",
            (F.col("fulfillment_mode") != "BOPIS")
            & ~F.col("is_cancelled")
            & (d.u(["order_id_ext", "line_num"], "onl_backorder") < F.lit(BACKORDER_RATE)),
        )
        .withColumn(
            "fulfillment_status",
            F.when(F.col("is_cancelled"), "CANCELLED")
            .when(F.col("is_backordered"), "SHIPPED")
            .otherwise("DELIVERED"),
        )
        .drop("_snid", "_store_bps")
    )

    # --- lifecycle (non-cancelled lines only; cancelled lines keep NULL ts)
    is_bopis = F.col("fulfillment_mode") == "BOPIS"
    u_pick = d.u(["order_id_ext", "line_num"], "onl_pick")
    # BOPIS: picked 4-24h after order; ship modes: 30-240 min
    pick_secs = F.when(
        is_bopis, (F.lit(4 * 3600) + u_pick * F.lit(20 * 3600.0)).cast("long")
    ).otherwise((F.lit(30 * 60) + u_pick * F.lit(210 * 60.0)).cast("long"))
    ship_secs = (
        F.lit(2 * 3600) + d.u(["order_id_ext", "line_num"], "onl_ship") * F.lit(2 * 3600.0)
    ).cast("long")
    deliver_secs = (
        F.lit(1 * 86400) + d.u(["order_id_ext", "line_num"], "onl_dlv") * F.lit(2 * 86400.0)
    ).cast("long")
    live = ~F.col("is_cancelled")
    lines = (
        lines.withColumn(
            "picked_ts", F.when(live, F.timestamp_seconds(F.unix_timestamp("event_ts") + pick_secs))
        )
        # BOPIS never ships: shipped_ts stays NULL
        .withColumn(
            "shipped_ts",
            F.when(
                live & ~is_bopis, F.timestamp_seconds(F.unix_timestamp("picked_ts") + ship_secs)
            ),
        )
        .withColumn(
            "delivered_ts",
            F.when(F.col("is_backordered"), F.lit(None).cast("timestamp"))
            .when(live & is_bopis, F.col("picked_ts"))
            .when(live, F.timestamp_seconds(F.unix_timestamp("shipped_ts") + deliver_secs)),
        )
    )

    # TMDL-bound legacy pandas-index column: hash-derived via legacy_index.
    fact_online_order_lines = (
        lines.withColumn("__index_level_0__", legacy_index("order_id_ext", "line_num"))
        .select(
            F.col("order_id_ext").alias("order_id"),
            "product_id",
            F.col("line_num").cast("long").alias("line_num"),
            "quantity",
            _fmt(F.col("unit_cents")).alias("unit_price"),
            "unit_cents",
            _fmt(F.col("ext_cents")).alias("ext_price"),
            "ext_cents",
            "promo_code",
            "fulfillment_mode",
            "fulfillment_status",
            "node_type",
            "node_id",
            "picked_ts",
            "shipped_ts",
            "delivered_ts",
            "event_ts",
            "event_date",
            "__index_level_0__",
        )
        .select(*column_names("fact_online_order_lines"))
    )

    # --- headers: sum live lines; cancelled orders have financials zeroed
    sums = (
        lines.filter(live)
        .groupBy("order_id_ext")
        .agg(
            F.sum("ext_cents").alias("subtotal_cents"),
            F.sum("discount_cents").alias("discount_cents"),
            F.sum("line_tax_cents").alias("tax_cents"),
        )
    )
    headers_base = (
        orders.join(sums, "order_id_ext", "left")
        .withColumn("subtotal_cents", F.coalesce(F.col("subtotal_cents"), F.lit(0)).cast("long"))
        .withColumn("discount_cents", F.coalesce(F.col("discount_cents"), F.lit(0)).cast("long"))
        .withColumn("tax_cents", F.coalesce(F.col("tax_cents"), F.lit(0)).cast("long"))
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("payment_method", F.col("tender_type"))
        # IMP-007: subtotal_cents stays net (existing behavior); gross recovers
        # the pre-discount total. Attribution fields default NULL; enriched for
        # the ~5% of orders selected for attribution (attribution.py).
        .withColumn("gross_subtotal_cents", F.col("subtotal_cents") + F.col("discount_cents"))
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
    )
    fact_online_order_headers = headers_base.select(
        "order_id_ext",
        "customer_id",
        "subtotal_cents",
        "tax_cents",
        "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"),
        "payment_method",
        "event_ts",
        "event_date",
        "gross_subtotal_cents",
        "discount_cents",
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    ).select(*column_names("fact_online_order_headers"))

    # --- payments: one per NON-cancelled order; receipt_id_ext/store_id NULL
    u_dec = d.u(["order_id_ext"], "onl_decline")
    decline_p: Column = F.lit(0.0)
    for name, _, mult, _, _ in ONLINE_TENDERS:
        decline_p = F.when(F.col("payment_method") == name, F.lit(BASE_DECLINE * mult)).otherwise(
            decline_p
        )
    proc_lo: Column = F.lit(ONLINE_TENDERS[-1][3])
    proc_hi: Column = F.lit(ONLINE_TENDERS[-1][4])
    for name, _, _, lo, hi in ONLINE_TENDERS[:-1]:
        proc_lo = F.when(F.col("payment_method") == name, F.lit(lo)).otherwise(proc_lo)
        proc_hi = F.when(F.col("payment_method") == name, F.lit(hi)).otherwise(proc_hi)
    reason_idx = (d.h64(["order_id_ext"], "onl_reason") % len(DECLINE_REASONS)).cast("int")
    payments = (
        headers_base.filter(~F.col("is_cancelled"))
        .withColumn("receipt_id_ext", F.lit(None).cast("string"))
        .withColumn("store_id", F.lit(None).cast("long"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit("TXN_"),
                F.unix_timestamp("event_ts").cast("string"),
                F.lit("_"),
                fixed_width_decimal(
                    d.h64(["order_id_ext"], "onl_txn") % 1_000_000,
                    6,
                    "online transaction suffix",
                ),
            ),
        )
        .withColumn("status", F.when(u_dec < decline_p, "DECLINED").otherwise("APPROVED"))
        .withColumn(
            "decline_reason",
            F.when(
                F.col("status") == "DECLINED",
                F.element_at(F.array(*[F.lit(r) for r in DECLINE_REASONS]), reason_idx + 1),
            ),
        )
        .withColumn(
            "processing_time_ms",
            (proc_lo + d.u(["order_id_ext"], "onl_proc") * (proc_hi - proc_lo)).cast("long"),
        )
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_online_order_headers": fact_online_order_headers,
        "fact_online_order_lines": fact_online_order_lines,
        "payments": payments,
    }

# --- retail_setup/generation/promotions.py ---
"""Promotions fact group, derived purely from receipt lines (no draws).

Semantics (datagen promotions_mixin):

- ``fact_promo_lines``: one row per SALE receipt line that has a non-NULL
  ``promo_code`` and a positive line discount
  (``discount_cents = unit_cents * quantity - ext_cents > 0``). Event fields
  come from the line; ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' +
  promo_code + '-' + line_num``.
- ``fact_promotions``: one row per (receipt_id_ext, promo_code) aggregating
  those lines — discount sums, distinct ``product_count``, ``product_ids`` as
  a comma-joined sorted id list. ``discount_type`` is 'BOGO' for BOGO codes and
  'PERCENTAGE' for all other generated promo codes.
  Store/customer/event fields are joined from the receipt header;
  ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' + promo_code``.

TMDL contract note: ``fact_promo_lines`` carries dual columns — the
snake_case plan names plus TMDL-bound PascalCase mirrors (ReceiptId,
PromoCode, LineNumber, ProductID, Qty, DiscountAmount, DiscountCents) and the
legacy pandas-index column ``__index_level_0__`` (hash-derived via
legacy_index(receipt_id_ext, promo_code, line_number)). The PascalCase columns
are exact copies of their snake_case twins.

Pure joins/groupBys over the sales group — deterministic, no randomness.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



def generate_promotions(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
) -> tuple[DataFrame, DataFrame]:
    """Derive (fact_promotions, fact_promo_lines) from the sales group."""

    sale_ids = (
        sales_group["fact_receipts"]
        .filter(F.col("receipt_type") == "SALE")
        .select("receipt_id_ext")
    )

    # --- promo lines: discounted SALE lines with a promo code
    lines = (
        sales_group["fact_receipt_lines"]
        .join(sale_ids, "receipt_id_ext")
        .filter(F.col("promo_code").isNotNull())
        .withColumn(
            "discount_cents",
            F.col("unit_cents") * F.col("quantity") - F.col("ext_cents"),
        )
        .filter(F.col("discount_cents") > 0)
        .withColumn("line_number", F.col("line_num").cast("long"))
        .withColumn("quantity", F.col("quantity").cast("long"))
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-PRM-"),
                F.col("receipt_id_ext"),
                F.lit("-"),
                F.col("promo_code"),
                F.lit("-"),
                F.col("line_number").cast("string"),
            ),
        )
    )

    promo_lines = (
        lines
        # TMDL-bound PascalCase mirrors of the snake_case columns
        .withColumn("ReceiptId", F.col("receipt_id_ext"))
        .withColumn("PromoCode", F.col("promo_code"))
        .withColumn("LineNumber", F.col("line_number"))
        .withColumn("ProductID", F.col("product_id"))
        .withColumn("Qty", F.col("quantity"))
        .withColumn("DiscountAmount", F.col("discount_amount"))
        .withColumn("DiscountCents", F.col("discount_cents"))
        # Legacy pandas-index column bound by the semantic model
        .withColumn(
            "__index_level_0__", legacy_index("receipt_id_ext", "promo_code", "line_number")
        )
        .select(*column_names("fact_promo_lines"))
    )

    # --- promotions: aggregate per (receipt_id_ext, promo_code)
    agg = lines.groupBy("receipt_id_ext", "promo_code").agg(
        F.sum("discount_cents").alias("discount_cents"),
        F.countDistinct("product_id").alias("product_count"),
        F.concat_ws(",", F.sort_array(F.collect_set("product_id"))).alias("product_ids"),
    )

    headers = sales_group["fact_receipts"].select(
        "receipt_id_ext", "event_ts", "event_date", "store_id", "customer_id"
    )
    promotions = (
        agg.join(headers, "receipt_id_ext")
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        # BOGO codes are buy-one-get-one; everything else is a percentage off.
        .withColumn(
            "discount_type",
            F.when(F.col("promo_code").startswith("BOGO"), F.lit("BOGO")).otherwise(
                F.lit("PERCENTAGE")
            ),
        )
        .withColumn(
            "trace_id",
            F.concat(F.lit("TRC-PRM-"), F.col("receipt_id_ext"), F.lit("-"), F.col("promo_code")),
        )
        # IMP-007: NULL unless the underlying receipt was selected for
        # attribution; enriched in attribution.py.
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .select(*column_names("fact_promotions"))
    )

    return promotions, promo_lines

# --- retail_setup/generation/marketing.py ---
"""Marketing impressions fact, Spark-native.

Ports datagen's marketing_campaign.py constants: four campaign archetypes,
each with its own channel mix and base impressions/day. Daily volumes are
scaled by `cfg.store_count / 86` — 86 is the legacy datagen fleet size, so a
fleet of 86 stores reproduces datagen's nominal volumes. `flash_sale` runs
only ~1 day in 7 (a per-day uniform gate, u < 1/7).

Per impression: channel uniform over the archetype's channels; device
MOBILE/DESKTOP/TABLET at 60/30/10 with cost multipliers 1.2/0.8/0.9; cost a
uniform draw within the channel's dollar band times the device multiplier;
a uniformly-assigned customer supplies `customer_ad_id` (dim_customers.AdId),
with `customer_id` populated for ~5% of impressions (CRM match) else NULL.
All draws via `runtime.seeded_draws`, so output is deterministic for a
(config, seed) pair regardless of cluster shape.

Schema note: fact_marketing is TMDL-bound with DUAL columns — PascalCase
`CustomerId`/`CostCents` (the TMDL sourceColumns) are exact mirrors of the
snake_case plan columns `customer_id`/`cost_cents`, and `__index_level_0__`
is the legacy pandas index (hash-derived via legacy_index).
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# (archetype, channels, base impressions/day at the legacy 86-store fleet,
#  campaign duration in days — datagen MarketingCampaignSimulator templates)
ARCHETYPES: list[tuple[str, list[str], int, int]] = [
    ("seasonal_sale", ["FACEBOOK", "GOOGLE", "EMAIL"], 1000, 7),
    ("product_launch", ["INSTAGRAM", "YOUTUBE", "DISPLAY"], 2000, 14),
    ("loyalty_program", ["EMAIL", "SOCIAL"], 500, 30),
    ("flash_sale", ["SOCIAL", "SEARCH"], 5000, 1),  # runs ~1 day in 7
]

# channel -> (lo, hi) cost-per-impression band in dollars (uniform draw)
CHANNEL_COSTS: dict[str, tuple[float, float]] = {
    "EMAIL": (0.005, 0.05),
    "DISPLAY": (0.10, 0.50),
    "SOCIAL": (0.08, 0.25),
    "SEARCH": (0.15, 0.75),
    "FACEBOOK": (0.10, 0.40),
    "GOOGLE": (0.25, 1.50),
    "INSTAGRAM": (0.08, 0.35),
    "YOUTUBE": (0.30, 1.50),
}

LEGACY_FLEET_SIZE = 86  # datagen's store count; volumes scale store_count/86

DEVICE_WEIGHTS = [("MOBILE", 0.6), ("DESKTOP", 0.3), ("TABLET", 0.1)]
DEVICE_MULTIPLIERS = {"MOBILE": 1.2, "DESKTOP": 0.8, "TABLET": 0.9}


def generate_marketing(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Ad-impression facts: day x archetype grid exploded to impressions."""

    d = seeded_draws(cfg.seed)
    scale = cfg.store_count / LEGACY_FLEET_SIZE

    # --- Spark-native day x archetype grid
    rows = [
        (idx + 1, name, channels, base, dur)
        for idx, (name, channels, base, dur) in enumerate(ARCHETYPES)
    ]
    archetypes = spark.createDataFrame(
        rows,
        "archetype_idx int, archetype string, "
        "channels array<string>, base_impressions int, duration int",
    )
    grid = (
        date_grid(spark, cfg.start_date, cfg.end_date)
        .withColumn(
            "day_offset",
            F.datediff(F.col("day"), F.lit(cfg.start_date)),
        )
        .crossJoin(archetypes)
        .withColumn(
            # Campaigns span their archetype duration: the day's impressions
            # belong to the window [start + k*duration, ...) it falls in.
            "campaign_start",
            F.date_add(
                F.lit(cfg.start_date),
                (F.col("day_offset") / F.col("duration")).cast("int") * F.col("duration"),
            ),
        )
    )

    # flash_sale runs ~1 day in 7 (uniform gate per day)
    grid = grid.filter(
        (F.col("archetype") != "flash_sale")
        | (d.u([F.col("day")], "mk_flash_gate") < F.lit(1.0 / 7.0))
    )

    # per-day impression count: clamped normal around the scaled base
    lam = F.col("base_impressions") * F.lit(scale)
    n_imps = F.greatest(
        F.lit(1),
        F.round(lam + d.gauss(["day", "archetype"], "mk_n") * F.sqrt(lam)),
    ).cast("int")
    grid = grid.withColumn("n_impressions", n_imps)

    # --- explode to impressions; build IDs
    imps = (
        grid.withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_impressions"))))
        .withColumn(
            # campaign_id keyed on the campaign window so it spans multiple days
            "campaign_id",
            F.concat(
                F.lit("CAMP"),
                F.date_format("campaign_start", "yyyyMMdd"),
                fixed_width_decimal(F.col("archetype_idx"), 2, "campaign archetype index"),
            ),
        )
        .withColumn(
            # impression id stays day-unique (campaign_id now repeats across days)
            "impression_id_ext",
            F.concat(
                F.lit("IMP"),
                F.date_format("day", "yyyyMMdd"),
                fixed_width_decimal(F.col("archetype_idx"), 2, "impression archetype index"),
                fixed_width_decimal(F.col("seq"), 7, "daily impression sequence"),
            ),
        )
        # drops the 'IMP' prefix: CREAT + day + archetype + seq
        .withColumn(
            "creative_id",
            F.concat(F.lit("CREAT"), F.substring("impression_id_ext", 4, 30)),
        )
    )

    key = [F.col("impression_id_ext")]

    # --- channel uniform over the archetype's channels
    ch_idx = F.floor(d.u(key, "mk_channel") * F.size("channels")).cast("int")
    imps = imps.withColumn("channel", F.element_at("channels", ch_idx + 1))

    # --- device pick (60/30/10) and cost multiplier
    imps = imps.withColumn("device", d.pick_by_weights(key, "mk_device", DEVICE_WEIGHTS))
    mult = None
    for device, m in DEVICE_MULTIPLIERS.items():
        cond = F.col("device") == device
        mult = F.when(cond, m) if mult is None else mult.when(cond, m)
    imps = imps.withColumn("device_mult", mult)

    # --- cost: uniform within the channel band, times the device multiplier
    lo, hi = None, None
    for channel, (c_lo, c_hi) in CHANNEL_COSTS.items():
        cond = F.col("channel") == channel
        lo = F.when(cond, c_lo) if lo is None else lo.when(cond, c_lo)
        hi = F.when(cond, c_hi) if hi is None else hi.when(cond, c_hi)
    cost_dollars = (lo + d.u(key, "mk_cost") * (hi - lo)) * F.col("device_mult")
    imps = imps.withColumn("cost_cents", F.round(cost_dollars * 100).cast("long")).withColumn(
        "cost", F.format_string("%.2f", cost_dollars)
    )

    # --- uniform customer per impression; ~5% CRM-matched (customer_id set)
    imps = imps.withColumn(
        "customer_pick",
        (d.h64(key, "mk_customer") % F.lit(cfg.customer_count) + 1).cast("long"),
    )
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_pick"), F.col("AdId").alias("customer_ad_id")
    )
    imps = imps.join(customers, "customer_pick", "left")
    imps = imps.withColumn(
        "customer_id",
        F.when(
            d.u(key, "mk_crm") < F.lit(0.05),
            F.col("customer_pick").cast("double"),
        ),
    )

    # --- event_ts uniform within the day
    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    event_ts = F.timestamp_seconds(day_ts + F.floor(d.u(key, "mk_ts") * F.lit(86400)))

    out = (
        imps.select(
            event_ts.alias("event_ts"),
            F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")).alias("trace_id"),
            "channel",
            "campaign_id",
            "creative_id",
            "customer_ad_id",
            "customer_id",
            "cost_cents",
            # TMDL-bound PascalCase duals: exact mirrors of the snake_case columns
            F.col("customer_id").alias("CustomerId"),
            F.col("cost_cents").alias("CostCents"),
            "impression_id_ext",
            "cost",
            "device",
            F.col("day").alias("event_date"),
        )
        .withColumn(
            "__index_level_0__",
            legacy_index("impression_id_ext"),
        )
        .withColumn(
            # IMP-007: NULL for organic/background impressions. attribution.py
            # unions in the two synthetic per-journey touches, which carry a
            # non-NULL attribution_journey_id.
            "attribution_journey_id",
            F.lit(None).cast("string"),
        )
    )
    return out.select(*column_names("fact_marketing"))

# --- retail_setup/generation/attribution.py ---
"""Deterministic marketing-attribution fact (IMP-007), Spark-native.

Builds ``ag.fact_marketing_attribution``: one row per in-store SALE receipt or
online order, including attributed, unattributed, declined, and
reconciliation-failed outcomes, using deterministic last-touch attribution
within an inclusive 7-day window.

Pipeline:

1. Purchase candidates are built from SALE receipts and online order headers,
   each left-joined to its own payment row. A purchase is "reconciled" only
   when it has an APPROVED payment whose ``amount_cents`` exactly equals the
   purchase's ``total_cents``; anything else is ``PAYMENT_DECLINED`` (a
   payment exists but is DECLINED) or ``RECONCILIATION_FAILED`` (no payment
   row at all — e.g. a cancelled online order — or an approved payment that
   doesn't reconcile to the total).
2. A deterministic, seed-bound ~5% share of the reconciled purchases is
   selected for attribution (``seeded_draws.u`` keyed on the purchase's own
   id, so selection is independent of cluster shape and stable for a given
   (config, seed)). Each selected purchase gets a unique
   ``attribution_journey_id`` and two synthetic ad-impression touches for the
   SAME customer/campaign/journey: an "older" touch (3-7 days before
   purchase) and a "newer" touch (1 minute-2 days before purchase). Both land
   inside the 7-day attribution window, and the two offset ranges never
   overlap, so the newer touch is always the more recent of the two.
3. The two touches are unioned into ``fact_marketing`` (customer_ad_id/
   customer_id populated, ``attribution_journey_id`` set). Last-touch
   attribution is then DERIVED — not copied — via ``_last_touch``: a real
   window/rank join keyed on ``touch_ts <= purchase_ts <= touch_ts + 7d``,
   ordered by ``touch_ts`` desc then ``impression_id_ext`` desc. Because the
   "newer" touch's timestamp is always later than the "older" touch's (by
   construction — the offset ranges never overlap), the newer touch always
   wins the rank; the winner is computed, not assumed, so a swapped
   offset/ordering bug would surface as a failing rank/tie-break test rather
   than silently passing.
4. All other reconciled purchases get ``UNATTRIBUTED_NO_JOURNEY`` (no journey
   was selected for them); ``attributed_revenue_cents`` is 0 for every
   non-ATTRIBUTED row and equals ``net_subtotal_cents`` (tax-exclusive) for
   ATTRIBUTED rows.
5. Selected receipts/orders/payments/promotions are enriched in place with
   the *derived* ``attribution_journey_id``/``campaign_id``/
   ``impression_id_ext`` — every other row keeps the NULL placeholders the
   base generators already emit.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


ATTRIBUTION_MODEL = "LAST_TOUCH_7D"
ATTRIBUTION_WINDOW_DAYS = 7
SELECTION_RATE = 0.05  # deterministic bounded share (~5%) of eligible purchases

# Linked journeys reuse the product-launch and loyalty campaign families from
# marketing.py. Their 14/30-day durations provide real background spend while
# still allowing a conversion up to seven days after the selected touch.
JOURNEY_CAMPAIGNS = [
    (2, ARCHETYPES[1][1], ARCHETYPES[1][3]),
    (3, ARCHETYPES[2][1], ARCHETYPES[2][3]),
]

# Touch offsets (seconds before purchase_ts). The ranges never overlap
# (older_min = 3d > newer_max = 2d), so the "newer" touch is always the more
# recent one of the pair — attribution is still DERIVED by real ranking (see
# _last_touch); this just keeps the pipeline's own construction sane.
OLDER_OFFSET_MIN_S = 3 * 86400
OLDER_OFFSET_MAX_S = ATTRIBUTION_WINDOW_DAYS * 86400
NEWER_OFFSET_MIN_S = 60
NEWER_OFFSET_MAX_S = 2 * 86400


def _last_touch(touches: DataFrame, purchases: DataFrame) -> DataFrame:
    """Deterministic last-touch join: one output row per input purchase row.

    ``touches`` columns: match_key, touch_ts, impression_id_ext, campaign_id,
    creative_id, channel, customer_ad_id, customer_id.
    ``purchases`` columns: match_key, purchase_ts (any other columns are
    passed through unchanged on the output).

    For each purchase, selects the touch with
    ``touch_ts <= purchase_ts <= touch_ts + ATTRIBUTION_WINDOW_DAYS days``,
    ranked by the greatest ``touch_ts``, ties broken by ``impression_id_ext``
    descending — a real window/rank computation, not a lookup of a
    pre-known "right answer". Purchases with no qualifying touch keep all
    touch columns NULL (left join).
    """
    window_s = ATTRIBUTION_WINDOW_DAYS * 86400
    t = touches.select(
        F.col("match_key").alias("_t_match_key"),
        F.col("touch_ts"),
        F.col("impression_id_ext"),
        F.col("campaign_id"),
        F.col("creative_id"),
        F.col("channel"),
        F.col("customer_ad_id"),
        F.col("customer_id").alias("touch_customer_id"),
    )
    p_cols = purchases.columns
    cond = (
        (F.col("_t_match_key") == F.col("match_key"))
        & (F.col("touch_ts").cast("long") <= F.col("purchase_ts").cast("long"))
        & (F.col("touch_ts").cast("long") >= F.col("purchase_ts").cast("long") - F.lit(window_s))
    )
    joined = purchases.join(t, cond, "left")
    w = Window.partitionBy("match_key").orderBy(
        F.col("touch_ts").desc_nulls_last(),
        F.col("impression_id_ext").desc_nulls_last(),
    )
    ranked = (
        joined.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_t_match_key")
        .withColumn(
            "lag_seconds",
            F.when(
                F.col("touch_ts").isNotNull(),
                F.col("purchase_ts").cast("long") - F.col("touch_ts").cast("long"),
            ).cast("long"),
        )
    )
    return ranked.select(
        *p_cols,
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "touch_customer_id",
        "lag_seconds",
    )


def _journey_touch_rows(
    selected: DataFrame,
    d: seeded_draws,
    ts_col: str,
    suffix: str,
) -> DataFrame:
    """Build one fact_marketing row per selected journey for a single touch
    ("older" or "newer"). Device/cost draws mirror marketing.py's organic
    impressions, keyed on this touch's own impression id so the two touches
    of a journey draw independently of each other.
    """
    rows = selected.withColumn("touch_ts", F.col(ts_col)).withColumn(
        "impression_id_ext",
        F.concat(F.lit("IMPJRN"), F.col("_jhash"), F.lit(suffix[0].upper())),
    )
    key = [F.col("impression_id_ext")]
    rows = rows.withColumn(
        "device", d.pick_by_weights(key, f"attr_device_{suffix}", DEVICE_WEIGHTS)
    )
    mult = None
    for device, m in DEVICE_MULTIPLIERS.items():
        cond = F.col("device") == device
        mult = F.when(cond, m) if mult is None else mult.when(cond, m)
    rows = rows.withColumn("device_mult", mult)
    lo, hi = None, None
    for channel, (c_lo, c_hi) in CHANNEL_COSTS.items():
        cond = F.col("channel") == channel
        lo = F.when(cond, c_lo) if lo is None else lo.when(cond, c_lo)
        hi = F.when(cond, c_hi) if hi is None else hi.when(cond, c_hi)
    cost_dollars = (lo + d.u(key, f"attr_cost_{suffix}") * (hi - lo)) * F.col("device_mult")
    rows = (
        rows.withColumn("cost_cents", F.round(cost_dollars * 100).cast("long"))
        .withColumn("cost", F.format_string("%.2f", cost_dollars))
        .withColumn("trace_id", F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")))
        .withColumn("creative_id", F.concat(F.col("creative_id"), F.lit(f"-{suffix.upper()}")))
        .withColumn("event_date", F.col("touch_ts").cast("date"))
    )
    out = rows.select(
        F.col("touch_ts").alias("event_ts"),
        "trace_id",
        "channel",
        "campaign_id",
        "creative_id",
        "customer_ad_id",
        F.col("customer_id").cast("double").alias("customer_id"),
        "cost_cents",
        F.col("customer_id").cast("double").alias("CustomerId"),
        F.col("cost_cents").alias("CostCents"),
        "impression_id_ext",
        "cost",
        "device",
        "event_date",
        "attribution_journey_id",
    ).withColumn("__index_level_0__", legacy_index("impression_id_ext"))
    return out.select(*column_names("fact_marketing"))


def generate_attribution(
    spark: SparkSession,
    t: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Derive fact_marketing_attribution and enrich receipts/orders/payments/
    promotions/marketing with the selected attribution mapping.

    Must run after fact_receipts, fact_receipt_lines, fact_online_order_headers,
    fact_payments, fact_promotions and the background fact_marketing all exist
    in ``t`` — wired in engine.py right after marketing.generate_marketing.
    """
    d = seeded_draws(cfg.seed)

    # --- 1. purchase candidates: SALE receipts + online orders, each
    # left-joined to its own payment.
    receipts = t["fact_receipts"].filter(F.col("receipt_type") == "SALE")
    payments = t["fact_payments"]
    store_pay = payments.filter(F.col("receipt_id_ext").isNotNull()).select(
        F.col("receipt_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )
    online_pay = payments.filter(F.col("order_id_ext").isNotNull()).select(
        F.col("order_id_ext"),
        F.col("status").alias("payment_status"),
        F.col("amount_cents").alias("payment_cents"),
    )

    store_purchases = receipts.join(store_pay, "receipt_id_ext", "left").select(
        F.col("receipt_id_ext").alias("match_key"),
        F.lit("STORE").alias("purchase_type"),
        F.col("receipt_id_ext"),
        F.lit(None).cast("string").alias("order_id_ext"),
        F.col("event_ts").alias("purchase_ts"),
        F.col("event_date"),
        F.col("store_id"),
        F.col("customer_id"),
        F.col("gross_subtotal_cents"),
        F.col("discount_cents"),
        F.col("subtotal_cents").alias("net_subtotal_cents"),
        F.col("tax_cents"),
        F.col("total_cents"),
        F.col("payment_status"),
        F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
    )

    online_purchases = (
        t["fact_online_order_headers"]
        .join(online_pay, "order_id_ext", "left")
        .select(
            F.col("order_id_ext").alias("match_key"),
            F.lit("ONLINE").alias("purchase_type"),
            F.lit(None).cast("string").alias("receipt_id_ext"),
            F.col("order_id_ext"),
            F.col("event_ts").alias("purchase_ts"),
            F.col("event_date"),
            F.lit(None).cast("long").alias("store_id"),
            F.col("customer_id"),
            F.col("gross_subtotal_cents"),
            F.col("discount_cents"),
            F.col("subtotal_cents").alias("net_subtotal_cents"),
            F.col("tax_cents"),
            F.col("total_cents"),
            F.col("payment_status"),
            F.coalesce(F.col("payment_cents"), F.lit(0).cast("long")).alias("payment_cents"),
        )
    )

    purchases_all = store_purchases.unionByName(online_purchases)

    # --- 2. reconciliation status: an APPROVED payment whose amount exactly
    # equals the purchase total is the only path to attribution eligibility.
    reconciled = (F.col("payment_status") == "APPROVED") & (
        F.col("payment_cents") == F.col("total_cents")
    )
    purchases_all = purchases_all.withColumn(
        "base_status",
        F.when(F.col("payment_status").isNull(), F.lit("RECONCILIATION_FAILED"))
        .when(F.col("payment_status") == "DECLINED", F.lit("PAYMENT_DECLINED"))
        .when(~reconciled, F.lit("RECONCILIATION_FAILED"))
        .otherwise(F.lit("ELIGIBLE")),
    )

    # --- 3. deterministic ~5% selection of eligible (reconciled) purchases,
    # with a known customer to anchor the journey's touches to.
    earliest_purchase_ts = F.date_add(F.lit(cfg.start_date), ATTRIBUTION_WINDOW_DAYS).cast(
        "timestamp"
    )
    eligible = purchases_all.filter(
        (F.col("base_status") == "ELIGIBLE")
        & F.col("customer_id").isNotNull()
        # Keep generated touches inside the requested historical range.
        & (F.col("purchase_ts") >= earliest_purchase_ts)
    )
    selected = (
        eligible.withColumn("_sel_u", d.u(["match_key"], "attr_select"))
        .filter(F.col("_sel_u") < F.lit(SELECTION_RATE))
        .drop("_sel_u")
    )

    selected = (
        selected.withColumn(
            "_jhash", F.hex(F.xxhash64(F.col("match_key"), F.lit(f"attr_journey|{cfg.seed}")))
        )
        .withColumn("attribution_journey_id", F.concat(F.lit("JRN"), F.col("_jhash")))
        .withColumn("creative_id", F.concat(F.lit("CREATJRN"), F.substring(F.col("_jhash"), 1, 12)))
        .withColumn(
            "_campaign_choice",
            (d.h64(["match_key"], "attr_campaign") % len(JOURNEY_CAMPAIGNS)).cast("int"),
        )
        .withColumn(
            "_campaign_index",
            F.when(F.col("_campaign_choice") == 0, F.lit(JOURNEY_CAMPAIGNS[0][0])).otherwise(
                F.lit(JOURNEY_CAMPAIGNS[1][0])
            ),
        )
        .withColumn(
            "_campaign_duration",
            F.when(F.col("_campaign_choice") == 0, F.lit(JOURNEY_CAMPAIGNS[0][2])).otherwise(
                F.lit(JOURNEY_CAMPAIGNS[1][2])
            ),
        )
        .withColumn(
            "channel",
            F.when(
                F.col("_campaign_choice") == 0,
                d.pick_by_weights(
                    ["match_key"],
                    "attr_channel_product",
                    [(channel, 1.0) for channel in JOURNEY_CAMPAIGNS[0][1]],
                ),
            ).otherwise(
                d.pick_by_weights(
                    ["match_key"],
                    "attr_channel_loyalty",
                    [(channel, 1.0) for channel in JOURNEY_CAMPAIGNS[1][1]],
                )
            ),
        )
        .withColumn(
            "older_offset",
            (
                F.lit(OLDER_OFFSET_MIN_S)
                + d.u(["match_key"], "attr_older") * F.lit(OLDER_OFFSET_MAX_S - OLDER_OFFSET_MIN_S)
            ).cast("long"),
        )
        .withColumn(
            "newer_offset",
            (
                F.lit(NEWER_OFFSET_MIN_S)
                + d.u(["match_key"], "attr_newer") * F.lit(NEWER_OFFSET_MAX_S - NEWER_OFFSET_MIN_S)
            ).cast("long"),
        )
        .withColumn(
            "touch_ts_older",
            F.timestamp_seconds(F.col("purchase_ts").cast("long") - F.col("older_offset")),
        )
        .withColumn(
            "touch_ts_newer",
            F.timestamp_seconds(F.col("purchase_ts").cast("long") - F.col("newer_offset")),
        )
        .withColumn("_campaign_touch_day", F.to_date("touch_ts_newer"))
        .withColumn(
            "_campaign_start",
            F.date_add(
                F.lit(cfg.start_date),
                (
                    F.datediff(F.col("_campaign_touch_day"), F.lit(cfg.start_date))
                    / F.col("_campaign_duration")
                ).cast("int")
                * F.col("_campaign_duration"),
            ),
        )
        .withColumn(
            "campaign_id",
            F.concat(
                F.lit("CAMP"),
                F.date_format("_campaign_start", "yyyyMMdd"),
                fixed_width_decimal(F.col("_campaign_index"), 2, "campaign archetype index"),
            ),
        )
    )

    cust_ad = t["dim_customers"].select(
        F.col("ID").alias("customer_id"), F.col("AdId").alias("customer_ad_id")
    )
    selected = selected.join(cust_ad, "customer_id", "left")

    # --- 4. the two linked touches per journey, unioned into fact_marketing.
    older_rows = _journey_touch_rows(selected, d, "touch_ts_older", "older")
    newer_rows = _journey_touch_rows(selected, d, "touch_ts_newer", "newer")
    fact_marketing = t["fact_marketing"].unionByName(older_rows).unionByName(newer_rows)

    # --- 5. derive last-touch attribution via a real window/rank join,
    # scoped to each journey's own two touches (never mixed with organic
    # background impressions or other journeys — deterministic regardless of
    # catalog size/config) but still a genuine ranking computation.
    touches = older_rows.select(
        F.col("attribution_journey_id").alias("match_key"),
        F.col("event_ts").alias("touch_ts"),
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "customer_id",
    ).unionByName(
        newer_rows.select(
            F.col("attribution_journey_id").alias("match_key"),
            F.col("event_ts").alias("touch_ts"),
            "impression_id_ext",
            "campaign_id",
            "creative_id",
            "channel",
            "customer_ad_id",
            "customer_id",
        )
    )
    purchase_for_rank = selected.select(
        F.col("attribution_journey_id").alias("match_key"), "purchase_ts"
    )
    best_touch = _last_touch(touches, purchase_for_rank).select(
        F.col("match_key").alias("attribution_journey_id"),
        "touch_ts",
        "impression_id_ext",
        "campaign_id",
        "creative_id",
        "channel",
        "customer_ad_id",
        "lag_seconds",
    )

    attributed = (
        selected.drop(
            "channel",
            "campaign_id",
            "creative_id",
            "customer_ad_id",
            "_jhash",
            "older_offset",
            "newer_offset",
            "touch_ts_older",
            "touch_ts_newer",
            "_campaign_choice",
            "_campaign_index",
            "_campaign_duration",
            "_campaign_touch_day",
            "_campaign_start",
        )
        .join(best_touch, "attribution_journey_id")
        .withColumn("attribution_status", F.lit("ATTRIBUTED"))
    )

    non_attributed = (
        purchases_all.join(selected.select("match_key"), "match_key", "left_anti")
        .withColumn(
            "attribution_status",
            F.when(F.col("base_status") == "ELIGIBLE", F.lit("UNATTRIBUTED_NO_JOURNEY")).otherwise(
                F.col("base_status")
            ),
        )
        .withColumn("attribution_journey_id", F.lit(None).cast("string"))
        .withColumn("touch_ts", F.lit(None).cast("timestamp"))
        .withColumn("impression_id_ext", F.lit(None).cast("string"))
        .withColumn("campaign_id", F.lit(None).cast("string"))
        .withColumn("creative_id", F.lit(None).cast("string"))
        .withColumn("channel", F.lit(None).cast("string"))
        .withColumn("customer_ad_id", F.lit(None).cast("string"))
        .withColumn("lag_seconds", F.lit(None).cast("long"))
    )

    combined = attributed.unionByName(non_attributed)
    fact_marketing_attribution = (
        combined.withColumn(
            "attribution_id",
            F.concat(
                F.lit("ATTR-"),
                F.col("purchase_type"),
                F.lit("-"),
                F.col("match_key"),
            ),
        )
        .withColumn("attribution_model", F.lit(ATTRIBUTION_MODEL))
        .withColumn("attribution_window_days", F.lit(ATTRIBUTION_WINDOW_DAYS).cast("int"))
        .withColumn(
            "attributed_revenue_cents",
            F.when(
                F.col("attribution_status") == "ATTRIBUTED", F.col("net_subtotal_cents")
            ).otherwise(F.lit(0).cast("long")),
        )
        .select(*column_names("fact_marketing_attribution"))
    )

    # --- 6. enrich receipts/orders/payments/promotions with the DERIVED
    # (not pre-assigned) journey/campaign/impression for attributed purchases.
    enrich_map = attributed.select(
        "match_key", "purchase_type", "attribution_journey_id", "campaign_id", "impression_id_ext"
    )
    store_enrich = enrich_map.filter(F.col("purchase_type") == "STORE").select(
        F.col("match_key").alias("receipt_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    online_enrich = enrich_map.filter(F.col("purchase_type") == "ONLINE").select(
        F.col("match_key").alias("order_id_ext"),
        "attribution_journey_id",
        "campaign_id",
        "impression_id_ext",
    )
    pay_enrich = enrich_map.select(F.col("match_key").alias("pay_key"), "attribution_journey_id")

    fact_receipts = (
        t["fact_receipts"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(store_enrich, "receipt_id_ext", "left")
        .select(*column_names("fact_receipts"))
    )
    fact_online_order_headers = (
        t["fact_online_order_headers"]
        .drop("attribution_journey_id", "campaign_id", "impression_id_ext")
        .join(online_enrich, "order_id_ext", "left")
        .select(*column_names("fact_online_order_headers"))
    )
    fact_payments = (
        t["fact_payments"]
        .drop("attribution_journey_id")
        .withColumn("pay_key", F.coalesce(F.col("receipt_id_ext"), F.col("order_id_ext")))
        .join(pay_enrich, "pay_key", "left")
        .drop("pay_key")
        .select(*column_names("fact_payments"))
    )
    fact_promotions = (
        t["fact_promotions"]
        .drop("attribution_journey_id")
        .join(
            store_enrich.select("receipt_id_ext", "attribution_journey_id"),
            "receipt_id_ext",
            "left",
        )
        .select(*column_names("fact_promotions"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_online_order_headers": fact_online_order_headers,
        "fact_payments": fact_payments,
        "fact_promotions": fact_promotions,
        "fact_marketing": fact_marketing,
        "fact_marketing_attribution": fact_marketing_attribution,
    }

# --- retail_setup/generation/sensors.py ---
"""BLE pings + customer zone changes, Spark-native.

Derives from SALE receipts (one BLE visit per receipt):
- 30% "known" visits: customer_ble_id from dim_customers.BLEId joined on the
  receipt's customer_id; customer_id (double) carried through.
- 70% anonymous: customer_ble_id = 'ANON-<store_id>-<h64 % 900000 + 100000>';
  customer_id NULL.

Per visit, 2-5 zones are selected without replacement by sorting the five zones
on a per-(receipt, zone) draw in place. Each zone gets 2-5 pings. Ping event_ts
is based on the receipt timestamp plus a deterministic visit offset, then
bounded to that store-day's open interval and the requested history. All
per-visit sequencing is row-local array work, so the billion-row ping stream is
never shuffled.

Deviation from spec's "pandas-UDF island" suggestion: uses Spark-native array
functions instead. The correlation between zones is mild enough for this
approach and avoids applyInPandas/F.rand() (see self-review checklist).

Schema notes (TMDL arbiter results):
- fact_ble_pings: BOTH customer_id (double, snake) AND CustomerId (double,
  TMDL-bound) with mirrored values; __index_level_0__ (long) = hash-derived
  via legacy_index(trace_id).
- fact_customer_zone_changes: BOTH snake_case (store_id, customer_ble_id,
  from_zone, to_zone) AND PascalCase (StoreID, CustomerBLEId, FromZone, ToZone)
  with mirrored values; __index_level_0__ = legacy_index(trace_id).
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# BLE zone names (5 total); zone selection is without replacement
BLE_ZONES = ["ENTRANCE", "ELECTRONICS", "GROCERY", "CLOTHING", "CHECKOUT"]
_N_ZONES = len(BLE_ZONES)  # 5


def generate_ble(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> tuple[DataFrame, DataFrame]:
    """Generate BLE ping and zone-change facts from SALE receipts.

    Returns:
        (pings_df, zone_changes_df)
    """
    d = seeded_draws(cfg.seed)

    # --- filter to SALE receipts (defensively; receipts may already be SALE-only)
    visits = receipts.filter(
        (F.col("receipt_type") == "SALE")
        & F.col("event_date").between(F.lit(cfg.start_date), F.lit(cfg.end_date))
    )

    store_hours = dims["dim_stores"].select(
        F.col("ID").alias("store_id"),
        "operating_hours",
    )
    visits = visits.join(store_hours, "store_id")
    is_24h = F.col("operating_hours") == "24h"
    hour_parts = F.split(F.col("operating_hours"), "-")
    open_hour = F.when(is_24h, 0).otherwise(hour_parts.getItem(0).cast("int"))
    close_hour = F.when(is_24h, 24).otherwise(hour_parts.getItem(1).cast("int"))
    day_epoch = F.unix_timestamp(F.col("event_date").cast("timestamp"))
    visits = visits.withColumn(
        "_open_epoch",
        day_epoch + F.coalesce(open_hour, F.lit(6)) * F.lit(3600),
    ).withColumn(
        "_close_epoch",
        day_epoch + F.coalesce(close_hour, F.lit(22)) * F.lit(3600),
    )

    # --- 30% known (u < 0.3): join dim_customers for BLEId; 70% anonymous
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_id_long"),
        F.col("BLEId").alias("ble_id_from_dim"),
    )

    # Receipt customer_id may be stored as a non-double type; cast for join
    visits = visits.withColumn("_cust_key", F.col("customer_id").cast("long"))
    visits = visits.join(
        customers,
        visits._cust_key == customers.customer_id_long,
        "left",
    )

    # known flag: u(receipt_id_ext, "known") < 0.3
    visits = visits.withColumn("_known", d.u(["receipt_id_ext"], "known") < F.lit(0.3))

    # customer_ble_id and customer_id (double)
    visits = visits.withColumn(
        "customer_ble_id",
        F.when(
            F.col("_known") & F.col("ble_id_from_dim").isNotNull(),
            F.col("ble_id_from_dim"),
        ).otherwise(
            F.concat(
                F.lit("ANON-"),
                F.col("store_id").cast("string"),
                F.lit("-"),
                (d.h64(["receipt_id_ext"], "anon_id") % F.lit(900000) + F.lit(100000)).cast(
                    "string"
                ),
            )
        ),
    ).withColumn(
        "visit_customer_id",
        F.when(
            F.col("_known") & F.col("ble_id_from_dim").isNotNull(),
            F.col("customer_id").cast("double"),
        ).cast("double"),
    )

    # --- draw n_zones per visit (2-5 inclusive)
    # u_zones in [0,1) → floor(u * 4) + 2 = 2..5
    visits = visits.withColumn(
        "n_zones",
        (F.floor(d.u(["receipt_id_ext"], "nz") * F.lit(4)) + F.lit(2)).cast("int"),
    )

    # --- zone selection without replacement: rank the visit's five zones by a
    # per-(receipt, zone) draw. Sorting the small zone array in place (draw, then
    # zone position as the tie-break) replaces a per-receipt window shuffle.
    zone_entries = F.array(
        *[
            F.struct(
                d.u([F.col("receipt_id_ext"), F.lit(str(pos))], "z_shuffle").alias("zone_draw"),
                F.lit(pos).alias("zone_pos"),
                F.lit(zone).alias("zone"),
            )
            for pos, zone in enumerate(BLE_ZONES)
        ]
    )
    selected = F.slice(F.array_sort(zone_entries), 1, F.col("n_zones"))

    # --- draw n_pings per selected zone (2-5 inclusive), keyed by zone rank
    ping_counts = F.transform(
        selected,
        lambda _zone, index: (
            F.floor(
                d.u([F.col("receipt_id_ext"), (index + F.lit(1)).cast("string")], "np") * F.lit(4)
            )
            + F.lit(2)
        ).cast("int"),
    )

    zone_index = F.col("_zone_index")
    selected_zones = (
        visits.select(
            "receipt_id_ext",
            "store_id",
            "event_ts",
            "event_date",
            "customer_ble_id",
            "visit_customer_id",
            "_open_epoch",
            "_close_epoch",
            selected.alias("_zones"),
            ping_counts.alias("_ping_counts"),
        )
        .select(
            "*",
            F.posexplode("_zones").alias("_zone_index", "_zone"),
        )
        .select(
            "receipt_id_ext",
            "store_id",
            "event_ts",
            "event_date",
            "customer_ble_id",
            "visit_customer_id",
            "_open_epoch",
            "_close_epoch",
            (zone_index + F.lit(1)).alias("zone_rank"),
            F.col("_zone.zone").alias("zone"),
            F.element_at("_ping_counts", zone_index + F.lit(1)).alias("n_pings"),
            # Pings before this zone in visit order: the visit's ping sequence offset.
            F.aggregate(
                F.slice("_ping_counts", 1, zone_index),
                F.lit(0),
                lambda total, n: total + n,
            ).alias("_prior_pings"),
            F.when(
                zone_index > 0,
                F.element_at("_zones", zone_index).getField("zone"),
            ).alias("_prev_zone"),
        )
    )

    # --- explode to individual pings; the visit-wide sequence is (zone_rank,
    # ping_seq) order, i.e. the pings of earlier zones plus this zone's ping_seq
    pings = selected_zones.select(
        "*",
        F.explode(F.sequence(F.lit(1), F.col("n_pings"))).alias("ping_seq"),
    ).withColumn(
        "ping_visit_seq",
        (F.col("_prior_pings") + F.col("ping_seq")).cast("long"),
    )

    ping_key = [
        F.col("receipt_id_ext"),
        F.col("zone_rank").cast("string"),
        F.col("ping_seq").cast("string"),
    ]

    # offset_min = (zone_rank - 1) * 7 + ping_seq + u - 15
    offset_min = (
        (F.col("zone_rank") - F.lit(1)) * F.lit(7)
        + F.col("ping_seq")
        + d.u(ping_key, "ts_offset")
        - F.lit(15.0)
    )
    raw_ping_epoch = F.unix_timestamp("event_ts") + (offset_min * F.lit(60.0)).cast("long")
    history_start = F.unix_timestamp(
        F.to_timestamp(F.lit(f"{cfg.start_date.isoformat()} 00:00:00"))
    )
    history_end = F.unix_timestamp(F.to_timestamp(F.lit(f"{cfg.end_date.isoformat()} 23:59:59")))
    ping_epoch = F.greatest(
        F.col("_open_epoch"),
        history_start,
        F.least(
            raw_ping_epoch,
            F.col("_close_epoch") - F.lit(1),
            history_end,
        ),
    )
    ping_ts = F.timestamp_seconds(ping_epoch)

    # rssi uniform [-80, -29] (integer long)
    rssi = F.floor(d.u(ping_key, "rssi") * F.lit(52.0)).cast("long") - F.lit(80)

    pings = pings.withColumn("ping_ts", ping_ts).withColumn("rssi", rssi)
    pings = pings.withColumn(
        "trace_id",
        F.concat(
            F.lit("TRC-BLE-"),
            F.col("receipt_id_ext"),
            F.lit("-"),
            F.col("ping_visit_seq").cast("string"),
        ),
    ).withColumn(
        "beacon_id",
        F.format_string("BEACON_%03d_%s", F.col("store_id"), F.col("zone")),
    )

    # __index_level_0__ = hash-derived via legacy_index(trace_id)
    pings_out = pings.select(
        F.col("ping_ts").alias("event_ts"),
        "trace_id",
        F.col("store_id").cast("long").alias("store_id"),
        "beacon_id",
        "customer_ble_id",
        F.col("visit_customer_id").alias("customer_id"),
        # TMDL-bound PascalCase mirror
        F.col("visit_customer_id").alias("CustomerId"),
        F.col("rssi").cast("long").alias("rssi"),
        "zone",
        F.to_date("ping_ts").alias("event_date"),
    ).withColumn(
        "__index_level_0__",
        legacy_index("trace_id"),
    )
    pings_df = pings_out.select(*column_names("fact_ble_pings"))

    # --- zone changes. Within a visit, offset_min strictly increases in
    # (zone_rank, ping_seq) order and the clamps above are monotone, so ping_ts
    # never decreases along the visit sequence; zones are distinct per visit.
    # The previous ping (by ping_ts, then sequence) therefore changes zone
    # exactly at the first ping of every zone after the first, and the change
    # sequence equals zone_rank - 1 — no per-visit window shuffle needed.
    zc_base = pings.filter((F.col("zone_rank") > 1) & (F.col("ping_seq") == 1)).select(
        "store_id",
        "customer_ble_id",
        "receipt_id_ext",
        "ping_ts",
        "zone",
        F.col("_prev_zone").alias("prev_zone"),
        (F.col("zone_rank") - F.lit(1)).cast("long").alias("zc_seq"),
    )

    zc_out = zc_base.select(
        F.col("ping_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-ZC-"),
            F.col("receipt_id_ext"),
            F.lit("-"),
            F.col("zc_seq").cast("string"),
        ).alias("trace_id"),
        # snake_case (extras)
        F.col("store_id").cast("long").alias("store_id"),
        F.col("customer_ble_id").alias("customer_ble_id"),
        F.col("prev_zone").alias("from_zone"),
        F.col("zone").alias("to_zone"),
        # TMDL-bound PascalCase mirrors
        F.col("store_id").cast("long").alias("StoreID"),
        F.col("customer_ble_id").alias("CustomerBLEId"),
        F.col("prev_zone").alias("FromZone"),
        F.col("zone").alias("ToZone"),
        F.to_date("ping_ts").alias("event_date"),
    ).withColumn(
        "__index_level_0__",
        legacy_index("trace_id"),
    )
    zc_df = zc_out.select(*column_names("fact_customer_zone_changes"))

    return pings_df, zc_df

# --- retail_setup/generation/inventory_balances.py ---
"""Balance + stockout helpers for the inventory chain (Plan 2b Task 9).

Split out of ``inventory.py`` per the plan's ~400-line guidance. Covers
stages 7-8: day-0 INITIAL seed txns plus the running-balance window, and the
balance-crossing stockout extraction. Shared draw/column primitives used by
both modules live here to keep the import direction one-way
(``inventory`` -> ``inventory_balances``).
"""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Canonical column layout for raw (pre-balance) inventory txn streams.
TXN_COLS = [
    "node_id",
    "product_id",
    "quantity",
    "txn_type",
    "source",
    "event_ts",
    "event_date",
    "trace_id",
]


def draw_int(u: Column, lo: int, hi: int) -> Column:
    """Uniform integer draw in [lo, hi] from a [0,1) uniform column."""
    return (F.lit(lo) + F.floor(u * F.lit(hi - lo + 1))).cast("long")


# ---------------------------------------------------------------------------
# Stage 7: initial-stock seeds + running balances
# ---------------------------------------------------------------------------


def with_balances(
    txns: DataFrame,
    lo: int,
    hi: int,
    tag: str,
    d: seeded_draws,
    cfg: GenerationConfig,
    *,
    keys: DataFrame | None = None,
) -> DataFrame:
    """Fold a day-0 INITIAL seed txn per (node, product) into the stream and
    compute the running balance ordered by (event_ts, trace_id).

    Opening stock is a direct seeded draw in ``[lo, hi]``. It deliberately
    ignores later transactions: extending the requested history must not
    rewrite day-one stock by backsolving from future demand."""
    keys = txns.select("node_id", "product_id").distinct() if keys is None else keys
    seeds = (
        keys.withColumn(
            "quantity",
            draw_int(
                d.u(["node_id", "product_id"], f"seed-stock-{tag}"),
                lo,
                hi,
            ),
        )
        .withColumn("txn_type", F.lit("INITIAL"))
        .withColumn("source", F.lit("SEED"))
        # String-built timestamp (session-TZ semantics) to match every
        # other event_ts in the chain; a naive Python datetime literal
        # would shift with the driver's local timezone.
        .withColumn("event_ts", F.to_timestamp(F.lit(f"{cfg.start_date.isoformat()} 00:00:00")))
        .withColumn("event_date", F.lit(cfg.start_date).cast("date"))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit(f"TRC-INIT-{tag}-"),
                F.col("node_id").cast("string"),
                F.lit("-"),
                F.col("product_id").cast("string"),
            ),
        )
        .select(*TXN_COLS)
    )
    run_w = (
        Window.partitionBy("node_id", "product_id")
        .orderBy("event_ts", "trace_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    return txns.unionByName(seeds).withColumn("balance", F.sum("quantity").over(run_w).cast("long"))


# ---------------------------------------------------------------------------
# Stage 8: stockouts
# ---------------------------------------------------------------------------


def stockouts(balanced: DataFrame, tag: str, node_as: str) -> DataFrame:
    """Balance crossings to <=0 (previous balance > 0); deduped to one per
    (node, product, day). ``node_as`` is 'StoreID' or 'DCID' — the other
    contract column stays NULL (double, per the TMDL contract)."""
    order_w = Window.partitionBy("node_id", "product_id").orderBy("event_ts", "trace_id")
    day_w = Window.partitionBy("node_id", "product_id", "event_date").orderBy(
        "event_ts", "trace_id"
    )
    other = "DCID" if node_as == "StoreID" else "StoreID"
    return (
        balanced.withColumn("_prev", F.lag("balance").over(order_w))
        .filter((F.col("balance") <= 0) & (F.col("_prev") > 0))
        .withColumn("_dup", F.row_number().over(day_w))
        .filter(F.col("_dup") == 1)
        .select(
            "event_ts",
            F.concat(
                F.lit(f"TRC-SO-{tag}-"),
                F.col("node_id").cast("string"),
                F.lit("-"),
                F.col("product_id").cast("string"),
                F.lit("-"),
                F.date_format("event_date", "yyyyMMdd"),
            ).alias("trace_id"),
            F.col("node_id").cast("double").alias(node_as),
            F.lit(None).cast("double").alias(other),
            F.col("product_id").alias("ProductID"),
            F.abs("quantity").cast("long").alias("LastKnownQuantity"),
            "event_date",
        )
    )

# --- retail_setup/generation/inventory.py ---
"""Inventory/logistics ledgers derived from the causal fulfillment plan.

Store stock, reorders and delivery times come from inventory_policy, never
from a second independent draw. DC supply is an explicit unlimited supplier
baseline: every outbound load is replenished one second before loading.
Truck moves retain the six-stage schedule (including future scheduled legs);
stock-changing ledgers contain only events inside the requested history.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def _with_index(df: DataFrame, table: str) -> DataFrame:
    """Add the TMDL-bound __index_level_0__ column and apply the contract order."""
    return df.withColumn("__index_level_0__", legacy_index("trace_id")).select(*column_names(table))


# ---------------------------------------------------------------------------
# Stage 2: reorders
# ---------------------------------------------------------------------------


def _store_dc_map(dims: dict[str, DataFrame]) -> DataFrame:
    """Route each store to its nearest DC by geography (same state > same region
    > lowest dc_id), restoring datagen's geography-aware assignment in place of a
    ``store_id % dc_count`` modulo. Returns one (store_id, dc_id) row per store."""
    geo = dims["dim_geographies"].select(F.col("ID").alias("geo_id"), "State", "Region")
    stores = (
        dims["dim_stores"]
        .select(F.col("ID").alias("store_id"), "GeographyID")
        .join(geo, F.col("GeographyID") == F.col("geo_id"))
        .select("store_id", F.col("State").alias("s_state"), F.col("Region").alias("s_region"))
    )
    dcs = (
        dims["dim_distribution_centers"]
        .select(F.col("ID").alias("dc_id"), "GeographyID")
        .join(geo, F.col("GeographyID") == F.col("geo_id"))
        .select("dc_id", F.col("State").alias("d_state"), F.col("Region").alias("d_region"))
    )
    scored = stores.crossJoin(dcs).withColumn(
        "_score",
        F.when(F.col("s_state") == F.col("d_state"), 0)
        .when(F.col("s_region") == F.col("d_region"), 1)
        .otherwise(2),
    )
    nearest = Window.partitionBy("store_id").orderBy("_score", "dc_id")
    return (
        scored.withColumn("_r", F.row_number().over(nearest))
        .filter(F.col("_r") == 1)
        .select("store_id", F.col("dc_id").cast("long").alias("dc_id"))
    )


def _reorders(events: DataFrame, store_dc: DataFrame) -> DataFrame:
    return (
        events.filter(F.col("kind") == "REORDER")
        .select(
            "store_id",
            "product_id",
            "event_ts",
            "event_date",
            "trace_id",
            F.col("balance").alias("current_quantity"),
            F.col("quantity").alias("reorder_quantity"),
            "reorder_point",
            "priority",
        )
        .join(store_dc, "store_id")
    )


def _split_reorders_by_capacity(
    reorders: DataFrame,
    capacity: int,
) -> DataFrame:
    """Pack reorder units into capacity-bounded legs, splitting rows as needed."""
    if capacity < 1:
        raise ValueError("capacity must be positive")
    ordering = (
        Window.partitionBy("store_id", "event_date")
        .orderBy("product_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    capacity_col = F.lit(capacity).cast("long")
    return (
        reorders.withColumn("_end_unit", F.sum("reorder_quantity").over(ordering))
        .withColumn("_start_unit", F.col("_end_unit") - F.col("reorder_quantity"))
        .withColumn(
            "_first_leg",
            F.floor(F.col("_start_unit") / capacity_col).cast("long"),
        )
        .withColumn(
            "_last_leg",
            F.floor((F.col("_end_unit") - 1) / capacity_col).cast("long"),
        )
        .withColumn(
            "leg",
            F.explode(F.sequence(F.col("_first_leg"), F.col("_last_leg"))),
        )
        .withColumn(
            "reorder_quantity",
            (
                F.least(F.col("_end_unit"), (F.col("leg") + 1) * capacity_col)
                - F.greatest(F.col("_start_unit"), F.col("leg") * capacity_col)
            ).cast("long"),
        )
        .drop("_end_unit", "_start_unit", "_first_leg", "_last_leg")
    )


# ---------------------------------------------------------------------------
# Stage 3: shipments (one per store-day with reorders) + truck assignment
# ---------------------------------------------------------------------------


def _truck_lookup(
    spark: SparkSession, dims: dict[str, DataFrame], cfg: GenerationConfig
) -> DataFrame:
    """(dc_id, slot, truck_id, n_trucks) rows for round-robin-by-day joins."""
    trucks = dims["dim_trucks"].select("ID", "DCID").collect()
    assigned: dict[int, list[int]] = {}
    pool: list[int] = []
    for r in trucks:
        if r.DCID is None:
            pool.append(int(r.ID))
        else:
            assigned.setdefault(int(r.DCID), []).append(int(r.ID))
    rows = []
    for dc in range(1, cfg.dc_count + 1):
        # Fall back to pool trucks for DCs with no assigned trucks (documented).
        fleet = sorted(assigned.get(dc) or pool)
        if not fleet:
            raise ValueError(f"DC {dc} has neither assigned nor shared trucks")
        rows.extend((dc, slot, tid, len(fleet)) for slot, tid in enumerate(fleet))
    return spark.createDataFrame(rows, "dc_id long, slot long, truck_id long, n_trucks long")


def _shipments(
    spark: SparkSession, planned: DataFrame, dims: dict[str, DataFrame], cfg: GenerationConfig
) -> DataFrame:
    """Assign trucks without redrawing the policy's delivery times or quantities."""
    base = planned.withColumn(
        "_day_num",
        F.datediff(F.col("event_date"), F.lit(cfg.start_date)),
    )
    lookup = _truck_lookup(spark, dims, cfg)
    sizes = lookup.select("dc_id", "n_trucks").distinct()
    return (
        base.join(sizes, "dc_id")
        .withColumn(
            "slot",
            F.pmod(
                F.col("_day_num") + F.col("product_id") + F.col("leg"),
                F.col("n_trucks"),
            ),
        )
        .join(lookup, ["dc_id", "slot", "n_trucks"])
        .withColumn("unload_minutes", (F.unix_timestamp("etd") - F.unix_timestamp("eta")) / 60.0)
        .drop("_day_num", "slot", "n_trucks")
    )


def _truck_moves(shipments: DataFrame) -> DataFrame:
    stages = F.array(
        *[
            F.struct(F.lit(status).alias("status"), F.col(ts_col).alias("event_ts"))
            for status, ts_col in [
                ("SCHEDULED", "scheduled_ts"),
                ("LOADING", "loading_ts"),
                ("IN_TRANSIT", "depart_ts"),
                ("ARRIVED", "eta"),
                ("UNLOADING", "unloading_ts"),
                ("COMPLETED", "etd"),
            ]
        ]
    )
    done = F.col("status") == "COMPLETED"
    return (
        shipments.withColumn("_s", F.explode(stages))
        .select(
            F.col("_s.event_ts").alias("event_ts"),
            F.col("truck_id"),
            F.col("dc_id"),
            F.col("store_id"),
            F.col("shipment_id"),
            F.col("_s.status").alias("status"),
            # eta/etd are known at scheduling -> populated on all rows;
            # departure_time/actual_unload_duration only once COMPLETED.
            F.col("eta"),
            F.col("etd"),
            F.col("unload_minutes"),
        )
        .withColumn("departure_time", F.when(done, F.col("etd")).cast("timestamp"))
        .withColumn("actual_unload_duration", F.when(done, F.col("unload_minutes")).cast("double"))
        .withColumn(
            "trace_id",
            F.concat(F.lit("TRC-TM-"), F.col("shipment_id"), F.lit("-"), F.col("status")),
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )


def _truck_inventory(shipments: DataFrame) -> DataFrame:
    actions = F.array(
        F.struct(
            F.lit("LOAD").alias("action"),
            F.col("loading_ts").alias("event_ts"),
            F.col("dc_id").alias("location_id"),
            F.lit("DC").alias("location_type"),
        ),
        F.struct(
            F.lit("UNLOAD").alias("action"),
            F.col("unloading_ts").alias("event_ts"),
            F.col("store_id").alias("location_id"),
            F.lit("STORE").alias("location_type"),
        ),
    )
    return (
        shipments.withColumn("_a", F.explode(actions))
        .select(
            F.col("_a.event_ts").alias("event_ts"),
            "truck_id",
            "shipment_id",
            "product_id",
            "quantity",
            F.col("_a.action").alias("action"),
            F.col("_a.location_id").alias("location_id"),
            F.col("_a.location_type").alias("location_type"),
            "dc_id",
            "store_id",
        )
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-TI-"),
                F.col("shipment_id"),
                F.lit("-"),
                F.col("product_id").cast("string"),
                F.lit("-"),
                F.col("action"),
            ),
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )


# ---------------------------------------------------------------------------
# Stages 5-6: DC txns + store inbound mirrors
# ---------------------------------------------------------------------------


def _dc_txns(truck_inv: DataFrame) -> DataFrame:
    # --- outbound: mirror truck LOADs (negative qty, source = shipment_id).
    outbound = truck_inv.filter(F.col("action") == "LOAD").select(
        F.col("dc_id").alias("node_id"),
        "product_id",
        (-F.col("quantity")).cast("long").alias("quantity"),
        F.lit("OUTBOUND_SHIPMENT").alias("txn_type"),
        F.col("shipment_id").alias("source"),
        "event_ts",
        "event_date",
        F.concat(
            F.lit("TRC-DC-"), F.col("shipment_id"), F.lit("-"), F.col("product_id").cast("string")
        ).alias("trace_id"),
    )
    inbound = (
        outbound.withColumn("quantity", -F.col("quantity"))
        .withColumn("txn_type", F.lit("INBOUND_SHIPMENT"))
        .withColumn("source", F.concat(F.lit("SUPPLIER-"), F.col("source")))
        .withColumn("trace_id", F.concat(F.lit("TRC-SUP-"), F.col("trace_id")))
        .withColumn("event_ts", F.col("event_ts") - F.expr("INTERVAL 1 SECOND"))
        .withColumn("event_date", F.to_date("event_ts"))
    )
    return inbound.unionByName(outbound).select(*TXN_COLS)


def _store_inbound(truck_inv: DataFrame) -> DataFrame:
    return truck_inv.filter(F.col("action") == "UNLOAD").select(
        F.col("store_id").alias("node_id"),
        "product_id",
        F.col("quantity").cast("long").alias("quantity"),
        F.lit("INBOUND_SHIPMENT").alias("txn_type"),
        F.col("shipment_id").alias("source"),
        "event_ts",
        "event_date",
        F.concat(
            F.lit("TRC-SI-"), F.col("shipment_id"), F.lit("-"), F.col("product_id").cast("string")
        ).alias("trace_id"),
    )


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------


def generate_inventory_chain(
    spark: SparkSession,
    sales: dict[str, DataFrame],
    rets: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
    *,
    fulfillment: FulfillmentResult | None = None,
) -> dict[str, DataFrame]:
    """Generate six ledgers from the plan used to fulfill ``sales``.

    Independent callers must first call ``resolve_fulfillment`` and pass its
    result here, using its sales and return header plan for dependent tables.
    Silently accepting unconstrained sales would recreate the negative-stock bug.
    """
    if fulfillment is None:
        raise ValueError("resolve_fulfillment is required before generate_inventory_chain")
    d = seeded_draws(cfg.seed)
    events = fulfillment.events
    store_dc = _store_dc_map(dims)
    reorders = _reorders(events, store_dc)
    planned = events.filter(F.col("kind") == "SHIPMENT").join(store_dc, "store_id")
    shipments = _shipments(spark, planned, dims, cfg)
    truck_moves = _truck_moves(shipments)
    truck_inv = _truck_inventory(shipments).filter(F.col("event_date") <= F.lit(cfg.end_date))
    dc_raw = _dc_txns(truck_inv)
    store_bal = (
        events.filter(~F.col("kind").isin("REORDER", "SHIPMENT"))
        .withColumnRenamed("store_id", "node_id")
        .withColumnRenamed("kind", "txn_type")
    )
    dc_keys = (
        dims["dim_distribution_centers"]
        .select(F.col("ID").alias("node_id"))
        .crossJoin(
            dims["dim_products"].select(F.col("ID").alias("product_id")),
        )
    )
    dc_bal = with_balances(dc_raw, 500, 2000, "DC", d, cfg, keys=dc_keys)

    fact_store_txn = _with_index(
        store_bal.withColumnRenamed("node_id", "store_id"), "fact_store_inventory_txn"
    )
    fact_dc_txn = _with_index(
        dc_bal.withColumnRenamed("node_id", "dc_id")
        # Rename lowercase source -> Source (TMDL-bound PascalCase). Keeping
        # both would be a case-insensitive duplicate that Delta rejects.
        .withColumnRenamed("source", "Source"),
        "fact_dc_inventory_txn",
    )

    stockouts_df = stockouts(store_bal, "ST", "StoreID").unionByName(
        stockouts(dc_bal, "DC", "DCID")
    )

    return {
        "fact_store_inventory_txn": fact_store_txn,
        "fact_dc_inventory_txn": fact_dc_txn,
        "fact_truck_moves": _with_index(truck_moves, "fact_truck_moves"),
        "fact_truck_inventory": _with_index(truck_inv, "fact_truck_inventory"),
        "fact_reorders": _with_index(reorders, "fact_reorders"),
        "fact_stockouts": _with_index(stockouts_df, "fact_stockouts"),
    }

# --- retail_setup/generation/gold.py ---
"""Gold aggregates — exact port of 02-historical-data-load.ipynb Part 3.

Operates on the in-memory ``GenerationResult.tables`` dict (``tables[...]``
replaces the legacy ``read_silver(...)``). Definitions are identical in
02-historical-data-load and 04-streaming-to-gold (verified 2026-06-12).

Deliberate decisions (vs the legacy notebooks):
- Money sums: legacy summed the formatted STRING columns (total_amount,
  subtotal_amount, tax_amount, ext_price, cost) relying on Spark's implicit
  cast — here we cast explicitly to double. Identical results, honest types.
- ``computed_at`` (top_products_15m) and ``as_of`` (both inventory positions)
  are produced exactly as the legacy code does, even though the TMDL doesn't
  bind them — extra columns are allowed.
- Legacy quirks preserved: truck_dwell_daily derives ``site`` as
  ``STORE_<id>`` / ``DC_<id>`` and filters ``dwell_min > 0``; the two
  inventory positions take the latest balance per key via a row_number
  window over ``event_ts`` and ``trace_id`` descending.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


GOLD_TABLES: list[str] = [
    "sales_minute_store",
    "top_products_15m",
    "inventory_position_current",
    "dc_inventory_position_current",
    "truck_dwell_daily",
    "online_sales_daily",
    "zone_dwell_minute",
    "marketing_cost_daily",
    "campaign_performance_daily",
    "tender_mix_daily",
]


def _money(col: str):
    """Explicit double cast for the legacy formatted string money columns."""
    return F.col(col).cast("double")


def _latest_inventory_position(
    transactions: DataFrame,
    node_column: str,
    gold_table: str,
) -> DataFrame:
    """Select the final deterministic balance for each node/product key."""
    latest = Window.partitionBy(node_column, "product_id").orderBy(
        F.desc("event_ts"),
        F.desc("trace_id"),
    )
    return (
        transactions.withColumn("rn", F.row_number().over(latest))
        .filter(F.col("rn") == 1)
        .select(
            node_column,
            "product_id",
            F.col("balance").alias("on_hand"),
            F.col("event_ts").alias("as_of"),
        )
        .select(*column_names(gold_table))
    )


def generate_campaign_performance(
    fact_marketing: DataFrame,
    fact_marketing_attribution: DataFrame,
) -> DataFrame:
    """Aggregate campaign spend and reconciled conversions by touch day."""

    campaign_spend = (
        fact_marketing.withColumn("day", F.to_date("event_ts"))
        .groupBy("campaign_id", "channel", "day")
        .agg(
            F.countDistinct("impression_id_ext").alias("impressions"),
            F.sum("cost_cents").cast("long").alias("spend_cents"),
        )
    )
    campaign_conversions = (
        fact_marketing_attribution.filter(F.col("attribution_status") == "ATTRIBUTED")
        .withColumn("day", F.to_date("touch_ts"))
        .groupBy("campaign_id", "channel", "day")
        .agg(
            F.countDistinct("attribution_id").alias("conversions"),
            F.sum("attributed_revenue_cents").cast("long").alias("attributed_revenue_cents"),
            F.sum("discount_cents").cast("long").alias("discount_cents"),
            F.sum("tax_cents").cast("long").alias("tax_cents"),
            F.sum("payment_cents").cast("long").alias("payment_cents"),
        )
    )
    return (
        campaign_spend.join(campaign_conversions, ["campaign_id", "channel", "day"], "left")
        .fillna(
            0,
            subset=[
                "conversions",
                "attributed_revenue_cents",
                "discount_cents",
                "tax_cents",
                "payment_cents",
            ],
        )
        .withColumn(
            "conversion_rate",
            F.when(F.col("impressions") == 0, F.lit(0.0)).otherwise(
                F.col("conversions") / F.col("impressions")
            ),
        )
        .withColumn(
            "roas",
            F.when(F.col("spend_cents") == 0, F.lit(0.0)).otherwise(
                F.col("attributed_revenue_cents") / F.col("spend_cents")
            ),
        )
        .select(*column_names("campaign_performance_daily"))
    )


def generate_gold(spark: SparkSession, tables: dict[str, DataFrame]) -> dict[str, DataFrame]:
    """Build the 10 Gold aggregate frames from the generated fact tables."""
    gold: dict[str, DataFrame] = {}

    # Sales by minute per store
    gold["sales_minute_store"] = (
        tables["fact_receipts"]
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "ts")
        .agg(
            F.sum(_money("total_amount")).alias("total_sales"),
            F.count("*").alias("receipts"),
            F.avg(_money("total_amount")).alias("avg_basket"),
        )
        .select(*column_names("sales_minute_store"))
    )

    # Top products by revenue (15m windows). Real-time tile: keep only the most
    # recent 15-minute window so SUM(revenue)/SUM(units) reflect the latest
    # window, not lifetime totals (the legacy build retained every window).
    _top15 = (
        tables["fact_receipt_lines"]
        .withColumn("window_15m", F.window(F.col("event_ts"), "15 minutes"))
        .groupBy("product_id", "window_15m")
        .agg(
            F.sum(_money("ext_price")).alias("revenue"),
            F.sum("quantity").alias("units"),
        )
        .withColumn("computed_at", F.col("window_15m.end"))
        .drop("window_15m")
    )
    _latest_15m = _top15.agg(F.max("computed_at").alias("m")).collect()[0]["m"]
    if _latest_15m is not None:
        _top15 = _top15.filter(F.col("computed_at") == F.lit(_latest_15m))
    gold["top_products_15m"] = _top15.select(*column_names("top_products_15m"))

    # Current store inventory position (latest balance per store/product).
    gold["inventory_position_current"] = _latest_inventory_position(
        tables["fact_store_inventory_txn"],
        "store_id",
        "inventory_position_current",
    )

    # DC inventory position (latest balance per dc/product).
    gold["dc_inventory_position_current"] = _latest_inventory_position(
        tables["fact_dc_inventory_txn"],
        "dc_id",
        "dc_inventory_position_current",
    )

    # Truck dwell time daily
    gold["truck_dwell_daily"] = (
        tables["fact_truck_moves"]
        .withColumn("day", F.to_date("event_ts"))
        .withColumn(
            "site",
            F.when(
                F.col("store_id").isNotNull(), F.concat(F.lit("STORE_"), F.col("store_id"))
            ).otherwise(F.concat(F.lit("DC_"), F.col("dc_id"))),
        )
        .withColumn(
            "dwell_min",
            (F.unix_timestamp("etd") - F.unix_timestamp("eta")) / 60,
        )
        .filter(F.col("dwell_min").isNotNull() & (F.col("dwell_min") > 0))
        .groupBy("site", "day")
        .agg(
            F.avg("dwell_min").alias("avg_dwell_min"),
            F.countDistinct("truck_id").alias("trucks"),
        )
        .select(*column_names("truck_dwell_daily"))
    )

    # Online sales daily
    gold["online_sales_daily"] = (
        tables["fact_online_order_headers"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day")
        .agg(
            F.count("*").alias("orders"),
            F.sum(_money("subtotal_amount")).alias("subtotal"),
            F.sum(_money("tax_amount")).alias("tax"),
            F.sum(_money("total_amount")).alias("total"),
            F.avg(_money("total_amount")).alias("avg_order_value"),
        )
        .select(*column_names("online_sales_daily"))
    )

    # Zone dwell per minute
    gold["zone_dwell_minute"] = (
        tables["fact_foot_traffic"]
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "zone", "ts")
        .agg(
            F.avg("dwell_seconds").alias("avg_dwell"),
            F.sum("count").alias("customers"),
        )
        .select(*column_names("zone_dwell_minute"))
    )

    # Marketing cost daily
    gold["marketing_cost_daily"] = (
        tables["fact_marketing"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("campaign_id", "day")
        .agg(
            F.count("*").alias("impressions"),
            F.sum(_money("cost")).alias("cost"),
        )
        .select(*column_names("marketing_cost_daily"))
    )

    gold["campaign_performance_daily"] = generate_campaign_performance(
        tables["fact_marketing"], tables["fact_marketing_attribution"]
    )

    # Tender mix daily
    gold["tender_mix_daily"] = (
        tables["fact_receipts"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day", "payment_method")
        .agg(
            F.count("*").alias("transactions"),
            F.sum(_money("total_amount")).alias("total_amount"),
        )
        .select(*column_names("tender_mix_daily"))
    )

    return gold

# --- retail_setup/generation/invariants.py ---
"""Cross-table invariant checks. Pure reads; raises nothing — returns a report."""

from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

from pyspark import inheritable_thread_target
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Each check is one small Spark job; at demo scale their fixed per-job overhead
# dominates. The publish notebook evaluates checks concurrently over Delta
# reads; the default stays sequential because concurrent checks over lineage
# that still runs Python (mapInArrow) can exhaust local Python workers.
PUBLISH_INVARIANT_WORKERS = 8
_ROW_COUNT = object()  # marks queued row counts in the check queue


@dataclass
class InvariantReport:
    """Checks are queued, then evaluated concurrently on first read.

    Reading ``checks``, ``failures``, or ``passed`` evaluates every queued
    check; results are recorded in the order the checks were declared.
    """

    row_counts: dict[str, int] = field(default_factory=dict)
    max_workers: int = 1
    _pending: list[tuple[object, Callable[[], int]]] = field(default_factory=list, repr=False)
    _checks: list[str] = field(default_factory=list, repr=False)
    _failures: list[str] = field(default_factory=list, repr=False)

    @property
    def checks(self) -> list[str]:
        self.evaluate()
        return self._checks

    @property
    def failures(self) -> list[str]:
        self.evaluate()
        return self._failures

    @property
    def passed(self) -> bool:
        return not self.failures

    def evaluate(self) -> None:
        pending, self._pending = self._pending, []
        if not pending:
            return
        if self.max_workers <= 1:
            results = [(name, count()) for name, count in pending]
        else:
            with ThreadPoolExecutor(self.max_workers) as pool:
                futures = [
                    (name, pool.submit(inheritable_thread_target(count))) for name, count in pending
                ]
                results = [(name, future.result()) for name, future in futures]
        for name, bad_count in results:
            if name is _ROW_COUNT:
                continue
            self._checks.append(name)
            if bad_count:
                self._failures.append(f"{name}: {bad_count} violations")


def _check(report: InvariantReport, name: str, bad_count: Callable[[], int]) -> None:
    report._pending.append((name, bad_count))


def _check_count(report: InvariantReport, name: str, df: DataFrame) -> None:
    """Queue a row count; it is recorded as data, not as a check."""

    def count() -> int:
        report.row_counts[name] = df.count()
        return 0

    report._pending.append((_ROW_COUNT, count))


def run_invariants(
    spark: SparkSession,
    t: dict[str, DataFrame],
    *,
    max_workers: int = 1,
) -> InvariantReport:
    r = InvariantReport(max_workers=max_workers)
    for name, df in t.items():
        _check_count(r, name, df)
    r.evaluate()

    receipts, lines, pay = t["fact_receipts"], t["fact_receipt_lines"], t["fact_payments"]
    _check(
        r,
        "fact_receipts.receipt_id_ext unique",
        lambda: (
            r.row_counts["fact_receipts"] - receipts.select("receipt_id_ext").distinct().count()
        ),
    )
    _check(
        r,
        "fact_receipt_lines -> fact_receipts FK",
        lambda: lines.join(receipts, "receipt_id_ext", "left_anti").count(),
    )
    _check(
        r,
        "fact_payments xor keys",
        lambda: pay.filter(
            F.col("receipt_id_ext").isNotNull() == F.col("order_id_ext").isNotNull()
        ).count(),
    )
    _check(
        r,
        "fact_payments -> receipts FK",
        lambda: (
            pay.filter(F.col("receipt_id_ext").isNotNull())
            .join(receipts, "receipt_id_ext", "left_anti")
            .count()
        ),
    )

    products = t["dim_products"].select(F.col("ID").alias("product_id"))
    for tbl in [
        "fact_receipt_lines",
        "fact_online_order_lines",
        "fact_promo_lines",
        "fact_store_inventory_txn",
        "fact_dc_inventory_txn",
        "fact_reorders",
    ]:
        _check(
            r,
            f"{tbl} -> dim_products FK",
            lambda tbl=tbl: t[tbl].join(products, "product_id", "left_anti").count(),
        )

    stores = t["dim_stores"].select(F.col("ID").alias("store_id"))
    for tbl in [
        "fact_receipts",
        "fact_store_inventory_txn",
        "fact_reorders",
        "fact_store_ops",
        "fact_foot_traffic",
        "fact_ble_pings",
    ]:
        _check(
            r,
            f"{tbl} -> dim_stores FK",
            lambda tbl=tbl: t[tbl].join(stores, "store_id", "left_anti").count(),
        )

    for tbl, df in t.items():
        if tbl.startswith("fact_") and "event_date" in df.columns:
            _check(
                r,
                f"{tbl}.event_date not null",
                lambda df=df: df.filter(F.col("event_date").isNull()).count(),
            )

    oh, ol = t["fact_online_order_headers"], t["fact_online_order_lines"]
    _check(
        r,
        "online order ids unique",
        lambda: (
            r.row_counts["fact_online_order_headers"] - oh.select("order_id_ext").distinct().count()
        ),
    )
    _check(
        r,
        "online lines -> headers FK",
        lambda: ol.join(oh, ol.order_id == oh.order_id_ext, "left_anti").count(),
    )

    so = t["fact_stockouts"]
    _check(
        r,
        "stockouts StoreID xor DCID",
        lambda: so.filter(F.col("StoreID").isNotNull() == F.col("DCID").isNotNull()).count(),
    )

    # Per-receipt promo discount consistency: for receipts present in
    # fact_promo_lines (SALE only by construction), the summed discount must
    # equal the implied line-level discount sum(unit_cents*quantity - ext_cents).
    promo_sum = (
        t["fact_promo_lines"]
        .groupBy("receipt_id_ext")
        .agg(F.sum("discount_cents").alias("promo_discount"))
    )
    line_sum = lines.groupBy("receipt_id_ext").agg(
        F.sum(F.col("unit_cents") * F.col("quantity") - F.col("ext_cents")).alias("line_discount")
    )
    _check(
        r,
        "promo discount consistency",
        lambda: (
            promo_sum.join(line_sum, "receipt_id_ext", "left")
            .filter(
                F.col("line_discount").isNull()
                | (F.col("promo_discount") != F.col("line_discount"))
            )
            .count()
        ),
    )

    # --- dimension geography FK integrity (datagen foreign_key validator parity)
    geo_ids = t["dim_geographies"].select(F.col("ID").alias("geo_id"))
    for dim in ["dim_stores", "dim_distribution_centers", "dim_customers"]:
        _check(
            r,
            f"{dim} -> dim_geographies FK",
            lambda dim=dim: (
                t[dim]
                .select(F.col("GeographyID").alias("geo_id"))
                .join(geo_ids, "geo_id", "left_anti")
                .count()
            ),
        )

    # --- DC coverage on facts that reference a distribution center
    dc_ids = t["dim_distribution_centers"].select(F.col("ID").alias("dc_id"))
    for tbl in ["fact_dc_inventory_txn", "fact_truck_moves", "fact_reorders"]:
        _check(
            r,
            f"{tbl} -> dim_distribution_centers FK",
            lambda tbl=tbl: (
                t[tbl]
                .filter(F.col("dc_id").isNotNull())
                .select("dc_id")
                .join(dc_ids, "dc_id", "left_anti")
                .count()
            ),
        )

    # --- truck coverage on logistics facts
    truck_ids = t["dim_trucks"].select(F.col("ID").alias("truck_id"))
    for tbl in ["fact_truck_moves", "fact_truck_inventory"]:
        _check(
            r,
            f"{tbl} -> dim_trucks FK",
            lambda tbl=tbl: (
                t[tbl]
                .filter(F.col("truck_id").isNotNull())
                .select("truck_id")
                .join(truck_ids, "truck_id", "left_anti")
                .count()
            ),
        )

    # --- truck timing: arrival (eta) must not be after completion (etd)
    tm = t["fact_truck_moves"]
    _check(
        r,
        "fact_truck_moves etd >= eta",
        lambda: tm.filter(
            F.col("eta").isNotNull() & F.col("etd").isNotNull() & (F.col("etd") < F.col("eta"))
        ).count(),
    )

    # --- customer coverage on facts that resolve a customer (nullable for some)
    customer_ids = t["dim_customers"].select(F.col("ID").alias("customer_id"))
    for tbl in ["fact_receipts", "fact_online_order_headers"]:
        _check(
            r,
            f"{tbl} -> dim_customers FK",
            lambda tbl=tbl: (
                t[tbl]
                .filter(F.col("customer_id").isNotNull())
                .select("customer_id")
                .join(customer_ids, "customer_id", "left_anti")
                .count()
            ),
        )
    # fact_marketing.customer_id is a nullable double (low resolution rate); cast.
    _check(
        r,
        "fact_marketing -> dim_customers FK",
        lambda: (
            t["fact_marketing"]
            .filter(F.col("customer_id").isNotNull())
            .select(F.col("customer_id").cast("long").alias("customer_id"))
            .join(customer_ids, "customer_id", "left_anti")
            .count()
        ),
    )

    # --- dim_products pricing constraints (datagen pricing validator parity)
    prod = t["dim_products"]
    _check(
        r,
        "dim_products pricing Cost<SalePrice<=MSRP",
        lambda: prod.filter(
            ~(
                (F.col("Cost") > 0)
                & (F.col("Cost") < F.col("SalePrice"))
                & (F.col("SalePrice") <= F.col("MSRP"))
            )
        ).count(),
    )

    # --- IMP-007 marketing attribution -------------------------------------
    # gross - discount = net holds for every SALE receipt / online order,
    # attributed or not (discount is always applied before tax).
    _check(
        r,
        "fact_receipts gross - discount = net subtotal",
        lambda: receipts.filter(
            F.col("gross_subtotal_cents") - F.col("discount_cents") != F.col("subtotal_cents")
        ).count(),
    )
    _check(
        r,
        "fact_online_order_headers gross - discount = net subtotal",
        lambda: oh.filter(
            F.col("gross_subtotal_cents") - F.col("discount_cents") != F.col("subtotal_cents")
        ).count(),
    )

    if "fact_marketing_attribution" in t:
        fma = t["fact_marketing_attribution"]
        _check(
            r,
            "fact_marketing_attribution.attribution_id unique",
            lambda: (
                r.row_counts["fact_marketing_attribution"]
                - fma.select("attribution_id").distinct().count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution xor purchase keys",
            lambda: fma.filter(
                F.col("receipt_id_ext").isNotNull() == F.col("order_id_ext").isNotNull()
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution purchase_type matches xor key",
            lambda: fma.filter(
                ((F.col("purchase_type") == "STORE") != F.col("receipt_id_ext").isNotNull())
                | ((F.col("purchase_type") == "ONLINE") != F.col("order_id_ext").isNotNull())
            ).count(),
        )
        # one journey per purchase, one purchase per journey: distinct
        # non-NULL attribution_journey_id count must equal ATTRIBUTED row count.
        attributed = fma.filter(F.col("attribution_status") == "ATTRIBUTED")
        _check(
            r,
            "fact_marketing_attribution one journey per purchase",
            lambda: (
                attributed.count() - attributed.select("attribution_journey_id").distinct().count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution journey_id set iff ATTRIBUTED",
            lambda: fma.filter(
                (F.col("attribution_status") == "ATTRIBUTED")
                != F.col("attribution_journey_id").isNotNull()
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution -> fact_receipts FK",
            lambda: (
                fma.filter(F.col("receipt_id_ext").isNotNull())
                .select("receipt_id_ext")
                .join(receipts.select("receipt_id_ext"), "receipt_id_ext", "left_anti")
                .count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution -> online headers FK",
            lambda: (
                fma.filter(F.col("order_id_ext").isNotNull())
                .select("order_id_ext")
                .join(oh.select("order_id_ext"), "order_id_ext", "left_anti")
                .count()
            ),
        )
        # financial reconciliation: gross-discount=net, net+tax=total, and for
        # any row that reached a payment decision (not RECONCILIATION_FAILED)
        # the recorded payment equals the purchase total.
        _check(
            r,
            "fact_marketing_attribution gross - discount = net",
            lambda: fma.filter(
                F.col("gross_subtotal_cents") - F.col("discount_cents")
                != F.col("net_subtotal_cents")
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution net + tax = total",
            lambda: fma.filter(
                F.col("net_subtotal_cents") + F.col("tax_cents") != F.col("total_cents")
            ).count(),
        )
        _check(
            r,
            "fact_marketing_attribution approved payment = total",
            lambda: (
                fma.filter(
                    F.col("attribution_status").isin("ATTRIBUTED", "UNATTRIBUTED_NO_JOURNEY")
                )
                .filter(F.col("payment_cents") != F.col("total_cents"))
                .count()
            ),
        )
        _check(
            r,
            "fact_marketing_attribution attributed_revenue matches status",
            lambda: fma.filter(
                (
                    (F.col("attribution_status") == "ATTRIBUTED")
                    & (F.col("attributed_revenue_cents") != F.col("net_subtotal_cents"))
                )
                | (
                    (F.col("attribution_status") != "ATTRIBUTED")
                    & (F.col("attributed_revenue_cents") != 0)
                )
            ).count(),
        )
        # window/tie correctness: every ATTRIBUTED row's last touch must be
        # inside the inclusive 7-day window, with a non-negative lag.
        _check(
            r,
            "fact_marketing_attribution touch within 7-day window",
            lambda: attributed.filter(
                F.col("touch_ts").isNull()
                | (F.col("lag_seconds") < 0)
                | (F.col("lag_seconds") > F.col("attribution_window_days") * 86400)
            ).count(),
        )

        # fact_marketing: each journey's touches are exactly the two rows
        # created for it (older + newer), never orphaned or duplicated.
        mk = t["fact_marketing"]
        journey_touch_counts = (
            mk.filter(F.col("attribution_journey_id").isNotNull())
            .groupBy("attribution_journey_id")
            .agg(F.count("*").alias("n"))
        )
        _check(
            r,
            "fact_marketing journeys have exactly 2 touches",
            lambda: journey_touch_counts.filter(F.col("n") != 2).count(),
        )
        _check(
            r,
            "fact_marketing journeys <-> attributed purchases 1:1",
            lambda: (
                journey_touch_counts.join(
                    attributed.select("attribution_journey_id"),
                    "attribution_journey_id",
                    "full_outer",
                )
                .filter(F.col("n").isNull() | F.col("attribution_journey_id").isNull())
                .count()
            ),
        )
    _run_inventory_invariants(r, t)
    _run_business_invariants(r, t)
    r.evaluate()
    return r


def _inventory_balance_checks(
    report: InvariantReport,
    table: str,
    transactions: DataFrame,
    node: str,
) -> None:
    ordered = Window.partitionBy(node, "product_id").orderBy("event_ts", "trace_id")
    running = ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    checked = (
        transactions.withColumn("_expected_balance", F.sum("quantity").over(running))
        .withColumn("_position", F.row_number().over(ordered))
        .withColumn(
            "_initial_count",
            F.sum(F.when(F.col("txn_type") == "INITIAL", 1).otherwise(0)).over(
                Window.partitionBy(node, "product_id")
            ),
        )
    )
    invalid = (
        F.col(node).isNull()
        | F.col("product_id").isNull()
        | F.col("event_ts").isNull()
        | F.col("trace_id").isNull()
        | F.col("quantity").isNull()
        | F.col("balance").isNull()
        | (F.col("balance") < 0)
    )
    counts = checked.agg(
        F.count(F.when(invalid, 1)).alias("physical"),
        F.count(F.when(~F.col("balance").eqNullSafe(F.col("_expected_balance")), 1)).alias(
            "conservation"
        ),
        F.count(
            F.when(
                (F.col("_initial_count") != 1)
                | ((F.col("_position") == 1) & ~F.col("txn_type").eqNullSafe(F.lit("INITIAL"))),
                1,
            )
        ).alias("opening"),
        (F.count("*") - F.countDistinct("trace_id")).alias("duplicate_traces"),
    ).first()
    _check(report, f"{table} nonnegative physical balances", lambda: counts.physical)
    _check(report, f"{table} running quantity conservation", lambda: counts.conservation)
    _check(report, f"{table} exactly one opening INITIAL", lambda: counts.opening)
    _check(report, f"{table}.trace_id unique", lambda: counts.duplicate_traces)


def _inventory_mirror_violations(actual: DataFrame, expected: DataFrame) -> int:
    """Compare both directions as multisets, so duplicates cannot cancel out."""
    return actual.exceptAll(expected).unionByName(expected.exceptAll(actual)).count()


def _run_inventory_invariants(r: InvariantReport, t: dict[str, DataFrame]) -> None:
    """Check physical stock only; independent online fulfillment is not reserved here."""
    store = t["fact_store_inventory_txn"]
    dc = t["fact_dc_inventory_txn"]
    truck = t["fact_truck_inventory"]
    _inventory_balance_checks(r, "fact_store_inventory_txn", store, "store_id")
    _inventory_balance_checks(r, "fact_dc_inventory_txn", dc, "dc_id")

    requested, unfulfilled = F.col("requested_quantity"), F.col("unfulfilled_quantity")
    valid_demand = (
        F.when(
            F.col("txn_type") == "SALE",
            (F.col("quantity") < 0) & (requested == -F.col("quantity")) & (unfulfilled == 0),
        )
        .when(
            F.col("txn_type") == "LOST_SALE",
            (F.col("quantity") == 0)
            & (requested > 0)
            & (unfulfilled == requested)
            & (requested > F.col("balance")),
        )
        .otherwise((requested == 0) & (unfulfilled == 0))
    )
    _check(
        r,
        "store requested = fulfilled + unfulfilled quantity",
        lambda: store.filter(~F.coalesce(valid_demand, F.lit(False))).count(),
    )

    receipt_stock = (
        t["fact_receipt_lines"]
        .join(
            t["fact_receipts"].select("receipt_id_ext", "store_id", "receipt_type"),
            "receipt_id_ext",
        )
        .select(
            F.concat(F.lit("TRC-INV-"), "receipt_id_ext", F.lit("-"), "line_num").alias("trace_id"),
            "store_id",
            "product_id",
            "event_ts",
            (-F.col("quantity")).cast("long").alias("quantity"),
            "receipt_type",
        )
    )
    receipt_keys = ["trace_id", "store_id", "product_id", "event_ts", "quantity"]
    _check(
        r,
        "fulfilled receipt lines <-> store SALE conservation",
        lambda: _inventory_mirror_violations(
            store.filter(F.col("txn_type") == "SALE").select(*receipt_keys),
            receipt_stock.filter(F.col("receipt_type") == "SALE").select(*receipt_keys),
        ),
    )
    # Refunds may be destroyed rather than restocked. Every actual add-back
    # must match a refunded line, but not every refund must add stock.
    _check(
        r,
        "store RETURN backed by refunded lines",
        lambda: (
            store.filter(F.col("txn_type") == "RETURN")
            .select(*receipt_keys)
            .exceptAll(
                receipt_stock.filter(F.col("receipt_type") == "RETURN").select(*receipt_keys)
            )
            .count()
        ),
    )

    for action, ledger, kind, node, source, sign in [
        ("LOAD", dc, "OUTBOUND_SHIPMENT", "dc_id", "Source", -1),
        ("UNLOAD", store, "INBOUND_SHIPMENT", "store_id", "source", 1),
    ]:
        expected = truck.filter(F.col("action") == action).select(
            F.col("location_id").alias("node_id"),
            "product_id",
            "event_ts",
            (F.col("quantity") * sign).alias("quantity"),
            "shipment_id",
        )
        actual = ledger.filter(F.col("txn_type") == kind).select(
            F.col(node).alias("node_id"),
            "product_id",
            "event_ts",
            "quantity",
            F.col(source).alias("shipment_id"),
        )
        _check(
            r,
            f"truck {action} <-> {node} ledger conservation",
            lambda actual=actual, expected=expected: _inventory_mirror_violations(actual, expected),
        )

    valid_cargo = (
        F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("shipment_id").isNotNull()
        & F.col("truck_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("event_ts").isNotNull()
        & F.col("location_id").isNotNull()
        & F.col("trace_id").isNotNull()
        & (
            ((F.col("action") == "LOAD") & (F.col("location_type") == "DC"))
            | ((F.col("action") == "UNLOAD") & (F.col("location_type") == "STORE"))
        )
    )
    cargo_order = (
        Window.partitionBy("shipment_id", "product_id", "truck_id")
        .orderBy("event_ts", "trace_id")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    cargo_counts = (
        truck.withColumn(
            "_cargo_balance",
            F.sum(
                F.when(F.col("action") == "LOAD", F.col("quantity")).otherwise(-F.col("quantity"))
            ).over(cargo_order),
        )
        .agg(
            F.count(F.when(~F.coalesce(valid_cargo, F.lit(False)), 1)).alias("invalid"),
            F.count(F.when(F.col("_cargo_balance") < 0, 1)).alias("negative"),
            (F.count("*") - F.countDistinct("trace_id")).alias("duplicate_traces"),
            (F.count("*") - F.countDistinct("shipment_id", "product_id", "action")).alias(
                "duplicate_actions"
            ),
        )
        .first()
    )
    _check(r, "truck cargo positive with valid action and location", lambda: cargo_counts.invalid)
    _check(r, "truck nonnegative cargo balances", lambda: cargo_counts.negative)
    _check(r, "fact_truck_inventory.trace_id unique", lambda: cargo_counts.duplicate_traces)
    _check(r, "truck shipment/product actions unique", lambda: cargo_counts.duplicate_actions)
    loads = truck.filter(F.col("action") == "LOAD").select(
        "shipment_id",
        "product_id",
        "truck_id",
        F.col("quantity").alias("_loaded"),
        F.col("event_ts").alias("_loaded_at"),
    )
    # A final-day LOAD may still be in transit; only UNLOAD requires a prior LOAD.
    _check(
        r,
        "truck UNLOAD backed by prior equal LOAD",
        lambda: (
            truck.filter(F.col("action") == "UNLOAD")
            .join(loads, ["shipment_id", "product_id", "truck_id"], "left")
            .filter(
                F.col("_loaded").isNull()
                | ~F.col("quantity").eqNullSafe(F.col("_loaded"))
                | (F.col("event_ts") < F.col("_loaded_at")),
            )
            .count()
        ),
    )


def _run_business_invariants(r: InvariantReport, t: dict[str, DataFrame]) -> None:
    """IMP-010 shared business invariants: deliberate seeds must not produce
    sales while a store is closed, pre-launch sales, same-day returns, or
    same-time dimension lifecycles."""

    receipts = t["fact_receipts"]

    # --- sales-while-closed: every SALE receipt's local hour must fall inside
    # its store's operating window. RETURN rows are stamped at noon, always open.
    store_hours = t["dim_stores"].select(F.col("ID").alias("store_id"), "operating_hours")
    open_h, close_h = _open_close_cols(F.col("operating_hours"))
    hour_checked = (
        receipts.select("store_id", F.hour("event_ts").alias("_h"))
        .join(store_hours, "store_id")
        .withColumn("_open", open_h)
        .withColumn("_close", close_h)
    )
    _check(
        r,
        "fact_receipts within store operating hours",
        lambda: hour_checked.filter(
            (F.col("_h") < F.col("_open")) | (F.col("_h") >= F.col("_close"))
        ).count(),
    )

    # --- pre-launch sales: no receipt line may sell a product before its
    # LaunchDate. Compares the calendar day of the sale to the product launch.
    launch = t["dim_products"].select(
        F.col("ID").alias("product_id"), F.to_date("LaunchDate").alias("_launch_date")
    )
    _check(
        r,
        "fact_receipt_lines no pre-launch sales",
        lambda: (
            t["fact_receipt_lines"]
            .filter(F.col("quantity") > 0)  # SALE lines only; RETURN lines are negative
            .join(launch, "product_id")
            .filter(F.col("event_date") < F.col("_launch_date"))
            .count()
        ),
    )

    # --- same-day returns: a RETURN must post strictly after its originating
    # SALE's day. The final fact_receipts contract can't carry the sale link, so
    # the per-receipt guarantee is enforced in returns.build_return_headers and
    # proven by an adversarial unit test. Here we assert the checkable global
    # surrogates: every RETURN is dated at noon (return lifecycle) and strictly
    # after the earliest SALE on record (no return predates all sales).
    returns = receipts.filter(F.col("receipt_type") == "RETURN")
    min_sale = (
        receipts.filter(F.col("receipt_type") == "SALE")
        .agg(F.min("event_date").alias("_min"))
        .collect()[0]["_min"]
    )
    if min_sale is not None:
        _check(
            r,
            "fact_receipts no return before first sale",
            lambda: returns.filter(F.col("event_date") <= F.lit(min_sale)).count(),
        )
    _check(
        r,
        "fact_receipts returns posted at noon",
        lambda: returns.filter(F.hour("event_ts") != F.lit(12)).count(),
    )

    # --- same-time lifecycles: a product's LaunchDate must be a single instant
    # (no null) and strictly before the far-future horizon; guards against
    # degenerate zero-length lifecycles introduced by adversarial seeds.
    _check(
        r,
        "dim_products LaunchDate present",
        lambda: t["dim_products"].filter(F.col("LaunchDate").isNull()).count(),
    )

# --- retail_setup/generation/handoff.py ---
"""Hand generated stage outputs between setup notebooks.

Each setup notebook runs a slice of the generation stage graph in its own Spark
session. Stage outputs that a later notebook consumes are written as handoff
tables named ``gen__<stage>__<output>`` in the Silver staging schema
(``<silver_db>_stage``). The publish notebook promotes them into Silver and then
drops every handoff.

Historical setup is single-writer, so handoff names are fixed and an activity
retry simply overwrites them. Every handoff records the generation
fingerprint (the configuration it was generated for) atomically with its data
(Delta commit ``userMetadata``); loading a handoff produced for a different
configuration fails instead of silently mixing runs.

Path mode (``base_path=``) stores parquet plus a fingerprint marker file so
local tests need no Delta dependency.
"""

import hashlib
import shutil
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession


HANDOFF_PREFIX = "gen__"
_FINGERPRINT_MARKER = "_retail_generation"


def delete_lakehouse_path(spark: SparkSession, path: str) -> None:
    """Recursively delete a path on the session's default filesystem (if present)."""
    hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(path)
    hadoop_path.getFileSystem(spark._jsc.hadoopConfiguration()).delete(hadoop_path, True)


def generation_fingerprint(cfg: GenerationConfig) -> str:
    """Stable identity of everything that shapes generated data."""
    payload = cfg.model_dump_json(exclude={"dictionary_root"})
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:32]


class HandoffMismatchError(RuntimeError):
    """A handoff is missing or belongs to a different generation."""


class HandoffStore:
    def __init__(
        self,
        spark: SparkSession,
        cfg: GenerationConfig,
        *,
        lakehouse: str | None = None,
        base_path: str | None = None,
    ) -> None:
        if (lakehouse is None) == (base_path is None):
            raise ValueError("Provide exactly one of lakehouse= or base_path=")
        self.spark = spark
        self.fingerprint = generation_fingerprint(cfg)
        self.lakehouse = lakehouse
        self.schema = f"{cfg.silver_db}_stage"
        self.base_path = Path(base_path) if base_path is not None else None

    def name(self, stage: str, output: str) -> str:
        table = f"{HANDOFF_PREFIX}{stage}__{output}"
        if self.base_path is not None:
            return str(self.base_path / self.schema / table)
        return f"{self.lakehouse}.{self.schema}.{table}"

    def ensure_schema(self) -> None:
        """Create the handoff schema (setup-02 does this before any parallel writer)."""
        if self.base_path is None:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.lakehouse}.{self.schema}")

    def save(self, stage: str, output: str, frame: DataFrame) -> DataFrame:
        """Write one stage output and return a frame that reads it back."""
        name = self.name(stage, output)
        if self.base_path is None:
            (
                frame.write.format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option("userMetadata", self.fingerprint)
                .saveAsTable(name)
            )
            return self.spark.table(name)
        frame.write.mode("overwrite").parquet(name)
        (Path(name) / _FINGERPRINT_MARKER).write_text(self.fingerprint, encoding="utf-8")
        return self.spark.read.parquet(name)

    def load(self, stage: str, output: str) -> DataFrame:
        name = self.name(stage, output)
        found = self._fingerprint_of(name)
        if found != self.fingerprint:
            reason = (
                "is missing" if found is None else "was generated for a different configuration"
            )
            raise HandoffMismatchError(
                f"Handoff {name} {reason}; rerun the setup notebook that produces "
                f"stage {stage!r} (or the whole setup pipeline)."
            )
        if self.base_path is None:
            return self.spark.table(name)
        return self.spark.read.parquet(name)

    def drop_all(self) -> None:
        """Remove every handoff, including leftovers from failed runs."""
        if self.base_path is not None:
            root = self.base_path / self.schema
            for path in root.glob(f"{HANDOFF_PREFIX}*") if root.exists() else []:
                shutil.rmtree(path)
            return
        self.ensure_schema()
        for row in self.spark.sql(f"SHOW TABLES IN {self.lakehouse}.{self.schema}").collect():
            if row.tableName.startswith(HANDOFF_PREFIX):
                self.spark.sql(
                    f"DROP TABLE IF EXISTS {self.lakehouse}.{self.schema}.{row.tableName}"
                )

    def _fingerprint_of(self, name: str) -> str | None:
        if self.base_path is not None:
            marker = Path(name) / _FINGERPRINT_MARKER
            return marker.read_text(encoding="utf-8") if marker.exists() else None
        if not self.spark.catalog.tableExists(name):
            return None
        latest = self.spark.sql(f"DESCRIBE HISTORY {name} LIMIT 1").collect()
        return latest[0]["userMetadata"] if latest else None

# --- retail_setup/generation/engine.py ---
"""Generation stage graph, shared by local runs and the setup pipeline notebooks.

Generation is a dependency graph of stages (``STAGES``). Each stage builds one
or more frames from the dimensions and the outputs of the stages it depends
on. ``generate_all`` runs the whole graph in one Spark session (local tests);
the setup pipeline instead runs each notebook's slice of the graph in its own
session (``run_generation_notebook``), hands outputs between notebooks as
Delta tables, and ``load_publication`` assembles the Silver tables for the
publish notebook. In both modes, stages whose inputs are ready run
concurrently. Generation is deterministic, so neither the split nor the
completion order changes the output.
"""

import warnings
from collections.abc import Callable, Iterable
from concurrent.futures import FIRST_COMPLETED, Future, ThreadPoolExecutor, wait
from dataclasses import dataclass
from datetime import date

from pyspark import inheritable_thread_target
from pyspark.sql import DataFrame, SparkSession



@dataclass
class GenerationResult:
    tables: dict[str, DataFrame]


def _shift_year(d: date, years: int) -> date:
    """Shift a date by whole years; Feb 29 falls back to Feb 28."""
    try:
        return d.replace(year=d.year + years)
    except ValueError:
        return date(d.year + years, d.month, 28)


DICTIONARIES_NOTEBOOK = "setup-01-seed-dictionaries"
DIMENSIONS_NOTEBOOK = "setup-02-generate-dimensions"
PUBLISH_NOTEBOOK = "setup-10-publish-silver"
# Ready stages run concurrently; Spark schedules their jobs side by side so
# narrow stages, the Python inventory policy, and stragglers do not leave the
# pool idle.
MAX_PARALLEL_STAGES = 6

StageOutputs = dict[str, DataFrame]


@dataclass(frozen=True)
class GenerationContext:
    spark: SparkSession
    dicts: DictionarySet
    cfg: GenerationConfig
    dims: dict[str, DataFrame]


@dataclass(frozen=True)
class Stage:
    """One generation step: ``build(ctx, *outputs_of_deps) -> outputs``.

    ``notebook`` is the setup pipeline notebook that runs the stage. Stages
    with ``handoff=False`` are only consumed inside their own notebook and
    are never written as handoff tables.
    """

    name: str
    notebook: str
    deps: tuple[str, ...]
    outputs: tuple[str, ...]
    build: Callable[..., StageOutputs]
    handoff: bool = True


_RECEIPT_GROUP = ("fact_receipts", "fact_receipt_lines", "fact_payments")


def _accepted(plan: StageOutputs) -> StageOutputs:
    return {name: plan[name] for name in _RECEIPT_GROUP}


def _build_candidates(ctx: GenerationContext) -> StageOutputs:
    return generate_receipts_group(ctx.spark, ctx.dims, ctx.dicts.profile, ctx.cfg)


def _build_plan(ctx: GenerationContext, candidates: StageOutputs) -> StageOutputs:
    # Inventory-constrained fulfillment precedes every receipt consumer.
    plan = resolve_fulfillment(candidates, ctx.dims, ctx.cfg)
    return {"return_headers": plan.return_headers, "events": plan.events, **plan.sales}


def _build_online(ctx: GenerationContext) -> StageOutputs:
    return generate_online_orders(ctx.spark, ctx.dims, ctx.dicts.profile, ctx.cfg)


def _build_ads(ctx: GenerationContext) -> StageOutputs:
    return {"fact_marketing": generate_marketing(ctx.spark, ctx.dims, ctx.cfg)}


def _build_refunds(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    return generate_returns(
        ctx.spark, _accepted(plan), ctx.dims, ctx.cfg, return_headers=plan["return_headers"]
    )


def _build_promos(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    promos, promo_lines = generate_promotions(ctx.spark, _accepted(plan))
    return {"fact_promotions": promos, "fact_promo_lines": promo_lines}


def _build_store_ops(ctx: GenerationContext) -> StageOutputs:
    return {"fact_store_ops": generate_store_ops(ctx.spark, ctx.dims, ctx.cfg)}


def _build_foot_traffic(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    return {
        "fact_foot_traffic": generate_foot_traffic(
            ctx.spark, plan["fact_receipts"], ctx.dims, ctx.cfg
        )
    }


def _build_ble(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    pings, zone_changes = generate_ble(ctx.spark, plan["fact_receipts"], ctx.dims, ctx.cfg)
    return {"fact_ble_pings": pings, "fact_customer_zone_changes": zone_changes}


def _build_ledgers(ctx: GenerationContext, plan: StageOutputs) -> StageOutputs:
    result = FulfillmentResult(_accepted(plan), plan["return_headers"], plan["events"])
    # Restocking returns are already events in the fulfillment plan, so the
    # ledgers need no refund frames.
    return generate_inventory_chain(
        ctx.spark, _accepted(plan), {}, ctx.dims, ctx.cfg, fulfillment=result
    )


def _union(frames: list[DataFrame]) -> DataFrame:
    combined = frames[0]
    for frame in frames[1:]:
        combined = combined.unionByName(frame)
    return combined


def _build_attributed(
    ctx: GenerationContext,
    plan: StageOutputs,
    refunds: StageOutputs,
    online: StageOutputs,
    promos: StageOutputs,
    ads: StageOutputs,
) -> StageOutputs:
    # IMP-007: deterministic last-touch attribution over every purchase stream.
    inputs = {
        **ctx.dims,
        **promos,
        **ads,
        "fact_receipts": _union([plan["fact_receipts"], refunds["fact_receipts"]]),
        "fact_receipt_lines": _union([plan["fact_receipt_lines"], refunds["fact_receipt_lines"]]),
        "fact_online_order_headers": online["fact_online_order_headers"],
        "fact_payments": _union(
            [plan["fact_payments"], refunds["fact_payments"], online["payments"]]
        ),
    }
    out = generate_attribution(ctx.spark, inputs, ctx.cfg)
    return {name: out[name] for name in _ATTRIBUTED}


_ATTRIBUTED = (
    "fact_receipts",
    "fact_online_order_headers",
    "fact_payments",
    "fact_promotions",
    "fact_marketing",
    "fact_marketing_attribution",
)
_LEDGERS = (
    "fact_store_inventory_txn",
    "fact_dc_inventory_txn",
    "fact_truck_moves",
    "fact_truck_inventory",
    "fact_reorders",
    "fact_stockouts",
)

STAGES: tuple[Stage, ...] = (
    Stage(
        "candidates",
        "setup-03-generate-sales",
        (),
        _RECEIPT_GROUP,
        _build_candidates,
        handoff=False,
    ),
    Stage(
        "plan",
        "setup-03-generate-sales",
        ("candidates",),
        ("return_headers", "events", *_RECEIPT_GROUP),
        _build_plan,
    ),
    Stage(
        "online",
        "setup-04-generate-online-orders",
        (),
        ("fact_online_order_headers", "fact_online_order_lines", "payments"),
        _build_online,
    ),
    Stage("ads", "setup-05-generate-marketing", (), ("fact_marketing",), _build_ads),
    Stage(
        "refunds",
        "setup-06-generate-returns-and-promotions",
        ("plan",),
        _RECEIPT_GROUP,
        _build_refunds,
    ),
    Stage(
        "promos",
        "setup-06-generate-returns-and-promotions",
        ("plan",),
        ("fact_promotions", "fact_promo_lines"),
        _build_promos,
    ),
    Stage(
        "store_ops", "setup-07-generate-store-activity", (), ("fact_store_ops",), _build_store_ops
    ),
    Stage(
        "foot_traffic",
        "setup-07-generate-store-activity",
        ("plan",),
        ("fact_foot_traffic",),
        _build_foot_traffic,
    ),
    Stage(
        "ble",
        "setup-07-generate-store-activity",
        ("plan",),
        ("fact_ble_pings", "fact_customer_zone_changes"),
        _build_ble,
    ),
    Stage("ledgers", "setup-08-generate-inventory", ("plan",), _LEDGERS, _build_ledgers),
    Stage(
        "attributed",
        "setup-09-attribute-marketing",
        ("plan", "refunds", "online", "promos", "ads"),
        _ATTRIBUTED,
        _build_attributed,
    ),
)
_STAGE_BY_NAME = {stage.name: stage for stage in STAGES}
STAGE_NOTEBOOKS = tuple(dict.fromkeys(stage.notebook for stage in STAGES))

# Published Silver fact table -> the stage outputs it is made of (unioned in order).
SILVER_SOURCES: dict[str, tuple[tuple[str, str], ...]] = {
    "fact_receipts": (("attributed", "fact_receipts"),),
    "fact_receipt_lines": (("plan", "fact_receipt_lines"), ("refunds", "fact_receipt_lines")),
    "fact_online_order_headers": (("attributed", "fact_online_order_headers"),),
    "fact_online_order_lines": (("online", "fact_online_order_lines"),),
    "fact_payments": (("attributed", "fact_payments"),),
    "fact_promotions": (("attributed", "fact_promotions"),),
    "fact_promo_lines": (("promos", "fact_promo_lines"),),
    "fact_marketing": (("attributed", "fact_marketing"),),
    "fact_marketing_attribution": (("attributed", "fact_marketing_attribution"),),
    "fact_store_ops": (("store_ops", "fact_store_ops"),),
    "fact_foot_traffic": (("foot_traffic", "fact_foot_traffic"),),
    "fact_ble_pings": (("ble", "fact_ble_pings"),),
    "fact_customer_zone_changes": (("ble", "fact_customer_zone_changes"),),
    **{name: (("ledgers", name),) for name in _LEDGERS},
}
_PUBLISHED_STAGES = {stage for sources in SILVER_SOURCES.values() for stage, _ in sources}


def setup_notebook_dependencies() -> dict[str, tuple[str, ...]]:
    """Upstream notebooks of each generation and publish notebook.

    The setup pipeline's activity graph must equal this mapping (tested), so
    the pipeline is the stage graph drawn at notebook granularity. Generation
    regenerates dimensions in memory, so root generation notebooks need only
    the dictionaries; setup-02's dimension validation runs alongside them and
    gates publication instead.
    """
    graph: dict[str, tuple[str, ...]] = {}
    for notebook in STAGE_NOTEBOOKS:
        upstream = {
            _STAGE_BY_NAME[dep].notebook
            for stage in STAGES
            if stage.notebook == notebook
            for dep in stage.deps
        } - {notebook}
        graph[notebook] = tuple(sorted(upstream)) or (DICTIONARIES_NOTEBOOK,)
    graph[PUBLISH_NOTEBOOK] = tuple(
        sorted(
            {DIMENSIONS_NOTEBOOK} | {_STAGE_BY_NAME[stage].notebook for stage in _PUBLISHED_STAGES}
        )
    )
    return graph


def generate_dimension_frames(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    dims = generate_dimensions(spark, dicts, cfg)
    # dim_date is padded ±5 years — the semantic model's date relationships
    # need the headroom around the generated fact window.
    dims["dim_date"] = generate_dim_date(
        spark, _shift_year(cfg.start_date, -5), _shift_year(cfg.end_date, 5)
    )
    return dims


def generate_all(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    *,
    checkpoint_dir: str | None = None,
) -> GenerationResult:
    """Run the whole stage graph in one session and return every Silver table.

    Each stage's outputs are checkpointed before dependants consume them (see
    ``_checkpoint_frames``). With ``checkpoint_dir`` (a shared filesystem
    path) checkpoints are written there and survive executor loss; without
    it they stay on the executors, which suits local runs only. The caller
    owns deleting the directory once the tables are no longer needed.
    """
    reliable = checkpoint_dir is not None
    if reliable:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    # Every builder reads the dimensions; materialize them once instead of
    # re-serializing their driver-built rows from Python for each consumer.
    dims = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    ctx = GenerationContext(spark, dicts, cfg, dims)
    out = run_stages(ctx, STAGES, {}, lambda _stage, frames: _checkpoint_frames(frames, reliable))
    tables = {
        **dims,
        **{
            name: _union([out[stage][output] for stage, output in sources])
            for name, sources in SILVER_SOURCES.items()
        },
    }
    # Release stage data the published tables no longer read.
    referenced = {
        id(out[stage][output]) for sources in SILVER_SOURCES.values() for stage, output in sources
    }
    _release(
        frame for frames in out.values() for frame in frames.values() if id(frame) not in referenced
    )
    return GenerationResult(tables=tables)


def run_generation_notebook(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    notebook: str,
    store: HandoffStore,
    *,
    checkpoint_dir: str | None = None,
) -> dict[str, StageOutputs]:
    """Run one setup notebook's stages, reading and writing handoff tables.

    Upstream stage outputs are loaded from handoffs (fingerprint-checked);
    this notebook's handoff stages are written to handoffs, and internal
    stages are checkpointed (reliably under ``checkpoint_dir`` when given,
    as the setup notebooks always do).
    """
    stages = [stage for stage in STAGES if stage.notebook == notebook]
    if not stages:
        raise ValueError(f"no generation stages run in notebook {notebook!r}")
    reliable = checkpoint_dir is not None
    if reliable:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    dims = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    ctx = GenerationContext(spark, dicts, cfg, dims)
    local = {stage.name for stage in stages}
    upstream = {dep for stage in stages for dep in stage.deps} - local
    inputs = {
        dep: {output: store.load(dep, output) for output in _STAGE_BY_NAME[dep].outputs}
        for dep in sorted(upstream)
    }

    def materialize(stage: Stage, frames: StageOutputs) -> StageOutputs:
        if not stage.handoff:
            return _checkpoint_frames(frames, reliable)
        return _persist_frames(frames, lambda output, frame: store.save(stage.name, output, frame))

    out = run_stages(ctx, stages, inputs, materialize)
    _release(dims.values())
    return {name: out[name] for name in local if _STAGE_BY_NAME[name].handoff}


def load_publication(
    spark: SparkSession,
    dicts: DictionarySet,
    cfg: GenerationConfig,
    store: HandoffStore,
    *,
    checkpoint_dir: str | None = None,
) -> tuple[dict[str, DataFrame], dict[str, str]]:
    """Assemble every Silver table from handoffs for the publish notebook.

    Returns ``(tables, prestaged)``: ``prestaged`` maps each table that is a
    single handoff to that handoff's name, so publication promotes it directly
    instead of copying it to a second staging table.
    """
    reliable = checkpoint_dir is not None
    if reliable:
        spark.sparkContext.setCheckpointDir(checkpoint_dir)
    tables = _checkpoint_frames(generate_dimension_frames(spark, dicts, cfg), reliable)
    prestaged: dict[str, str] = {}
    for name, sources in SILVER_SOURCES.items():
        tables[name] = _union([store.load(stage, output) for stage, output in sources])
        if len(sources) == 1:
            prestaged[name] = store.name(*sources[0])
    return tables, prestaged


def run_stages(
    ctx: GenerationContext,
    stages: Iterable[Stage],
    inputs: dict[str, StageOutputs],
    materialize: Callable[[Stage, StageOutputs], StageOutputs],
) -> dict[str, StageOutputs]:
    """Run each stage once its dependencies finish, up to MAX_PARALLEL_STAGES at once.

    ``inputs`` holds already-available outputs of stages outside ``stages``.
    Each stage builds and materializes its frames in its own thread. A stage
    that publishes nothing is released once every consumer in this run has
    finished. After a failure no further stage starts; stages already
    running finish (cancelling their jobs mid-cache can deadlock Spark's cache
    manager), then the first error is raised.
    """
    pending = {stage.name: stage for stage in stages}
    results: dict[str, StageOutputs] = dict(inputs)
    consumers = {
        name: {other.name for other in pending.values() if name in other.deps} for name in pending
    }
    running: dict[Future, str] = {}
    failure: BaseException | None = None

    def run(stage: Stage) -> StageOutputs:
        frames = stage.build(ctx, *[results[dep] for dep in stage.deps])
        if set(frames) != set(stage.outputs):
            raise ValueError(
                f"stage {stage.name!r} built {sorted(frames)}, declared {sorted(stage.outputs)}"
            )
        return materialize(stage, frames)

    def release_finished() -> None:
        for name, users in consumers.items():
            if (
                name in results
                and name not in _PUBLISHED_STAGES
                and name not in inputs
                and users
                and all(user in results for user in users)
            ):
                _release(results[name].values())
                users.clear()

    with ThreadPoolExecutor(MAX_PARALLEL_STAGES) as pool:
        while pending or running:
            for name, stage in list(pending.items()):
                if failure is not None:
                    break
                if len(running) < MAX_PARALLEL_STAGES and all(d in results for d in stage.deps):
                    del pending[name]
                    # Copies the caller's Spark local properties (job group,
                    # Fabric run tagging) into the worker thread.
                    running[pool.submit(_inheriting(run), stage)] = name
            if not running:
                if failure is not None:
                    break
                raise ValueError(f"generation stages have unmet dependencies: {sorted(pending)}")
            done, _ = wait(running, return_when=FIRST_COMPLETED)
            for future in done:
                name = running.pop(future)
                try:
                    results[name] = future.result()
                except BaseException as exc:  # noqa: BLE001 - re-raised below
                    failure = failure or exc
            if failure is None:
                release_finished()
    if failure is not None:
        raise failure
    return {name: frames for name, frames in results.items() if name not in inputs}


def _inheriting(fn: Callable) -> Callable:
    """Wrap ``fn`` so its thread inherits the caller's Spark local properties.

    Uses the function-only form, which every supported PySpark accepts (the
    session form is not available on all Fabric runtimes). Newer PySpark warns
    that job tags are not inherited; generation does not use job tags.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Spark session is not provided")
        return inheritable_thread_target(fn)


def _persist_frames(
    frames: StageOutputs,
    persist: Callable[[str, DataFrame], DataFrame],
) -> StageOutputs:
    """Persist sibling frames one at a time, computing shared work once.

    Siblings are cached first so a later sibling reuses work an earlier one
    computed. They are persisted sequentially: concurrent jobs over related
    cached plans can deadlock Spark's cache manager when one of them fails.
    Stages, and the notebooks around them, still run concurrently.
    """
    cached = {name: frame.cache() for name, frame in frames.items()}
    try:
        return {name: persist(name, frame) for name, frame in cached.items()}
    finally:
        for frame in cached.values():
            frame.unpersist()


def _checkpoint_frames(frames: StageOutputs, reliable: bool) -> StageOutputs:
    """Materialize each frame once and cut its lineage.

    Generation reuses frames heavily, so without truncation every downstream
    plan embeds the full upstream lineage: at full-demo scale the driver spent
    most of its time re-analyzing and cache-matching those plans for every
    action. Reliable checkpoints are written to the SparkContext checkpoint
    directory and survive executor loss; local checkpoints keep rows on the
    executors that computed them and are only safe for local runs.
    """

    def checkpoint(_name: str, frame: DataFrame) -> DataFrame:
        return frame.checkpoint(eager=True) if reliable else frame.localCheckpoint(eager=True)

    return _persist_frames(frames, checkpoint)


def _release(frames: Iterable[DataFrame]) -> None:
    """Free checkpointed rows: executor blocks and, if reliable, their files.

    Frames that are not checkpoints (for example handoff table scans) are
    left alone.
    """
    for frame in frames:
        plan = frame._jdf.queryExecution().logical()
        if plan.getClass().getSimpleName() != "LogicalRDD":
            continue
        rdd = plan.rdd()
        rdd.unpersist(False)
        checkpoint_file = rdd.getCheckpointFile()
        if checkpoint_file.isDefined():
            delete_lakehouse_path(frame.sparkSession, checkpoint_file.get())

# --- retail_setup/generation/publication.py ---
"""Stage -> validate -> promote -> (rollback) coordinator for historical
setup publication (IMP-002).

Fabric Lakehouse targets are plain Delta tables; there is no cross-table
transaction spanning several ``saveAsTable``/``CREATE OR REPLACE TABLE``
calls. This module implements the staged-promotion pattern used by
setup-02/03/04 instead:

1. Every candidate table is first written to a run-scoped **staging**
   table/location — the final target is never touched during staging.
2. Each staged table is validated (schema + row count) against its source
   before any promotion happens. All staging and validation must succeed
   before a single final target changes.
3. Promotion swaps each staged table into its fixed final name. If the Nth
   promotion fails, every already-promoted target *and the Nth target
   itself* is rolled back: targets that existed before this run are
   restored to their pre-run Delta version (``RESTORE TABLE ... TO VERSION
   AS OF``); targets that did not exist before this run are dropped. The
   failed target is included because a Delta command can commit before a
   client/network error is observed by the caller — restoring/dropping it
   anyway is safe and idempotent (a no-op if it never actually committed)
   and keeps rollback fail-closed instead of trusting the exception.
   Rollback keeps trying every changed target even if one restore/drop
   fails, and the run is marked ROLLED_BACK (all recovered) or
   ROLLBACK_FAILED (some did not recover — this is never reported as
   success).
4. Staging is cleaned up once the run reaches a terminal state that no
   longer needs it (COMPLETED; FAILED before any promotion; ROLLED_BACK with
   a clean rollback). ROLLBACK_FAILED preserves staging for manual recovery
   and logs the preserved names. Cleanup itself never raises: it is
   attempted for every staged table and any failures are collected. A
   cleanup failure after an otherwise-successful promotion or rollback is
   never reported as plain COMPLETED/ROLLED_BACK (that would hide leftover
   staging) but is also never conflated with a data failure — it is
   reported as COMPLETED_CLEANUP_FAILED / ROLLED_BACK_CLEANUP_FAILED, with
   the message explicitly stating the data operation succeeded and only
   staging cleanup needs manual attention.

The coordinator itself has no Spark/Delta dependency: every side effect
(stage, validate, capture target state, promote, restore, drop, cleanup,
log) is injected through the ``PublicationBackend`` protocol, so the state
machine can be exercised deterministically in tests without a Spark session
or a Fabric workspace. ``writer.py`` supplies the concrete Spark/Delta
catalog backend for notebooks and a filesystem backend for local
path/parquet tests.
"""

from dataclasses import dataclass, field
from typing import Any, Callable, Protocol, Sequence

# Bound error text kept in setup_run_log so one runaway exception can't blow
# past the log column budget.
ERROR_TEXT_LIMIT = 1000

# Terminal coordinator states. STAGED/VALIDATED/PROMOTING/ROLLING_BACK are
# intermediate/log-only states; cleanup warnings preserve data success.
COMPLETED = "COMPLETED"
FAILED = "FAILED"
ROLLED_BACK = "ROLLED_BACK"
ROLLBACK_FAILED = "ROLLBACK_FAILED"
# The data operation (promote or rollback) succeeded, but cleaning up staging
# afterward failed. Deliberately distinct from COMPLETED/ROLLED_BACK — never
# silently reported as success — and distinct from FAILED/ROLLBACK_FAILED —
# never conflated with a data problem.
COMPLETED_CLEANUP_FAILED = "COMPLETED_CLEANUP_FAILED"
ROLLED_BACK_CLEANUP_FAILED = "ROLLED_BACK_CLEANUP_FAILED"


def bound_error(exc: BaseException) -> str:
    """Render an exception as ``Type: message``, truncated for log storage."""
    return f"{type(exc).__name__}: {exc}"[:ERROR_TEXT_LIMIT]


@dataclass(frozen=True)
class TableTarget:
    """One candidate table to publish.

    ``name`` is the bare logical table name (e.g. ``fact_receipts``) used as
    the ``setup_run_log.table_name`` value. ``db`` is the schema/database it
    belongs to (silver or gold). ``staging_name`` is the run-scoped staging
    identifier the backend actually writes to — it is opaque to the
    coordinator (a qualified catalog table name, a filesystem path, ...).
    ``prestaged`` marks a staging identifier that already holds the candidate
    (a generation handoff): the backend neither writes nor cleans it up.
    """

    name: str
    db: str
    staging_name: str
    prestaged: bool = False


@dataclass(frozen=True)
class TargetState:
    """Pre-promotion snapshot of one target, captured before any promotion.

    ``existed`` distinguishes a target that must be *restored* on rollback
    from one that must be *dropped*. ``restore_token`` is backend-defined (a
    Delta version int for catalog targets, a backup path for filesystem
    targets) and is handed back to ``backend.restore()`` unchanged.
    """

    existed: bool
    restore_token: Any = None


@dataclass
class PublicationOutcome:
    """Terminal result of one ``PublicationCoordinator.publish()`` call."""

    # COMPLETED | FAILED | ROLLED_BACK | ROLLBACK_FAILED |
    # COMPLETED_CLEANUP_FAILED | ROLLED_BACK_CLEANUP_FAILED
    state: str
    promoted: list[str] = field(default_factory=list)
    error: str | None = None
    rollback_failures: list[tuple[str, str]] = field(default_factory=list)
    # Staging identifiers left behind for manual recovery: every staged
    # target on ROLLBACK_FAILED, or just the ones whose cleanup() call
    # itself failed on *_CLEANUP_FAILED.
    preserved_staging: list[str] = field(default_factory=list)
    cleanup_failures: list[tuple[str, str]] = field(default_factory=list)

    @property
    def ok(self) -> bool:
        """Whether promotion succeeded; cleanup warnings remain explicit in state."""
        return self.state in {COMPLETED, COMPLETED_CLEANUP_FAILED}


class PublicationBackend(Protocol):
    """Side effects the coordinator needs; implemented per storage mode."""

    def stage(self, target: TableTarget) -> int:
        """Write the candidate to ``target.staging_name``; return its row count."""
        ...

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        """Raise if the staged table's schema/row count disagree with the source."""
        ...

    def target_state(self, target: TableTarget) -> TargetState:
        """Snapshot whether the final target exists and its restore token."""
        ...

    def promote(self, target: TableTarget) -> int:
        """Swap staged data into the final target; return the promoted row count."""
        ...

    def restore(self, target: TableTarget, state: TargetState) -> None:
        """Undo a promotion of a pre-existing target using its captured state."""
        ...

    def drop(self, target: TableTarget) -> None:
        """Undo a promotion of a target that did not exist before this run."""
        ...

    def cleanup(self, target: TableTarget) -> None:
        """Remove the run-scoped staging artifact for ``target``."""
        ...


# (table_name, status, row_count, error) — mirrors setup_run_log's columns.
LogFn = Callable[[str, str, "int | None", "str | None"], None]


class PublicationCoordinator:
    """Runs the stage -> validate -> promote (-> rollback) state machine.

    ``targets`` are processed in the given order for staging, validation and
    promotion; rollback undoes them in reverse (LIFO) order. All logging
    goes through the injected ``log`` callback so callers can persist an
    append-only ``setup_run_log``-shaped history.
    """

    def __init__(
        self,
        backend: PublicationBackend,
        log: LogFn,
        *,
        max_workers: int = 1,
    ) -> None:
        self.backend = backend
        self.log = log
        # >1 runs each phase's per-table operations concurrently. Phases stay
        # strictly ordered (nothing is promoted until every table staged and
        # validated), so the all-or-nothing guarantees are unchanged.
        self.max_workers = max_workers

    def publish(self, targets: Sequence[TableTarget]) -> PublicationOutcome:
        if self.max_workers > 1:
            return self._publish_concurrently(targets)
        self.log("__run__", "STARTED", None, None)

        staged: list[TableTarget] = []
        # Keyed by the full TableTarget (name+db+staging_name), not the bare
        # name: a silver and a gold table can legitimately share a name, and
        # keying by name alone would let one target's staged count/state
        # silently shadow the other's.
        staged_counts: dict[TableTarget, int] = {}

        # Phase 1 — stage every candidate. Final targets are untouched here.
        for target in targets:
            try:
                count = self.backend.stage(target)
            except Exception as exc:  # noqa: BLE001 — recorded, not swallowed
                return self._fail(target, exc, staged)
            staged.append(target)
            staged_counts[target] = count
            self.log(target.name, "STAGED", count, None)

        # Phase 2 — validate every staged table before any promotion.
        for target in targets:
            try:
                self.backend.validate(target, staged_counts[target])
            except Exception as exc:  # noqa: BLE001
                return self._fail(target, exc, staged)
        self.log("__run__", "VALIDATED", None, None)

        # Phase 3 — snapshot pre-promotion state (existed + restore token) for
        # every target before touching any of them. Keyed by TableTarget for
        # the same same-name-different-db reason as staged_counts above.
        target_states = {t: self.backend.target_state(t) for t in targets}

        # Phase 4 — promote in order; roll back everything already promoted
        # (plus the target whose promotion just failed) if any promotion
        # fails.
        self.log("__run__", "PROMOTING", None, None)
        promoted: list[TableTarget] = []
        for target in targets:
            try:
                count = self.backend.promote(target)
            except Exception as exc:  # noqa: BLE001
                return self._roll_back(target, exc, promoted, target_states, staged)
            promoted.append(target)
            self.log(target.name, "COMPLETED", count, None)

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._completed_with_cleanup_failure(promoted, cleanup_failures)
        self.log("__run__", "COMPLETED", len(promoted), None)
        return PublicationOutcome(state=COMPLETED, promoted=[t.name for t in promoted])

    def _run_all(
        self,
        operation: Callable[[TableTarget], Any],
        targets: Sequence[TableTarget],
    ) -> list[tuple[TableTarget, Any, Exception | None]]:
        """Run ``operation`` for every target concurrently; results in target order."""
        from concurrent.futures import ThreadPoolExecutor

        try:
            # Keep the caller's Spark local properties (job group, Fabric run
            # tagging) on jobs started from worker threads.
            from pyspark import inheritable_thread_target as in_thread
        except ImportError:  # the coordinator itself does not need Spark

            def in_thread(function: Callable[..., Any]) -> Callable[..., Any]:
                return function

        with ThreadPoolExecutor(self.max_workers) as pool:
            futures = [(target, pool.submit(in_thread(operation), target)) for target in targets]
            results: list[tuple[TableTarget, Any, Exception | None]] = []
            for target, future in futures:
                try:
                    results.append((target, future.result(), None))
                except Exception as exc:  # noqa: BLE001 — returned to the caller
                    results.append((target, None, exc))
        return results

    def _publish_concurrently(self, targets: Sequence[TableTarget]) -> PublicationOutcome:
        """Same state machine as ``publish`` with each phase run concurrently.

        Every phase finishes for all targets before the next starts. A staging
        or validation failure fails the run before any promotion and cleans up
        everything that did stage. A promotion failure rolls back every target
        (promoted or failed), exactly as the sequential path rolls back
        everything attempted.
        """
        self.log("__run__", "STARTED", None, None)

        # Phase 1 — stage every candidate concurrently; finals untouched.
        stage_results = self._run_all(self.backend.stage, targets)
        staged = [target for target, _, error in stage_results if error is None]
        staged_counts = {target: count for target, count, error in stage_results if error is None}
        for target, count, error in stage_results:
            if error is None:
                self.log(target.name, "STAGED", count, None)
        first_failure = next((r for r in stage_results if r[2] is not None), None)
        if first_failure is not None:
            return self._fail(first_failure[0], first_failure[2], staged)

        # Phase 2 — validate every staged table before any promotion.
        validate_results = self._run_all(
            lambda target: self.backend.validate(target, staged_counts[target]),
            targets,
        )
        first_failure = next((r for r in validate_results if r[2] is not None), None)
        if first_failure is not None:
            return self._fail(first_failure[0], first_failure[2], staged)
        self.log("__run__", "VALIDATED", None, None)

        # Phase 3 — snapshot every target's pre-promotion state.
        state_results = self._run_all(self.backend.target_state, targets)
        first_failure = next((r for r in state_results if r[2] is not None), None)
        if first_failure is not None:
            return self._fail(first_failure[0], first_failure[2], staged)
        target_states = {target: state for target, state, _ in state_results}

        # Phase 4 — promote concurrently; any failure rolls back every target.
        self.log("__run__", "PROMOTING", None, None)
        promote_results = self._run_all(self.backend.promote, targets)
        promoted = [target for target, _, error in promote_results if error is None]
        for target, count, error in promote_results:
            if error is None:
                self.log(target.name, "COMPLETED", count, None)
        failed = [(target, error) for target, _, error in promote_results if error is not None]
        if failed:
            first_target, first_error = failed[0]
            # Other failed targets may also have partially committed; include
            # them so rollback restores or drops every target of the run.
            others = [target for target, _ in failed[1:]]
            return self._roll_back(
                first_target,
                first_error,
                [*promoted, *others],
                target_states,
                staged,
            )

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._completed_with_cleanup_failure(promoted, cleanup_failures)
        self.log("__run__", "COMPLETED", len(promoted), None)
        return PublicationOutcome(state=COMPLETED, promoted=[t.name for t in promoted])

    def _fail(
        self, target: TableTarget, exc: Exception, staged: list[TableTarget]
    ) -> PublicationOutcome:
        error = bound_error(exc)
        self.log(target.name, "FAILED", None, error)
        self.log("__run__", "FAILED", None, error)
        # Evidence (the STAGED/FAILED rows above) is logged before cleanup
        # runs. No data was touched, so the run stays FAILED regardless of
        # whether cleanup itself succeeds — but a cleanup failure is still
        # logged (not silently swallowed) and its staging left in place.
        cleanup_failures = self._cleanup_all(staged)
        preserved = [name for name, _ in cleanup_failures]
        if cleanup_failures:
            failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
            self.log(
                "__run__",
                "FAILED",
                None,
                f"staging cleanup also failed: {failure_text}"[:ERROR_TEXT_LIMIT],
            )
        return PublicationOutcome(
            state=FAILED,
            error=error,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

    def _roll_back(
        self,
        failed_target: TableTarget,
        exc: Exception,
        promoted: list[TableTarget],
        target_states: dict[TableTarget, TargetState],
        staged: list[TableTarget],
    ) -> PublicationOutcome:
        error = bound_error(exc)
        self.log(failed_target.name, "FAILED", None, error)
        self.log("__run__", "ROLLING_BACK", None, error)

        # Roll back everything already promoted *and* the target whose
        # promote() call just raised: a Delta command can commit before the
        # client observes a network/driver error, so the failed target may
        # have partially or fully landed. Restoring/dropping it anyway is
        # safe and idempotent (RESTORE to the pre-run version is a no-op if
        # nothing committed; DROP TABLE IF EXISTS is a no-op if it was never
        # created) and keeps rollback fail-closed instead of trusting the
        # exception.
        attempted = [*promoted, failed_target]
        failures = self._attempt_rollback(attempted, target_states)

        if failures:
            preserved = [t.staging_name for t in staged]
            names = ", ".join(preserved)
            failure_text = "; ".join(f"{name}: {err}" for name, err in failures)
            self.log(
                "__run__",
                ROLLBACK_FAILED,
                None,
                f"{error}; rollback failures: {failure_text}; "
                f"staging preserved for manual recovery: {names}"[:ERROR_TEXT_LIMIT],
            )
            return PublicationOutcome(
                state=ROLLBACK_FAILED,
                error=error,
                rollback_failures=failures,
                preserved_staging=preserved,
            )

        cleanup_failures = self._cleanup_all(staged)
        if cleanup_failures:
            return self._rolled_back_with_cleanup_failure(error, cleanup_failures)
        self.log("__run__", ROLLED_BACK, None, error)
        return PublicationOutcome(state=ROLLED_BACK, error=error)

    def _attempt_rollback(
        self, targets: list[TableTarget], target_states: dict[TableTarget, TargetState]
    ) -> list[tuple[str, str]]:
        """Undo every given target in reverse (attempted) order.

        ``targets`` is every target that was promoted *plus* the one whose
        promotion just failed (see ``_roll_back``); continues past a
        restore/drop failure so one bad target can't stop the rest from
        being rolled back.
        """
        failures: list[tuple[str, str]] = []
        for target in reversed(targets):
            state = target_states[target]
            try:
                if state.existed:
                    self.backend.restore(target, state)
                else:
                    self.backend.drop(target)
                self.log(target.name, ROLLED_BACK, None, None)
            except Exception as exc:  # noqa: BLE001 — keep rolling back the rest
                error = bound_error(exc)
                failures.append((target.name, error))
                self.log(target.name, ROLLBACK_FAILED, None, error)
        return failures

    def _cleanup_all(self, staged: list[TableTarget]) -> list[tuple[str, str]]:
        """Best-effort cleanup of every staged artifact; never raises.

        Attempts every staged target even if one fails, and returns the
        failures instead of raising: an unhandled exception here would blow
        past an already-successful promotion or rollback and could mask it
        (the caller would never see the terminal log/outcome it earned).
        """
        failures: list[tuple[str, str]] = []
        for target in staged:
            try:
                self.backend.cleanup(target)
            except Exception as exc:  # noqa: BLE001 — collected, not swallowed
                failures.append((target.staging_name, bound_error(exc)))
        return failures

    def _completed_with_cleanup_failure(
        self, promoted: list[TableTarget], cleanup_failures: list[tuple[str, str]]
    ) -> PublicationOutcome:
        """Promotion fully succeeded, but staging cleanup did not.

        Never reported as plain COMPLETED (that would hide leftover
        staging) and never conflated with a data failure — the message is
        explicit that the promoted data is fine and only cleanup needs
        manual attention.
        """
        preserved = [name for name, _ in cleanup_failures]
        failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
        message = (f"data promoted successfully; staging cleanup failed: {failure_text}")[
            :ERROR_TEXT_LIMIT
        ]
        self.log("__run__", COMPLETED_CLEANUP_FAILED, len(promoted), message)
        return PublicationOutcome(
            state=COMPLETED_CLEANUP_FAILED,
            promoted=[t.name for t in promoted],
            error=message,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

    def _rolled_back_with_cleanup_failure(
        self, error: str, cleanup_failures: list[tuple[str, str]]
    ) -> PublicationOutcome:
        """Rollback fully restored/dropped every target, but cleanup did not.

        Never reported as plain ROLLED_BACK (that would hide leftover
        staging) and never mislabeled as a data problem — the message states
        the rollback itself succeeded and only cleanup needs manual
        attention.
        """
        preserved = [name for name, _ in cleanup_failures]
        failure_text = "; ".join(f"{name}: {err}" for name, err in cleanup_failures)
        message = (
            f"{error}; data rolled back successfully; staging cleanup failed: {failure_text}"
        )[:ERROR_TEXT_LIMIT]
        self.log("__run__", ROLLED_BACK_CLEANUP_FAILED, None, message)
        return PublicationOutcome(
            state=ROLLED_BACK_CLEANUP_FAILED,
            error=message,
            cleanup_failures=cleanup_failures,
            preserved_staging=preserved,
        )

# --- retail_setup/generation/writer.py ---
"""Write layer. Notebooks call write_to_lakehouse/write_all; tests use
write_table with a format/location override (no delta-spark dependency
locally).

``write_all`` publishes every candidate table through the stage -> validate
-> promote (-> rollback) state machine in ``publication.py``: nothing final
is touched until every candidate has staged and validated successfully, and
any promotion failure rolls back everything already promoted in this run.
Catalog mode (``lakehouse=``) uses real Delta staging tables and
``RESTORE TABLE ... TO VERSION AS OF``/``DROP TABLE`` for rollback; local
path mode (``base_path=``) uses a filesystem staging/backup strategy so
tests don't need delta-spark.
"""

import re
import shutil
import warnings
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


_UNSAFE_IDENTIFIER = re.compile(r"[^0-9A-Za-z_]")
PUBLICATION_WORKERS = 8


def sanitize_identifier(value: str) -> str:
    """Make ``value`` safe to embed in a Spark/Delta table identifier.

    Used to derive run-scoped staging table names from ``run_id`` (which may
    contain ``-`` and other characters legal in a run id but not desirable in
    an unquoted identifier segment).
    """
    return _UNSAFE_IDENTIFIER.sub("_", value)


def assert_historical_replacement_safe(spark: SparkSession, lakehouse: str, silver_db: str) -> None:
    """Refuse to replace shared Silver data after streaming has initialized."""

    checkpoint_table = f"{lakehouse}.{silver_db}._watermarks"
    if spark.catalog.tableExists(checkpoint_table):
        raise RuntimeError(
            f"Historical replacement is blocked because {checkpoint_table} exists. "
            "Stop streaming and use the documented coordinated Silver/checkpoint "
            "reset before rebuilding history; never delete checkpoints alone."
        )


def write_table(df: DataFrame, location: str, fmt: str = "delta") -> None:
    df.write.format(fmt).mode("overwrite").save(location)


def write_to_lakehouse(df: DataFrame, lakehouse: str, schema: str, table: str) -> None:
    """Overwrite-by-design, matching 02-historical-data-load semantics."""
    df.write.format("delta").mode("overwrite").saveAsTable(f"{lakehouse}.{schema}.{table}")


def _schema_signature(df: DataFrame) -> list[tuple[str, str]]:
    return [(f.name, f.dataType.simpleString()) for f in df.schema.fields]


class _LakehouseBackend:
    """PublicationBackend for Fabric Lakehouse Delta tables.

    Every candidate is staged to ``{lakehouse}.{db}_stage.{run_token}__{name}``
    (a schema distinct from and never colliding with the final ``{db}``
    schema, or with another run's staging tables — ``run_token`` is derived
    from the already-unique ``run_id``). Promotion uses
    ``CREATE OR REPLACE TABLE ... USING DELTA AS SELECT * FROM <staging>``.
    Rollback restores pre-existing targets with
    ``RESTORE TABLE ... TO VERSION AS OF`` (version captured via
    ``DESCRIBE HISTORY ... LIMIT 1`` before promotion) and drops targets that
    were newly created by this run.
    """

    def __init__(
        self, spark, lakehouse: str, run_id: str, sources: dict[tuple[str, str], DataFrame]
    ) -> None:
        self.spark = spark
        self.lakehouse = lakehouse
        self.run_token = sanitize_identifier(run_id)
        self.sources = sources
        self._staging_dbs_created: set[str] = set()

    def _final(self, target: TableTarget) -> str:
        return f"{self.lakehouse}.{target.db}.{target.name}"

    def _ensure_staging_db(self, db: str) -> None:
        stage_db = f"{db}_stage"
        if stage_db not in self._staging_dbs_created:
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.lakehouse}.{stage_db}")
            self._staging_dbs_created.add(stage_db)

    def stage(self, target: TableTarget) -> int:
        if not target.prestaged:
            df = self.sources[(target.db, target.name)]
            self._ensure_staging_db(target.db)
            df.write.format("delta").mode("overwrite").saveAsTable(target.staging_name)
        return self.spark.table(target.staging_name).count()

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        df = self.sources[(target.db, target.name)]
        staged = self.spark.table(target.staging_name)
        staged_schema = _schema_signature(staged)
        source_schema = _schema_signature(df)
        if staged_schema != source_schema:
            raise ValueError(
                f"schema mismatch staging {target.staging_name!r}: "
                f"{staged_schema} != {source_schema}"
            )
        source_count = df.count()
        if staged_row_count != source_count:
            raise ValueError(
                f"row count mismatch staging {target.staging_name!r}: "
                f"staged={staged_row_count} source={source_count}"
            )

    def target_state(self, target: TableTarget) -> TargetState:
        final = self._final(target)
        if not self.spark.catalog.tableExists(final):
            return TargetState(existed=False)
        version = self.spark.sql(f"DESCRIBE HISTORY {final} LIMIT 1").collect()[0]["version"]
        return TargetState(existed=True, restore_token=version)

    def promote(self, target: TableTarget) -> int:
        final = self._final(target)
        self.spark.sql(
            f"CREATE OR REPLACE TABLE {final} USING DELTA AS SELECT * FROM {target.staging_name}"
        )
        return self.spark.table(final).count()

    def restore(self, target: TableTarget, state: TargetState) -> None:
        final = self._final(target)
        self.spark.sql(f"RESTORE TABLE {final} TO VERSION AS OF {state.restore_token}")

    def drop(self, target: TableTarget) -> None:
        self.spark.sql(f"DROP TABLE IF EXISTS {self._final(target)}")

    def cleanup(self, target: TableTarget) -> None:
        # Prestaged handoffs belong to the generation run; the publish
        # notebook drops them only after a successful publication.
        if not target.prestaged:
            self.spark.sql(f"DROP TABLE IF EXISTS {target.staging_name}")


class _FilesystemBackend:
    """PublicationBackend for local path/parquet test mode.

    No Delta dependency: staging writes to a run-scoped directory under
    ``<base_path>/.setup_staging/<run_token>/``, promotion replaces the final
    directory outright, and rollback restores from a pre-promotion backup
    copy (pre-existing targets, backed up under ``.setup_backup/<run_token>/``
    before the first promotion touches them) or simply removes the directory
    (targets created by this run).
    """

    def __init__(
        self,
        spark,
        base_path: Path,
        fmt: str,
        run_id: str,
        sources: dict[tuple[str, str], DataFrame],
    ) -> None:
        self.spark = spark
        self.base_path = base_path
        self.fmt = fmt
        self.run_token = sanitize_identifier(run_id)
        self.sources = sources
        self._staging_root = base_path / ".setup_staging" / self.run_token
        self._backup_root = base_path / ".setup_backup" / self.run_token

    def _final_path(self, target: TableTarget) -> Path:
        return self.base_path / target.db / target.name

    def _staging_path(self, target: TableTarget) -> Path:
        if target.prestaged:
            return Path(target.staging_name)
        return self._staging_root / target.db / target.name

    def _backup_path(self, target: TableTarget) -> Path:
        return self._backup_root / target.db / target.name

    def stage(self, target: TableTarget) -> int:
        path = self._staging_path(target)
        if not target.prestaged:
            df = self.sources[(target.db, target.name)]
            df.write.format(self.fmt).mode("overwrite").save(str(path))
        return self.spark.read.format(self.fmt).load(str(path)).count()

    def validate(self, target: TableTarget, staged_row_count: int) -> None:
        df = self.sources[(target.db, target.name)]
        staged = self.spark.read.format(self.fmt).load(str(self._staging_path(target)))
        staged_schema = _schema_signature(staged)
        source_schema = _schema_signature(df)
        if staged_schema != source_schema:
            raise ValueError(
                f"schema mismatch staging {target.name!r}: {staged_schema} != {source_schema}"
            )
        source_count = df.count()
        if staged_row_count != source_count:
            raise ValueError(
                f"row count mismatch staging {target.name!r}: "
                f"staged={staged_row_count} source={source_count}"
            )

    def target_state(self, target: TableTarget) -> TargetState:
        final = self._final_path(target)
        if not final.exists():
            return TargetState(existed=False)
        backup = self._backup_path(target)
        backup.parent.mkdir(parents=True, exist_ok=True)
        if backup.exists():
            shutil.rmtree(backup)
        shutil.copytree(final, backup)
        return TargetState(existed=True, restore_token=str(backup))

    def promote(self, target: TableTarget) -> int:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)
        final.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(self._staging_path(target), final)
        return self.spark.read.format(self.fmt).load(str(final)).count()

    def restore(self, target: TableTarget, state: TargetState) -> None:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)
        shutil.copytree(Path(str(state.restore_token)), final)

    def drop(self, target: TableTarget) -> None:
        final = self._final_path(target)
        if final.exists():
            shutil.rmtree(final)

    def cleanup(self, target: TableTarget) -> None:
        staged = self._staging_path(target)
        if staged.exists() and not target.prestaged:
            shutil.rmtree(staged)
        backup = self._backup_path(target)
        if backup.exists():
            shutil.rmtree(backup)


def write_all(
    tables: dict[str, DataFrame],
    gold: dict[str, DataFrame],
    cfg: GenerationConfig,
    run_id: str,
    *,
    lakehouse: str | None = None,
    base_path: str | None = None,
    fmt: str = "delta",
    prestaged: dict[str, str] | None = None,
) -> list[str]:
    """Publish dims+facts to silver, gold to gold, then setup_run_log.

    Two modes: catalog (lakehouse=...) used by notebooks, or path
    (base_path=...) used by local tests/E2E. Either ``tables`` or ``gold``
    (or both) may be non-empty; at least one of the two must contain a table.
    dim_date is written as generated by the engine (±5y padding preserved —
    the semantic model's date relationships need the headroom).

    ``prestaged`` maps Silver table names to an existing staging identifier
    (a generation handoff table, or path in path mode) that already holds
    exactly that table's rows. Those targets skip the staging copy, are
    validated and promoted from the handoff, and are never dropped here.

    Every candidate table is staged to a run-scoped location, validated
    against its source, and only then promoted to its fixed final name (see
    ``retail_setup.generation.publication``). If any promotion fails, every
    target already promoted in this run is rolled back and a
    ``RuntimeError`` is raised — this function never returns after a partial
    or rolled-back publish; a return only happens once every candidate has
    been promoted.

    Returns the list of written table names (silver + gold); the
    setup_run_log table itself is not included in the returned list.

    The Spark session is derived from the first DataFrame in ``tables`` or
    ``gold`` (``df.sparkSession``) — no explicit session parameter is needed.
    """
    if (lakehouse is None) == (base_path is None):
        raise ValueError("Provide exactly one of lakehouse= or base_path=")

    first_df = next(iter(tables.values()), None) or next(iter(gold.values()), None)
    if first_df is None:
        raise ValueError("write_all requires at least one table in tables or gold")
    spark = first_df.sparkSession
    if lakehouse is not None and tables:
        assert_historical_replacement_safe(spark, lakehouse, cfg.silver_db)

    log_name = "setup_run_log"
    log_table = f"{lakehouse}.{cfg.silver_db}.{log_name}" if lakehouse is not None else None
    log_path = f"{base_path}/{cfg.silver_db}/{log_name}" if base_path is not None else None

    def _log_exists() -> bool:
        if log_table is not None:
            return spark.catalog.tableExists(log_table)
        return Path(str(log_path)).exists()

    def _read_log() -> DataFrame:
        if log_table is not None:
            return spark.table(log_table)
        return spark.read.format(fmt).load(str(log_path))

    def _append_log(
        table_name: str,
        row_count: int | None,
        status: str,
        error: str | None = None,
    ) -> None:
        row = [
            (
                run_id,
                cfg.store_type,
                cfg.seed,
                cfg.start_date,
                cfg.end_date,
                table_name,
                row_count,
                status,
                error,
            )
        ]
        log_df = spark.createDataFrame(
            row,
            "run_id string, store_type string, seed long, start_date date, "
            "end_date date, table_name string, row_count long, status string, "
            "error string",
        ).withColumn("generated_at", F.current_timestamp())
        writer = log_df.write.format("delta" if log_table is not None else fmt)
        writer = writer.mode("append").option("mergeSchema", "true")
        if log_table is not None:
            writer.saveAsTable(log_table)
        else:
            writer.save(str(log_path))

    if _log_exists() and _read_log().filter(F.col("run_id") == run_id).limit(1).count():
        raise ValueError(f"setup run_id already exists: {run_id!r}")

    run_token = sanitize_identifier(run_id)
    prestaged = prestaged or {}
    unknown = sorted(set(prestaged) - set(tables))
    if unknown:
        raise ValueError(f"prestaged tables are not Silver candidates: {unknown}")
    sources: dict[tuple[str, str], DataFrame] = {}
    targets: list[TableTarget] = []
    for db, frames in ((cfg.silver_db, tables), (cfg.gold_db, gold)):
        for name, df in frames.items():
            sources[(db, name)] = df
            handoff = prestaged.get(name) if frames is tables else None
            if handoff is not None:
                targets.append(TableTarget(name=name, db=db, staging_name=handoff, prestaged=True))
                continue
            if lakehouse is not None:
                staging_name = f"{lakehouse}.{db}_stage.{run_token}__{name}"
            else:
                staging_name = f"{db}/{name}"
            targets.append(TableTarget(name=name, db=db, staging_name=staging_name))

    backend: PublicationBackend
    if lakehouse is not None:
        for db in (cfg.silver_db, cfg.gold_db):
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {lakehouse}.{db}")
            # Created up front: staging runs concurrently per table.
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {lakehouse}.{db}_stage")
        backend = _LakehouseBackend(spark, lakehouse, run_id, sources)
    else:
        assert base_path is not None  # enforced by the exactly-one-of check above
        backend = _FilesystemBackend(spark, Path(base_path), fmt, run_id, sources)

    def _log(table_name: str, status: str, row_count: int | None, error: str | None) -> None:
        _append_log(table_name, row_count, status, error)

    # Per-table staging, validation, and promotion are independent and each is
    # a separate Spark job; running them concurrently removes per-table
    # overhead without changing the phase ordering or rollback guarantees.
    coordinator = PublicationCoordinator(backend, _log, max_workers=PUBLICATION_WORKERS)
    outcome = coordinator.publish(targets)

    if not outcome.ok:
        raise RuntimeError(
            f"setup publication {outcome.state} for run_id={run_id!r}: {outcome.error}"
        )
    if outcome.state == COMPLETED_CLEANUP_FAILED:
        warnings.warn(
            f"setup publication completed but staging cleanup needs attention: {outcome.error}",
            RuntimeWarning,
            stacklevel=2,
        )

    return outcome.promoted

## Run this notebook's generation stages

In [ ]:
from datetime import date

SETUP_NOTEBOOK = "setup-03-generate-sales"

cfg = GenerationConfig(
    store_type=STORE_TYPE,
    start_date=date.fromisoformat(START_DATE),
    end_date=date.fromisoformat(END_DATE),
    store_count=STORE_COUNT,
    seed=SEED,
    silver_db=SILVER_DB,
    gold_db=GOLD_DB,
    dictionary_root="/lakehouse/default/Files/setup/dictionaries",
)
dicts = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type)

# Size shuffles to the generated volume (Spark's default 200 partitions makes each
# billion-row full-demo shuffle task spill millions of rows); adaptive execution
# coalesces small shuffles back down. Output does not depend on partitioning.
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set(
    "spark.sql.shuffle.partitions",
    str(shuffle_partitions_for(cfg, dicts.profile.basket_lambda)),
)
print(f"spark.sql.shuffle.partitions={spark.conf.get('spark.sql.shuffle.partitions')}")

# Dimensions are regenerated in memory in every notebook: generation is fully
# deterministic (same seed => identical dims), so this is cheaper than a
# catalog round-trip. Checkpoints live in the Lakehouse, not on executors, so
# autoscaling or a lost executor cannot drop rows a later step still needs
# (CHECKPOINT_RDD_BLOCK_ID_NOT_FOUND). Each notebook owns its folder, so
# concurrent notebooks never touch each other's files; clearing it first
# removes whatever a failed attempt left behind.
checkpoint_dir = f"Files/setup/_checkpoints/{SETUP_NOTEBOOK}"
delete_lakehouse_path(spark, checkpoint_dir)
store = HandoffStore(spark, cfg, lakehouse=LAKEHOUSE_NAME)
outputs = run_generation_notebook(
    spark, dicts, cfg, SETUP_NOTEBOOK, store, checkpoint_dir=checkpoint_dir,
)
delete_lakehouse_path(spark, checkpoint_dir)
for stage_name, frames in sorted(outputs.items()):
    for name, frame in sorted(frames.items()):
        print(f"{stage_name}/{name}: {frame.count():,} rows -> {store.name(stage_name, name)}")